# Amazon ML Challenge 2026 — Business Entity Resolution

This notebook implements the end-to-end business entity resolution pipeline for the Amazon ML Challenge 2026.

The challenge requires identifying matching records from Source 2 and Source 3 for every Source 1 business entity.

### Pipeline Overview

1. Load and verify the provided dataset
2. Inspect and use the reference entity-resolution pipeline
3. Generate candidate matches using blocking
4. Score candidate pairs
5. Generate the required submission files
6. Validate the final outputs before submission

All processing is designed to respect the Kaggle runtime and memory constraints.

In [2]:
# ============================================================
# CELL 1 — PATHS + ENVIRONMENT CHECK
# ============================================================

import os
import sys
import json
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(
    "/kaggle/input/datasets/tejasweerajput/"
    "amazon-ml-challenge-2026-student-resource/"
    "student_resource"
)

TRAIN_DIR = ROOT / "dataset" / "train"
TEST_DIR  = ROOT / "dataset" / "test"
SOLUTION_DIR = ROOT / "amazon_ml_solution"
UTILS_DIR = ROOT / "utils"

WORK_DIR = Path("/kaggle/working/amazon_ml_final")
WORK_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("ROOT:", ROOT)
print("TRAIN:", TRAIN_DIR)
print("TEST :", TEST_DIR)
print("SOLUTION:", SOLUTION_DIR)
print("WORK:", WORK_DIR)
print("=" * 70)

print("\nRoot exists:", ROOT.exists())
print("Train exists:", TRAIN_DIR.exists())
print("Test exists :", TEST_DIR.exists())
print("Solution exists:", SOLUTION_DIR.exists())

print("\nFiles under amazon_ml_solution:")
if SOLUTION_DIR.exists():
    for p in sorted(SOLUTION_DIR.rglob("*")):
        if p.is_file():
            print(" ", p.relative_to(SOLUTION_DIR))

print("\nTest files:")
for p in sorted(TEST_DIR.glob("*.tsv")):
    print(" ", p.name, f"{p.stat().st_size / 1024**2:.1f} MB")

print("\nTrain files:")
for p in sorted(TRAIN_DIR.glob("*.tsv")):
    print(" ", p.name, f"{p.stat().st_size / 1024**2:.1f} MB")

ROOT: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource
TRAIN: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train
TEST : /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/test
SOLUTION: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/amazon_ml_solution
WORK: /kaggle/working/amazon_ml_final

Root exists: True
Train exists: True
Test exists : True
Solution exists: True

Files under amazon_ml_solution:
  output/address_blocking_test.txt
  output/blocking_combinations.txt
  output/blocking_key_analysis.txt
  output/candidate_recall_missed.tsv
  output/candidate_recall_summary.txt
  output/missed_matches_inspection.tsv
  output/s2_candidate_recall.txt
  output/s2_missed_matches.tsv
  output/s2_missed_matches_inspection.tsv
  output/training_features_test.tsv
  src/analyze_address_key

## 2. Dataset Format Verification

The challenge uses tab-separated (`.tsv`) files.

Each Source 1, Source 2, and Source 3 file contains:

- `entity_id`
- `business_name`
- `business_address`
- `country`

The training ground-truth file contains:

- `source1_entity_id`
- `matched_entity_ids`

This cell verifies that the attached Kaggle dataset has the expected structure before any expensive processing begins.

In [3]:
# ============================================================
# CELL 2 — VERIFY DATA FORMAT
# ============================================================

TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

TRAIN_S1 = TRAIN_DIR / "train_source1.tsv"
TRAIN_S2 = TRAIN_DIR / "train_source2.tsv"
TRAIN_S3 = TRAIN_DIR / "train_source3.tsv"
TRAIN_GT = TRAIN_DIR / "train_ground_truth.tsv"

expected_source_cols = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

for name, path in {
    "TRAIN S1": TRAIN_S1,
    "TRAIN S2": TRAIN_S2,
    "TRAIN S3": TRAIN_S3,
    "TEST S1": TEST_S1,
    "TEST S2": TEST_S2,
    "TEST S3": TEST_S3,
}.items():

    cols = pd.read_csv(
        path,
        sep="\t",
        nrows=3
    ).columns.tolist()

    print(f"{name}:")
    print(" ", cols)

    assert cols == expected_source_cols, (
        f"Unexpected columns in {name}: {cols}"
    )

gt_cols = pd.read_csv(
    TRAIN_GT,
    sep="\t",
    nrows=3
).columns.tolist()

print("\nGROUND TRUTH:")
print(gt_cols)

assert gt_cols == [
    "source1_entity_id",
    "matched_entity_ids"
]

print("\n✅ DATA FORMAT CHECK PASSED")

TRAIN S1:
  ['entity_id', 'business_name', 'business_address', 'country']
TRAIN S2:
  ['entity_id', 'business_name', 'business_address', 'country']
TRAIN S3:
  ['entity_id', 'business_name', 'business_address', 'country']
TEST S1:
  ['entity_id', 'business_name', 'business_address', 'country']
TEST S2:
  ['entity_id', 'business_name', 'business_address', 'country']
TEST S3:
  ['entity_id', 'business_name', 'business_address', 'country']

GROUND TRUTH:
['source1_entity_id', 'matched_entity_ids']

✅ DATA FORMAT CHECK PASSED


## 3. Reference Entity-Resolution Pipeline

The attached student resource contains an existing reference implementation under `amazon_ml_solution`.

The pipeline contains the main components required for entity resolution:

- **Normalization** — normalizes business names and addresses
- **Blocking** — generates multiple candidate-generation keys
- **Indexing** — creates searchable indexes for Source 2 and Source 3
- **Candidate generation** — retrieves plausible matches for each Source 1 entity
- **Feature engineering** — calculates similarity features
- **Labels** — connects candidate pairs with training ground truth
- **Pipeline** — orchestrates the processing workflow

We will reuse and inspect this implementation rather than rebuilding the blocking system from scratch.

In [4]:
# ============================================================
# CELL 2 — VERIFY DATA FORMAT
# ============================================================

TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

TRAIN_S1 = TRAIN_DIR / "train_source1.tsv"
TRAIN_S2 = TRAIN_DIR / "train_source2.tsv"
TRAIN_S3 = TRAIN_DIR / "train_source3.tsv"
TRAIN_GT = TRAIN_DIR / "train_ground_truth.tsv"

expected_source_cols = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

for name, path in {
    "TRAIN S1": TRAIN_S1,
    "TRAIN S2": TRAIN_S2,
    "TRAIN S3": TRAIN_S3,
    "TEST S1": TEST_S1,
    "TEST S2": TEST_S2,
    "TEST S3": TEST_S3,
}.items():

    cols = pd.read_csv(
        path,
        sep="\t",
        nrows=3
    ).columns.tolist()

    print(f"{name}:")
    print(" ", cols)

    assert cols == expected_source_cols, (
        f"Unexpected columns in {name}: {cols}"
    )

gt_cols = pd.read_csv(
    TRAIN_GT,
    sep="\t",
    nrows=3
).columns.tolist()

print("\nGROUND TRUTH:")
print(gt_cols)

assert gt_cols == [
    "source1_entity_id",
    "matched_entity_ids"
]

print("\n✅ DATA FORMAT CHECK PASSED")

TRAIN S1:
  ['entity_id', 'business_name', 'business_address', 'country']
TRAIN S2:
  ['entity_id', 'business_name', 'business_address', 'country']
TRAIN S3:
  ['entity_id', 'business_name', 'business_address', 'country']
TEST S1:
  ['entity_id', 'business_name', 'business_address', 'country']
TEST S2:
  ['entity_id', 'business_name', 'business_address', 'country']
TEST S3:
  ['entity_id', 'business_name', 'business_address', 'country']

GROUND TRUTH:
['source1_entity_id', 'matched_entity_ids']

✅ DATA FORMAT CHECK PASSED


## 4. Locate the Reference Pipeline

The reference solution is already included in the provided student resource.

In this step, we locate the main `pipeline.py` file and inspect its structure before executing anything.

This is an inspection-only step. No training, indexing, candidate generation, or large dataset processing is performed here.

If `pipeline.py` is located successfully, we will inspect its command-line arguments in the next step so that the pipeline can be executed with the correct paths and resource-safe settings.

In [5]:
# ============================================================
# CELL 4 — LOCATE REFERENCE PIPELINE SAFELY
# pipeline.py is optional because this resource does not contain it.
# Continue with the available amazon_ml_solution/src modules.
# ============================================================

print("=" * 70)
print("CELL 4 — CHECK REFERENCE PIPELINE")
print("=" * 70)

pipeline_matches = list(ROOT.rglob("pipeline.py"))

print(f"Found {len(pipeline_matches)} pipeline.py file(s).")

if pipeline_matches:
    print("\nFound pipeline.py:")
    for i, path in enumerate(pipeline_matches, start=1):
        print(f"  {i}. {path}")
    PIPELINE_FILE = pipeline_matches[0]
else:
    print("\n⚠️ pipeline.py not found.")
    print("Continuing with the available source modules instead.")
    print("Using:")
    print(f"  {SRC_DIR}")

    PIPELINE_FILE = None

print("\nCell 4 completed successfully.")

CELL 4 — CHECK REFERENCE PIPELINE
Found 0 pipeline.py file(s).

⚠️ pipeline.py not found.
Continuing with the available source modules instead.
Using:


NameError: name 'SRC_DIR' is not defined

## 5. Inspect the Available Entity-Resolution Modules

The attached `amazon_ml_solution` contains an earlier entity-resolution implementation rather than the newer `pipeline.py` architecture.

We will inspect the available source modules and identify:

- how the training/test data is loaded,
- how records are normalized,
- how candidates are generated,
- how candidate pairs are scored,
- and which module can serve as the main execution entry point.

This step only inspects the source code. It does not load the full multi-million-row datasets or perform candidate generation.

In [6]:
# ============================================================
# CELL 5 — INSPECT AVAILABLE SOLUTION MODULES
# ============================================================

SRC_DIR = SOLUTION_DIR / "src"

print("=" * 70)
print("AVAILABLE SOURCE MODULES")
print("=" * 70)

source_files = sorted(SRC_DIR.glob("*.py"))

for p in source_files:
    print(f"\n{'-' * 70}")
    print(p.name)
    print(f"Size: {p.stat().st_size / 1024:.1f} KB")

print("\n" + "=" * 70)
print("KEY MODULES")
print("=" * 70)

key_modules = [
    "data_loader.py",
    "normalize.py",
    "similarity.py",
    "candidate_generator.py",
    "score_candidates.py",
    "build_training_features.py",
]

for filename in key_modules:

    path = SRC_DIR / filename

    if not path.exists():
        print(f"\n❌ {filename} — not found")
        continue

    print(f"\n{'=' * 70}")
    print(filename)
    print("=" * 70)

    text = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    lines = text.splitlines()

    # Print imports, functions and classes only.
    # This lets us understand the architecture without
    # dumping hundreds of lines into the notebook.
    for i, line in enumerate(lines, start=1):

        stripped = line.strip()

        if (
            stripped.startswith("def ")
            or stripped.startswith("class ")
            or stripped.startswith("import ")
            or stripped.startswith("from ")
        ):
            print(f"{i:04d}: {line}")

print("\n" + "=" * 70)
print("✅ SOURCE INSPECTION COMPLETE")
print("=" * 70)

AVAILABLE SOURCE MODULES

----------------------------------------------------------------------
analyze_address_key_frequency.py
Size: 2.3 KB

----------------------------------------------------------------------
analyze_ground_truth.py
Size: 1.5 KB

----------------------------------------------------------------------
build_training_features.py
Size: 5.7 KB

----------------------------------------------------------------------
candidate_generator.py
Size: 9.1 KB

----------------------------------------------------------------------
check_address_blocking.py
Size: 8.6 KB

----------------------------------------------------------------------
check_candidate_recall.py
Size: 5.1 KB

----------------------------------------------------------------------
check_training_features.py
Size: 1.1 KB

----------------------------------------------------------------------
data_loader.py
Size: 0.7 KB

----------------------------------------------------------------------
feature_experiment.py


## 6. Inspect Candidate Generation and Scoring

The available solution uses `candidate_generator.py` as its candidate-generation module.

This module contains:

- name blocking keys,
- address blocking keys,
- an index-building function,
- and the candidate lookup function.

`score_candidates.py` then combines candidate generation with the similarity functions.

Before running the pipeline on any large dataset, we inspect these functions to understand their inputs, outputs, and resource requirements.

No full dataset is loaded in this step.

In [7]:
# ============================================================
# CELL 6 — INSPECT CANDIDATE GENERATION
# ============================================================

CANDIDATE_FILE = SRC_DIR / "candidate_generator.py"
SCORE_FILE = SRC_DIR / "score_candidates.py"

print("=" * 70)
print("CANDIDATE GENERATOR")
print("=" * 70)

candidate_text = CANDIDATE_FILE.read_text(
    encoding="utf-8",
    errors="replace"
)

candidate_lines = candidate_text.splitlines()

for i, line in enumerate(candidate_lines, start=1):
    stripped = line.strip()

    if (
        stripped.startswith("def ")
        or stripped.startswith("class ")
        or stripped.startswith("import ")
        or stripped.startswith("from ")
    ):
        print(f"{i:04d}: {line}")

print("\n" + "=" * 70)
print("CANDIDATE GENERATOR — FUNCTION HEADERS")
print("=" * 70)

for i, line in enumerate(candidate_lines, start=1):
    if line.strip().startswith("def "):
        print(f"{i:04d}: {line.strip()}")

print("\n" + "=" * 70)
print("SCORING MODULE")
print("=" * 70)

score_text = SCORE_FILE.read_text(
    encoding="utf-8",
    errors="replace"
)

for i, line in enumerate(score_text.splitlines(), start=1):
    stripped = line.strip()

    if (
        stripped.startswith("def ")
        or stripped.startswith("class ")
        or stripped.startswith("import ")
        or stripped.startswith("from ")
    ):
        print(f"{i:04d}: {line}")

print("\n" + "=" * 70)
print("✅ CANDIDATE / SCORING INSPECTION COMPLETE")
print("=" * 70)

CANDIDATE GENERATOR
0001: from collections import defaultdict
0003: from normalize import normalize_business_name, normalize_address
0004: from data_loader import read_tsv
0011: def make_name_keys(name):
0084: def make_address_keys(address):
0198: def build_index(path, limit=None):
0268: def find_candidates(

CANDIDATE GENERATOR — FUNCTION HEADERS
0011: def make_name_keys(name):
0084: def make_address_keys(address):
0198: def build_index(path, limit=None):
0268: def find_candidates(

SCORING MODULE
0001: from candidate_generator import (
0006: from similarity import calculate_similarity

✅ CANDIDATE / SCORING INSPECTION COMPLETE


## 7. Inspect Candidate-Generation Implementation

This step examines the implementation of the two core candidate-generation functions:

- `build_index()` — builds the searchable index for Source 2 or Source 3.
- `find_candidates()` — retrieves candidate records for a Source 1 entity.

We will inspect only these functions and their surrounding code.

No large dataset is loaded and no index is built yet.

The purpose is to determine whether the existing implementation can be safely adapted for full-scale Kaggle execution without exceeding the memory or disk limits.

In [8]:
# ============================================================
# CELL 7 — INSPECT build_index() AND find_candidates()
# ============================================================

print("=" * 70)
print("BUILD_INDEX() IMPLEMENTATION")
print("=" * 70)

lines = candidate_text.splitlines()

start_build = next(
    i for i, line in enumerate(lines)
    if line.strip().startswith("def build_index")
)

start_find = next(
    i for i, line in enumerate(lines)
    if line.strip().startswith("def find_candidates")
)

# Print build_index() and the code immediately before find_candidates()
for i in range(start_build, start_find):
    print(f"{i + 1:04d}: {lines[i]}")

print("\n" + "=" * 70)
print("FIND_CANDIDATES() IMPLEMENTATION")
print("=" * 70)

# Print find_candidates() through the end of the file
for i in range(start_find, len(lines)):
    print(f"{i + 1:04d}: {lines[i]}")

print("\n" + "=" * 70)
print("✅ IMPLEMENTATION INSPECTION COMPLETE")
print("=" * 70)

BUILD_INDEX() IMPLEMENTATION
0198: def build_index(path, limit=None):
0199: 
0200:     name_index = defaultdict(list)
0201:     address_index = defaultdict(list)
0202: 
0203:     count = 0
0204: 
0205:     print(f"Building index from: {path}")
0206: 
0207:     for row in read_tsv(path):
0208: 
0209:         if limit is not None and count >= limit:
0210:             break
0211: 
0212:         entity_id = row.get("entity_id", "")
0213:         name = row.get("business_name", "")
0214:         address = row.get("business_address", "")
0215: 
0216:         country = row.get("country", "").strip().upper()
0217: 
0218:         # ----------------------------------------------------
0219:         # Name index
0220:         # ----------------------------------------------------
0221: 
0222:         for key_type, key_value in make_name_keys(name):
0223: 
0224:             full_key = (
0225:                 country,
0226:                 key_type,
0227:                 key_value
0228:            

## 8. Small-Scale Candidate Generation Benchmark

The existing candidate generator stores complete Source 2/Source 3 rows inside Python dictionary posting lists.

Because the full datasets contain millions of records, we will not build a full in-memory index at this stage.

Instead, this benchmark uses a small sample of the test data to measure:

- index construction time,
- number of generated index keys,
- candidate count per Source 1 record,
- memory usage.

This benchmark is intentionally small and is used only to determine whether the existing implementation can be safely reused or needs a memory-efficient replacement.

No full-scale processing is performed in this cell.

In [9]:
# ============================================================
# CELL 8 — SMALL CANDIDATE GENERATION BENCHMARK
# ============================================================

import sys
import time
import gc
import tracemalloc

# ------------------------------------------------------------
# Import the existing solution modules
# ------------------------------------------------------------

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_loader import read_tsv
from candidate_generator import build_index, find_candidates


# ------------------------------------------------------------
# VERY SMALL benchmark
# ------------------------------------------------------------

BENCH_S2_ROWS = 10_000
BENCH_S1_ROWS = 100

print("=" * 70)
print("SMALL CANDIDATE-GENERATION BENCHMARK")
print("=" * 70)

print(f"S2 rows used : {BENCH_S2_ROWS:,}")
print(f"S1 rows tested: {BENCH_S1_ROWS:,}")

gc.collect()

# ------------------------------------------------------------
# Build a small S2 index
# ------------------------------------------------------------

print("\nBuilding small S2 index...")

start = time.time()

tracemalloc.start()

name_index, address_index = build_index(
    str(TRAIN_S2),
    limit=BENCH_S2_ROWS
)

current_mem, peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

index_time = time.time() - start

print("\n" + "-" * 70)
print("INDEX BENCHMARK RESULT")
print("-" * 70)

print(f"Index time        : {index_time:.2f} seconds")
print(f"Name index keys   : {len(name_index):,}")
print(f"Address index keys: {len(address_index):,}")
print(
    f"Python traced peak: "
    f"{peak_mem / 1024**2:.1f} MB"
)


# ------------------------------------------------------------
# Test candidate generation on 100 S1 rows
# ------------------------------------------------------------

print("\nTesting candidate generation...")

s1_sample = []

for i, row in enumerate(read_tsv(str(TRAIN_S1))):

    s1_sample.append(row)

    if i + 1 >= BENCH_S1_ROWS:
        break

candidate_counts = []

start = time.time()

for row in s1_sample:

    candidates = find_candidates(
        row,
        name_index,
        address_index
    )

    candidate_counts.append(len(candidates))

candidate_time = time.time() - start


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("CANDIDATE BENCHMARK RESULT")
print("-" * 70)

print(f"S1 rows tested       : {len(candidate_counts):,}")
print(f"Candidate time       : {candidate_time:.3f} seconds")

print(
    f"Average candidates  : "
    f"{np.mean(candidate_counts):.2f}"
)

print(
    f"Median candidates   : "
    f"{np.median(candidate_counts):.2f}"
)

print(
    f"Maximum candidates  : "
    f"{np.max(candidate_counts):,}"
)

print(
    f"S1 rows with 0 cand.: "
    f"{sum(x == 0 for x in candidate_counts):,}"
)

print("\nCandidate count distribution:")
print(
    pd.Series(candidate_counts).describe()
)

print("\n" + "=" * 70)
print("✅ SMALL BENCHMARK COMPLETE")
print("=" * 70)

SMALL CANDIDATE-GENERATION BENCHMARK
S2 rows used : 10,000
S1 rows tested: 100

Building small S2 index...
Building index from: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
Finished indexing 10,000 records.
Name index keys    : 43,943
Address index keys : 24,902

----------------------------------------------------------------------
INDEX BENCHMARK RESULT
----------------------------------------------------------------------
Index time        : 3.17 seconds
Name index keys   : 43,943
Address index keys: 24,902
Python traced peak: 23.4 MB

Testing candidate generation...

----------------------------------------------------------------------
CANDIDATE BENCHMARK RESULT
----------------------------------------------------------------------
S1 rows tested       : 100
Candidate time       : 0.007 seconds
Average candidates  : 54.51
Median candidates   : 14.00
Maximum candidates  : 291
S1 rows with 0 cand.: 1

## 9. Candidate-Index Scaling Benchmark

The 10,000-row benchmark shows that the existing candidate generator is fast for small data.

However, the full competition Source 2 and Source 3 datasets contain millions of rows. The main concern is therefore the memory required by the Python posting-list index.

We will now repeat the benchmark with 100,000 Source 2 records.

This is still a controlled benchmark. We will measure:

- index construction time,
- number of index keys,
- Python memory usage,
- candidate counts.

We will **not** build the full multi-million-row index yet.

The result will determine whether the existing index can be reused directly or whether we need a more memory-efficient candidate-generation implementation.

In [10]:
import time
import gc
import tracemalloc

BENCH_S2_ROWS = 100_000
BENCH_S1_ROWS = 100

print("=" * 70)
print("100K CANDIDATE-INDEX SCALING BENCHMARK")
print("=" * 70)
print("S2 rows used :", BENCH_S2_ROWS)
print("S1 rows tested:", BENCH_S1_ROWS)

# Clean up anything from the previous benchmark
gc.collect()

print("\nBuilding 100K S2 index...")

tracemalloc.start()
start = time.time()

name_index_100k, address_index_100k = build_index(
    str(TRAIN_S2),
    limit=BENCH_S2_ROWS
)

index_time = time.time() - start

current_mem, peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("\n" + "-" * 70)
print("100K INDEX RESULT")
print("-" * 70)
print(f"Index time        : {index_time:.2f} seconds")
print(f"Name index keys   : {len(name_index_100k):,}")
print(f"Address index keys: {len(address_index_100k):,}")
print(f"Python traced peak: {peak_mem / 1024**2:.1f} MB")

# Reuse the same small S1 sample
candidate_counts_100k = []

print("\nTesting candidate generation...")

start = time.time()

for row in s1_sample:
    candidates = find_candidates(
        row,
        name_index_100k,
        address_index_100k
    )
    candidate_counts_100k.append(len(candidates))

candidate_time = time.time() - start

candidate_series = pd.Series(candidate_counts_100k)

print("\n" + "-" * 70)
print("100K CANDIDATE RESULT")
print("-" * 70)
print(f"S1 rows tested       : {len(s1_sample)}")
print(f"Candidate time       : {candidate_time:.3f} seconds")
print(f"Average candidates   : {candidate_series.mean():.2f}")
print(f"Median candidates    : {candidate_series.median():.2f}")
print(f"Maximum candidates   : {candidate_series.max()}")
print(f"S1 rows with 0 cand. : {(candidate_series == 0).sum()}")

print("\nCandidate count distribution:")
print(candidate_series.describe())

print("\n" + "=" * 70)
print("✅ 100K SCALING BENCHMARK COMPLETE")
print("=" * 70)

100K CANDIDATE-INDEX SCALING BENCHMARK
S2 rows used : 100000
S1 rows tested: 100

Building 100K S2 index...
Building index from: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
Finished indexing 100,000 records.
Name index keys    : 348,248
Address index keys : 216,462

----------------------------------------------------------------------
100K INDEX RESULT
----------------------------------------------------------------------
Index time        : 31.26 seconds
Name index keys   : 348,248
Address index keys: 216,462
Python traced peak: 177.8 MB

Testing candidate generation...

----------------------------------------------------------------------
100K CANDIDATE RESULT
----------------------------------------------------------------------
S1 rows tested       : 100
Candidate time       : 0.048 seconds
Average candidates   : 548.61
Median candidates    : 130.50
Maximum candidates   : 2850
S1 rows with 0 cand

## 10. Inspect the Existing Blocking Keys

The scaling benchmark shows that the current implementation is not suitable for direct full-scale indexing.

Before replacing it, we will inspect exactly which blocking keys the existing implementation generates.

This allows us to preserve the useful matching logic while designing a more selective and memory-efficient candidate generator.

This cell only tests the blocking-key functions on a few example records. It does not load or index the full datasets.

In [11]:
# Inspect the blocking keys used by the existing implementation

from candidate_generator import make_name_keys, make_address_keys

# Take a few real S1 examples
example_rows = []

for i, row in enumerate(read_tsv(str(TRAIN_S1))):
    example_rows.append(row)
    if i >= 4:
        break

print("=" * 70)
print("EXISTING BLOCKING KEY INSPECTION")
print("=" * 70)

for i, row in enumerate(example_rows, 1):

    print(f"\n{'-' * 70}")
    print(f"Example S1 record #{i}")
    print(f"Entity ID : {row.get('entity_id')}")
    print(f"Name     : {row.get('business_name')}")
    print(f"Address  : {row.get('business_address')}")
    print(f"Country  : {row.get('country')}")

    name_keys = make_name_keys(row.get("business_name", ""))
    address_keys = make_address_keys(row.get("business_address", ""))

    print("\nName blocking keys:")
    for key in name_keys:
        print("  ", repr(key))

    print("\nAddress blocking keys:")
    for key in address_keys:
        print("  ", repr(key))

print("\n" + "=" * 70)
print("✅ BLOCKING KEY INSPECTION COMPLETE")
print("=" * 70)

EXISTING BLOCKING KEY INSPECTION

----------------------------------------------------------------------
Example S1 record #1
Entity ID : S1-925783039
Name     : Orelee's Barbershop
Address  : 1795 Westchester Drive, High Point, NC
Country  : US

Name blocking keys:
   ('NP6', 'orelee')
   ('NP8', 'oreleesb')
   ('NS6', 'ershop')
   ('FL', 'orel|barb')
   ('FL3', 'ore|barbe')
   ('FML', 'ore|s|bar')

Address blocking keys:
   ('AN', '1795|westch')
   ('A2', '1795|west|dr')

----------------------------------------------------------------------
Example S1 record #2
Entity ID : S1-773889195
Name     : Prime Money
Address  : 17560 Ellis Road, Tahlequah, OK
Country  : US

Name blocking keys:
   ('NP6', 'primem')
   ('NP8', 'primemon')
   ('NS6', 'emoney')
   ('FL', 'prim|mone')
   ('FL3', 'pri|money')
   ('F2', 'prime|money')

Address blocking keys:
   ('AN', '17560|ellis')
   ('A2', '17560|elli|rd')

----------------------------------------------------------------------
Example S1 record 

## 11. Measure Blocking-Key Selectivity

The existing blocker creates several types of name and address keys.

Some keys are highly specific, while others may match a large number of businesses.

Because candidate-generation quality depends heavily on blocking selectivity, we will measure how frequently each key type occurs in a controlled 100,000-row Source 2 sample.

For each key type we will record:

- number of unique keys,
- average posting-list size,
- maximum posting-list size,
- number of keys occurring only once,
- number of very frequent keys.

This will show which blocking keys should be retained, capped, or removed in the final candidate generator.

Only 100,000 Source 2 records are processed.

In [12]:
from collections import defaultdict, Counter
import time
import gc

SEL_ROWS = 100_000

name_counts = defaultdict(int)
address_counts = defaultdict(int)

print("=" * 70)
print("BLOCKING-KEY SELECTIVITY ANALYSIS")
print("=" * 70)
print("Source 2 rows analyzed:", f"{SEL_ROWS:,}")

start = time.time()

for i, row in enumerate(read_tsv(str(TRAIN_S2))):

    if i >= SEL_ROWS:
        break

    country = row.get("country", "")

    # Name keys
    for key_type, key_value in make_name_keys(
        row.get("business_name", "")
    ):
        name_counts[(country, key_type, key_value)] += 1

    # Address keys
    for key_type, key_value in make_address_keys(
        row.get("business_address", "")
    ):
        address_counts[(country, key_type, key_value)] += 1

elapsed = time.time() - start

print(f"\nScan time: {elapsed:.2f} seconds")

def summarize_counts(counts, title):
    values = np.fromiter(counts.values(), dtype=np.int32)

    print("\n" + "-" * 70)
    print(title)
    print("-" * 70)

    print("Unique keys :", f"{len(values):,}")
    print("Mean freq   :", f"{values.mean():.2f}")
    print("Median freq :", f"{np.median(values):.2f}")
    print("Max freq    :", int(values.max()))
    print("Singletons  :", int((values == 1).sum()))
    print("Freq >= 10  :", int((values >= 10).sum()))
    print("Freq >= 100 :", int((values >= 100).sum()))
    print("Freq >= 1000:", int((values >= 1000).sum()))

    # Most frequent keys
    top = sorted(
        counts.items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]

    print("\nTop 10 most frequent keys:")
    for (country, key_type, key_value), freq in top:
        print(
            f"  {key_type:5s} | {country:8s} | "
            f"{freq:6d} | {key_value}"
        )

    return values


name_freq = summarize_counts(
    name_counts,
    "NAME BLOCKING KEYS"
)

address_freq = summarize_counts(
    address_counts,
    "ADDRESS BLOCKING KEYS"
)

print("\n" + "=" * 70)
print("KEY-TYPE BREAKDOWN")
print("=" * 70)

name_type_counts = Counter()
address_type_counts = Counter()

for (country, key_type, key_value), freq in name_counts.items():
    name_type_counts[key_type] += 1

for (country, key_type, key_value), freq in address_counts.items():
    address_type_counts[key_type] += 1

print("\nName key types:")
for k, v in sorted(name_type_counts.items()):
    print(f"  {k:5s}: {v:,}")

print("\nAddress key types:")
for k, v in sorted(address_type_counts.items()):
    print(f"  {k:5s}: {v:,}")

print("\n" + "=" * 70)
print("✅ SELECTIVITY ANALYSIS COMPLETE")
print("=" * 70)

BLOCKING-KEY SELECTIVITY ANALYSIS
Source 2 rows analyzed: 100,000

Scan time: 5.15 seconds

----------------------------------------------------------------------
NAME BLOCKING KEYS
----------------------------------------------------------------------
Unique keys : 348,248
Mean freq   : 1.58
Median freq : 1.00
Max freq    : 4252
Singletons  : 302741
Freq >= 10  : 3377
Freq >= 100 : 257
Freq >= 1000: 7

Top 10 most frequent keys:
  NS6   | India    |   4252 | rivate
  NS6   | US       |   2223 | center
  NS6   | US       |   2150 | rtners
  NS6   | US       |   1374 | rvices
  NS6   | US       |   1116 | ldings
  NS6   | India    |   1062 | rvices
  NS6   | India    |   1036 | center
  NS6   | US       |    837 | ciates
  NP6   | India    |    823 | privat
  NP6   | US       |    619 | pediat

----------------------------------------------------------------------
ADDRESS BLOCKING KEYS
----------------------------------------------------------------------
Unique keys : 216,462
Mean freq

## 12. Selective Blocking Threshold Analysis

The previous analysis shows that most blocking keys are rare, while a small number of keys are extremely frequent.

Very frequent keys are undesirable because they create large candidate buckets and contribute little discrimination.

We will therefore evaluate several posting-list frequency thresholds.

For each threshold we measure:

- percentage of keys retained,
- percentage of key occurrences retained,
- maximum allowed posting-list size.

The goal is to choose a conservative threshold that reduces candidate explosion without unnecessarily removing rare, useful blocking keys.

This analysis uses the 100,000-row benchmark statistics already collected in memory.

In [13]:
thresholds = [5, 10, 20, 50, 100, 200, 500, 1000]

print("=" * 70)
print("BLOCKING FREQUENCY THRESHOLD ANALYSIS")
print("=" * 70)

def threshold_report(counts, title):
    frequencies = np.fromiter(
        counts.values(),
        dtype=np.int32
    )

    total_occurrences = frequencies.sum()
    total_keys = len(frequencies)

    print("\n" + "-" * 70)
    print(title)
    print("-" * 70)

    print(
        f"{'Threshold':>10} "
        f"{'Keys kept':>12} "
        f"{'% keys':>10} "
        f"{'Occurrences kept':>18} "
        f"{'% occurrences':>15}"
    )

    for threshold in thresholds:

        mask = frequencies <= threshold

        keys_kept = int(mask.sum())
        occurrences_kept = int(frequencies[mask].sum())

        key_pct = 100 * keys_kept / total_keys
        occurrence_pct = 100 * occurrences_kept / total_occurrences

        print(
            f"{threshold:>10} "
            f"{keys_kept:>12,} "
            f"{key_pct:>9.2f}% "
            f"{occurrences_kept:>18,} "
            f"{occurrence_pct:>14.2f}%"
        )

threshold_report(
    name_counts,
    "NAME KEYS"
)

threshold_report(
    address_counts,
    "ADDRESS KEYS"
)

print("\n" + "=" * 70)
print("✅ THRESHOLD ANALYSIS COMPLETE")
print("=" * 70)

BLOCKING FREQUENCY THRESHOLD ANALYSIS

----------------------------------------------------------------------
NAME KEYS
----------------------------------------------------------------------
 Threshold    Keys kept     % keys   Occurrences kept   % occurrences
         5      341,548     98.08%            401,766          73.00%
        10      345,284     99.15%            429,618          78.06%
        20      347,062     99.66%            454,781          82.63%
        50      347,759     99.86%            476,063          86.49%
       100      347,993     99.93%            492,766          89.53%
       200      348,190     99.98%            520,493          94.57%
       500      348,234    100.00%            532,651          96.77%
      1000      348,241    100.00%            537,189          97.60%

----------------------------------------------------------------------
ADDRESS KEYS
----------------------------------------------------------------------
 Threshold    Keys kept

## 13. Validate Selective Blocking Against Ground Truth

Frequency capping reduces candidate explosion, but it must not remove true matches.

We will therefore test the proposed frequency threshold of 20 against the available training ground truth.

For the same 100 Source 1 records used in the earlier benchmark:

1. Generate candidates using the original blocking index.
2. Generate candidates using a frequency-capped index.
3. Compare the candidate sets against the known ground-truth matches that are present in the 100,000-row Source 2 sample.

The primary measurement is candidate recall:

    candidate recall =
    ground-truth matches found in candidates /
    ground-truth matches available in the benchmark subset

This is a diagnostic benchmark only. It does not modify the competition data.

In [14]:
from collections import defaultdict

CAP = 20

print("=" * 70)
print("SELECTIVE BLOCKING RECALL TEST")
print("=" * 70)
print("S2 benchmark rows :", f"{BENCH_S2_ROWS:,}")
print("S1 benchmark rows :", len(s1_sample))
print("Frequency cap     :", CAP)

# ------------------------------------------------------------------
# 1. Build a frequency-capped index from the 100K S2 sample
# ------------------------------------------------------------------

print("\nBuilding capped index...")

# First pass: count postings
name_freq_100k = defaultdict(int)
address_freq_100k = defaultdict(int)

for i, row in enumerate(read_tsv(str(TRAIN_S2))):
    if i >= BENCH_S2_ROWS:
        break

    country = row.get("country", "")

    for key_type, key_value in make_name_keys(
        row.get("business_name", "")
    ):
        name_freq_100k[(country, key_type, key_value)] += 1

    for key_type, key_value in make_address_keys(
        row.get("business_address", "")
    ):
        address_freq_100k[(country, key_type, key_value)] += 1


# Second pass: store only postings whose frequency <= CAP
capped_name_index = defaultdict(list)
capped_address_index = defaultdict(list)

for i, row in enumerate(read_tsv(str(TRAIN_S2))):
    if i >= BENCH_S2_ROWS:
        break

    country = row.get("country", "")
    entity_id = row.get("entity_id")

    for key_type, key_value in make_name_keys(
        row.get("business_name", "")
    ):
        key = (country, key_type, key_value)

        if name_freq_100k[key] <= CAP:
            capped_name_index[key].append(row)

    for key_type, key_value in make_address_keys(
        row.get("business_address", "")
    ):
        key = (country, key_type, key_value)

        if address_freq_100k[key] <= CAP:
            capped_address_index[key].append(row)


print("Capped name keys   :", f"{len(capped_name_index):,}")
print("Capped address keys:", f"{len(capped_address_index):,}")


# ------------------------------------------------------------------
# 2. Candidate function using the capped index
# ------------------------------------------------------------------

def find_capped_candidates(row):
    candidates = {}

    country = row.get("country", "")

    for key_type, key_value in make_name_keys(
        row.get("business_name", "")
    ):
        key = (country, key_type, key_value)

        for candidate in capped_name_index.get(key, []):
            candidates[candidate["entity_id"]] = candidate

    for key_type, key_value in make_address_keys(
        row.get("business_address", "")
    ):
        key = (country, key_type, key_value)

        for candidate in capped_address_index.get(key, []):
            candidates[candidate["entity_id"]] = candidate

    return list(candidates.values())


# ------------------------------------------------------------------
# 3. Load ground truth only for these 100 S1 IDs
# ------------------------------------------------------------------

sample_s1_ids = {
    row["entity_id"]
    for row in s1_sample
}

ground_truth = {}

for row in read_tsv(str(TRAIN_GT)):

    s1_id = row["source1_entity_id"]

    if s1_id in sample_s1_ids:
        value = row["matched_entity_ids"]

        if pd.isna(value) or str(value).strip() == "":
            ground_truth[s1_id] = set()
        else:
            ground_truth[s1_id] = {
                x.strip()
                for x in str(value).split(",")
                if x.strip()
            }


# ------------------------------------------------------------------
# 4. Evaluate original vs capped candidate recall
# ------------------------------------------------------------------

original_found = 0
capped_found = 0
available_gt = 0

original_counts = []
capped_counts = []

lost_matches = []

for row in s1_sample:

    s1_id = row["entity_id"]

    # Original
    original_candidates = find_candidates(
        row,
        name_index_100k,
        address_index_100k
    )

    original_ids = {
        x["entity_id"]
        for x in original_candidates
    }

    # Capped
    capped_candidates = find_capped_candidates(row)

    capped_ids = {
        x["entity_id"]
        for x in capped_candidates
    }

    gt_ids = ground_truth.get(s1_id, set())

    # Only GT records actually present in the 100K S2 benchmark
    gt_available = {
        x for x in gt_ids
        if x in original_ids
    }

    available_gt += len(gt_available)

    original_found += len(
        gt_available & original_ids
    )

    capped_found += len(
        gt_available & capped_ids
    )

    original_counts.append(len(original_ids))
    capped_counts.append(len(capped_ids))

    lost = gt_available - capped_ids

    if lost:
        lost_matches.append(
            (s1_id, lost)
        )


# ------------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------------

print("\n" + "=" * 70)
print("SELECTIVE BLOCKING RESULTS")
print("=" * 70)

print(f"Ground-truth pairs available in 100K S2 : {available_gt:,}")
print(f"Found by original blocker              : {original_found:,}")
print(f"Found by capped blocker                : {capped_found:,}")

if available_gt > 0:
    print(
        f"Original candidate recall             : "
        f"{original_found / available_gt:.4%}"
    )

    print(
        f"Capped candidate recall               : "
        f"{capped_found / available_gt:.4%}"
    )

print("\nCandidate counts:")
print(
    f"Original average : "
    f"{np.mean(original_counts):.2f}"
)

print(
    f"Capped average   : "
    f"{np.mean(capped_counts):.2f}"
)

print(
    f"Original median  : "
    f"{np.median(original_counts):.2f}"
)

print(
    f"Capped median    : "
    f"{np.median(capped_counts):.2f}"
)

print(
    f"GT matches lost by cap : "
    f"{sum(len(x[1]) for x in lost_matches):,}"
)

if lost_matches:
    print("\nExamples of lost GT matches:")
    for s1_id, lost in lost_matches[:10]:
        print(" ", s1_id, "->", list(lost)[:5])

print("\n" + "=" * 70)
print("✅ SELECTIVE BLOCKING TEST COMPLETE")
print("=" * 70)

SELECTIVE BLOCKING RECALL TEST
S2 benchmark rows : 100,000
S1 benchmark rows : 100
Frequency cap     : 20

Building capped index...
Capped name keys   : 347,062
Capped address keys: 216,065

SELECTIVE BLOCKING RESULTS
Ground-truth pairs available in 100K S2 : 3
Found by original blocker              : 3
Found by capped blocker                : 2
Original candidate recall             : 100.0000%
Capped candidate recall               : 66.6667%

Candidate counts:
Original average : 548.61
Capped average   : 9.47
Original median  : 130.50
Capped median    : 6.50
GT matches lost by cap : 1

Examples of lost GT matches:
  S1-504790211 -> ['S2-340882825']

✅ SELECTIVE BLOCKING TEST COMPLETE


## 14. Analyze the Lost Ground-Truth Match

The frequency-cap experiment showed that a hard threshold of 20 removes some true matches.

One known ground-truth pair was lost:

    S1-504790211 → S2-340882825

The purpose of this cell is to inspect the blocking keys generated by both records.

We will determine:

- which blocking keys the Source 1 record produces,
- which blocking keys the true Source 2 record produces,
- which key caused the match to be removed by the frequency cap,
- and how frequent that key is.

This will help us design a selective blocking rule that does not simply discard frequent keys.

Only the two relevant records are inspected.

In [15]:
TARGET_S1 = "S1-504790211"
TARGET_S2 = "S2-340882825"

print("=" * 70)
print("LOST MATCH ANALYSIS")
print("=" * 70)

s1_target = None
s2_target = None

# Find S1 record
for row in read_tsv(str(TRAIN_S1)):
    if row["entity_id"] == TARGET_S1:
        s1_target = row
        break

# Find S2 record
for row in read_tsv(str(TRAIN_S2)):
    if row["entity_id"] == TARGET_S2:
        s2_target = row
        break

assert s1_target is not None, "Target S1 not found"
assert s2_target is not None, "Target S2 not found"

print("\nSOURCE 1")
print("-" * 70)
print("Entity ID :", s1_target["entity_id"])
print("Name      :", s1_target["business_name"])
print("Address   :", s1_target["business_address"])
print("Country   :", s1_target["country"])

print("\nSOURCE 2")
print("-" * 70)
print("Entity ID :", s2_target["entity_id"])
print("Name      :", s2_target["business_name"])
print("Address   :", s2_target["business_address"])
print("Country   :", s2_target["country"])


# ---------------------------------------------------------------
# Generate blocking keys
# ---------------------------------------------------------------

s1_name_keys = make_name_keys(
    s1_target.get("business_name", "")
)

s1_address_keys = make_address_keys(
    s1_target.get("business_address", "")
)

s2_name_keys = make_name_keys(
    s2_target.get("business_name", "")
)

s2_address_keys = make_address_keys(
    s2_target.get("business_address", "")
)


print("\nSOURCE 1 NAME KEYS")
for key in s1_name_keys:
    print(" ", key)

print("\nSOURCE 2 NAME KEYS")
for key in s2_name_keys:
    print(" ", key)

print("\nSOURCE 1 ADDRESS KEYS")
for key in s1_address_keys:
    print(" ", key)

print("\nSOURCE 2 ADDRESS KEYS")
for key in s2_address_keys:
    print(" ", key)


# ---------------------------------------------------------------
# Find exact shared keys
# ---------------------------------------------------------------

s1_name_set = set(s1_name_keys)
s2_name_set = set(s2_name_keys)

s1_addr_set = set(s1_address_keys)
s2_addr_set = set(s2_address_keys)

shared_name = s1_name_set & s2_name_set
shared_address = s1_addr_set & s2_addr_set

print("\n" + "=" * 70)
print("SHARED BLOCKING KEYS")
print("=" * 70)

print("\nShared name keys:")
for key in shared_name:
    freq = name_freq_100k.get(key, 0)
    print(f"  {key}  -> frequency in 100K S2 = {freq}")

print("\nShared address keys:")
for key in shared_address:
    freq = address_freq_100k.get(key, 0)
    print(f"  {key}  -> frequency in 100K S2 = {freq}")

print("\n" + "=" * 70)
print("✅ LOST MATCH INSPECTION COMPLETE")
print("=" * 70)

LOST MATCH ANALYSIS

SOURCE 1
----------------------------------------------------------------------
Entity ID : S1-504790211
Name      : Crystal Lending PC
Address   : 11643 Prosperity Road, South Jordan, UT
Country   : US

SOURCE 2
----------------------------------------------------------------------
Entity ID : S2-340882825
Name      : Crystal PC Lending
Address   : 
Country   : US

SOURCE 1 NAME KEYS
  ('NP6', 'crysta')
  ('NP8', 'crystall')
  ('NS6', 'dingpc')
  ('F2', 'cryst|lendi')

SOURCE 2 NAME KEYS
  ('NP6', 'crysta')
  ('NP8', 'crystalp')
  ('NS6', 'ending')
  ('FL', 'crys|lend')
  ('FL3', 'cry|lendi')
  ('FML', 'cry|pc|len')

SOURCE 1 ADDRESS KEYS
  ('AN', '11643|prospe')
  ('A2', '11643|pros|rd')

SOURCE 2 ADDRESS KEYS

SHARED BLOCKING KEYS

Shared name keys:
  ('NP6', 'crysta')  -> frequency in 100K S2 = 0

Shared address keys:

✅ LOST MATCH INSPECTION COMPLETE


## 15. Diagnose the Capped-Blocking Recall Discrepancy

The lost-match inspection shows that the true S1/S2 pair shares the highly specific name key:

    ('NP6', 'crysta')

However, the frequency table reported zero occurrences for that key.

This is inconsistent with the fact that the original candidate generator found the true S2 record.

Before changing the blocking strategy, we will directly inspect the original and capped indexes.

This diagnostic checks:

- the exact index key,
- the number of records stored under that key,
- whether the true S2 entity is present,
- and whether the capped index removed it.

No large dataset processing is performed.

In [16]:
print("=" * 70)
print("DIAGNOSTIC: LOST MATCH / CAPPED INDEX")
print("=" * 70)

# Exact key from the two records
TARGET_KEY = ("US", "NP6", "crysta")

print("\nExact index key:")
print(TARGET_KEY)

# ---------------------------------------------------------------
# Original index
# ---------------------------------------------------------------

original_bucket = name_index_100k.get(TARGET_KEY, [])

print("\nORIGINAL INDEX")
print("-" * 70)
print("Bucket size:", len(original_bucket))

target_in_original = any(
    row.get("entity_id") == TARGET_S2
    for row in original_bucket
)

print("Target S2 in original bucket:", target_in_original)

if original_bucket:
    print("\nFirst 10 entity IDs:")
    for row in original_bucket[:10]:
        print(" ", row.get("entity_id"))


# ---------------------------------------------------------------
# Capped index
# ---------------------------------------------------------------

capped_bucket = capped_name_index.get(TARGET_KEY, [])

print("\nCAPPED INDEX")
print("-" * 70)
print("Bucket size:", len(capped_bucket))

target_in_capped = any(
    row.get("entity_id") == TARGET_S2
    for row in capped_bucket
)

print("Target S2 in capped bucket:", target_in_capped)


# ---------------------------------------------------------------
# Frequency dictionaries
# ---------------------------------------------------------------

print("\nFREQUENCY DICTIONARIES")
print("-" * 70)

print(
    "name_freq_100k[TARGET_KEY] =",
    name_freq_100k.get(TARGET_KEY, 0)
)

print(
    "name_freq_100k keys matching NP6/crysta:"
)

matching_freq_keys = [
    (k, v)
    for k, v in name_freq_100k.items()
    if k[1] == "NP6" and k[2] == "crysta"
]

for k, v in matching_freq_keys:
    print(" ", k, "->", v)

# ---------------------------------------------------------------
# Check all country variants
# ---------------------------------------------------------------

print("\nCOUNTRY VARIANTS FOR NP6/crysta")
print("-" * 70)

for k, v in name_freq_100k.items():
    if k[1] == "NP6" and k[2] == "crysta":
        print(
            "country =", repr(k[0]),
            "| frequency =", v
        )

# ---------------------------------------------------------------
# Final diagnosis
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("DIAGNOSTIC RESULT")
print("=" * 70)

if target_in_original and target_in_capped:
    print("The target exists in BOTH indexes.")
    print("The earlier recall-loss calculation needs correction.")

elif target_in_original and not target_in_capped:
    print("The frequency cap genuinely removed the target.")
    print("We need a safer selective-blocking rule.")

else:
    print("The target is not present in the expected original bucket.")
    print("The earlier benchmark needs further investigation.")

print("=" * 70)

DIAGNOSTIC: LOST MATCH / CAPPED INDEX

Exact index key:
('US', 'NP6', 'crysta')

ORIGINAL INDEX
----------------------------------------------------------------------
Bucket size: 102
Target S2 in original bucket: True

First 10 entity IDs:
  S2-402816908
  S2-779405377
  S2-337462823
  S2-340882825
  S2-352561696
  S2-808728262
  S2-776356778
  S2-954715854
  S2-846238992
  S2-837266835

CAPPED INDEX
----------------------------------------------------------------------
Bucket size: 0
Target S2 in capped bucket: False

FREQUENCY DICTIONARIES
----------------------------------------------------------------------
name_freq_100k[TARGET_KEY] = 102
name_freq_100k keys matching NP6/crysta:
  ('US', 'NP6', 'crysta') -> 102
  ('India', 'NP6', 'crysta') -> 10

COUNTRY VARIANTS FOR NP6/crysta
----------------------------------------------------------------------
country = 'US' | frequency = 102
country = 'India' | frequency = 10

DIAGNOSTIC RESULT
The frequency cap genuinely removed the target.

## 16. Test Multi-Key Blocking

A frequent blocking key can still be useful when another blocking key provides independent evidence.

For example, the true pair:

    Crystal Lending PC
    Crystal PC Lending

shares the `NP6` key `crysta`, but that key occurs 102 times in the 100K Source 2 sample.

Instead of discarding this key, we will test a two-level strategy:

1. Generate candidates from blocking keys.
2. Count how many independent blocking keys each candidate shares with the Source 1 record.
3. Keep candidates that share:
   - at least one rare key, OR
   - at least two blocking keys overall.

This cell evaluates the idea on the same 100 Source 1 benchmark records.

It does not process the full competition dataset.

In [17]:
print("=" * 70)
print("MULTI-KEY BLOCKING BENCHMARK")
print("=" * 70)

# ------------------------------------------------------------------
# Build a capped/frequency-aware index.
#
# We keep every key in the index, but we will NOT automatically
# accept every record from a frequent bucket.
# ------------------------------------------------------------------

MULTI_KEY_CAP = 100

print("Frequent-key threshold:", MULTI_KEY_CAP)

# Build compact posting lists containing rows
# (still only on the 100K benchmark)

multi_name_index = defaultdict(list)
multi_address_index = defaultdict(list)

for i, row in enumerate(read_tsv(str(TRAIN_S2))):
    if i >= BENCH_S2_ROWS:
        break

    country = row.get("country", "")
    
    for key_type, key_value in make_name_keys(
        row.get("business_name", "")
    ):
        multi_name_index[
            (country, key_type, key_value)
        ].append(row)

    for key_type, key_value in make_address_keys(
        row.get("business_address", "")
    ):
        multi_address_index[
            (country, key_type, key_value)
        ].append(row)


# ------------------------------------------------------------------
# Candidate generation using shared-key counts
# ------------------------------------------------------------------

def find_multikey_candidates(row):
    """
    Return candidates together with the number of independent
    blocking keys shared with the S1 record.
    """

    candidate_key_hits = defaultdict(int)
    candidate_rows = {}

    country = row.get("country", "")

    # Name keys
    for key_type, key_value in make_name_keys(
        row.get("business_name", "")
    ):
        key = (country, key_type, key_value)
        bucket = multi_name_index.get(key, [])

        for candidate in bucket:

            cid = candidate["entity_id"]

            candidate_key_hits[cid] += 1
            candidate_rows[cid] = candidate

    # Address keys
    for key_type, key_value in make_address_keys(
        row.get("business_address", "")
    ):
        key = (country, key_type, key_value)
        bucket = multi_address_index.get(key, [])

        for candidate in bucket:

            cid = candidate["entity_id"]

            candidate_key_hits[cid] += 1
            candidate_rows[cid] = candidate

    return candidate_rows, candidate_key_hits


# ------------------------------------------------------------------
# Evaluate candidate counts under different shared-key requirements
# ------------------------------------------------------------------

candidate_counts_1 = []
candidate_counts_2 = []
candidate_counts_3 = []

gt_available = 0
gt_found_1 = 0
gt_found_2 = 0
gt_found_3 = 0

for row in s1_sample:

    s1_id = row["entity_id"]

    candidate_rows, key_hits = find_multikey_candidates(row)

    # Keep if at least N independent blocking keys match
    ids_1 = {
        cid for cid, hits in key_hits.items()
        if hits >= 1
    }

    ids_2 = {
        cid for cid, hits in key_hits.items()
        if hits >= 2
    }

    ids_3 = {
        cid for cid, hits in key_hits.items()
        if hits >= 3
    }

    candidate_counts_1.append(len(ids_1))
    candidate_counts_2.append(len(ids_2))
    candidate_counts_3.append(len(ids_3))

    gt_ids = ground_truth.get(s1_id, set())

    # Only GT records present in the 100K S2 sample
    gt_1 = gt_ids & ids_1

    gt_available += len(gt_1)
    gt_found_1 += len(gt_1)
    gt_found_2 += len(gt_ids & ids_2)
    gt_found_3 += len(gt_ids & ids_3)


# ------------------------------------------------------------------
# Report
# ------------------------------------------------------------------

print("\n" + "=" * 70)
print("MULTI-KEY RESULTS")
print("=" * 70)

print(
    f"{'Requirement':<22}"
    f"{'Avg candidates':>18}"
    f"{'Median':>14}"
    f"{'GT found':>14}"
)

print("-" * 70)

print(
    f"{'≥ 1 shared key':<22}"
    f"{np.mean(candidate_counts_1):>18.2f}"
    f"{np.median(candidate_counts_1):>14.2f}"
    f"{gt_found_1:>14}"
)

print(
    f"{'≥ 2 shared keys':<22}"
    f"{np.mean(candidate_counts_2):>18.2f}"
    f"{np.median(candidate_counts_2):>14.2f}"
    f"{gt_found_2:>14}"
)

print(
    f"{'≥ 3 shared keys':<22}"
    f"{np.mean(candidate_counts_3):>18.2f}"
    f"{np.median(candidate_counts_3):>14.2f}"
    f"{gt_found_3:>14}"
)

print("\nGround-truth pairs available:", gt_available)

if gt_available:
    print(
        f"Recall ≥1 key: "
        f"{gt_found_1 / gt_available:.4%}"
    )

    print(
        f"Recall ≥2 keys: "
        f"{gt_found_2 / gt_available:.4%}"
    )

    print(
        f"Recall ≥3 keys: "
        f"{gt_found_3 / gt_available:.4%}"
    )

print("\n" + "=" * 70)
print("✅ MULTI-KEY BLOCKING TEST COMPLETE")
print("=" * 70)

MULTI-KEY BLOCKING BENCHMARK
Frequent-key threshold: 100

MULTI-KEY RESULTS
Requirement               Avg candidates        Median      GT found
----------------------------------------------------------------------
≥ 1 shared key                    548.61        130.50             3
≥ 2 shared keys                    23.51          2.00             2
≥ 3 shared keys                     9.41          0.00             2

Ground-truth pairs available: 3
Recall ≥1 key: 100.0000%
Recall ≥2 keys: 66.6667%
Recall ≥3 keys: 66.6667%

✅ MULTI-KEY BLOCKING TEST COMPLETE


## 17. Build an ID-Aligned Validation Set

The previous 100-S1 benchmark contains only three ground-truth matches that are
actually present in the 100,000-row Source 2 sample.

That is too small to make a reliable blocking decision.

We will therefore create a larger validation subset using the training ground truth
itself.

For each selected Source 1 record, we will identify its true Source 2 IDs and then
check whether those true IDs are available in a controlled Source 2 sample.

This keeps the validation aligned with the actual entity-resolution task instead
of relying on an arbitrary first-N row subset.

No full-scale indexing is performed.

In [18]:
print("=" * 70)
print("ID-ALIGNED VALIDATION SET")
print("=" * 70)

# ---------------------------------------------------------------
# Load the first 100K S2 IDs into a set
# ---------------------------------------------------------------

print("\nCollecting 100K Source 2 IDs...")

s2_100k_ids = set()

for i, row in enumerate(read_tsv(str(TRAIN_S2))):
    if i >= 100_000:
        break

    s2_100k_ids.add(row["entity_id"])

print("S2 IDs collected:", f"{len(s2_100k_ids):,}")


# ---------------------------------------------------------------
# Scan ground truth and find S1 records having GT matches
# inside the 100K S2 sample.
# ---------------------------------------------------------------

VALIDATION_SIZE = 2_000

aligned_s1_ids = []
aligned_gt = {}

print("\nFinding ID-aligned S1 records...")

for row in read_tsv(str(TRAIN_GT)):

    if len(aligned_s1_ids) >= VALIDATION_SIZE:
        break

    s1_id = row["source1_entity_id"]

    value = row["matched_entity_ids"]

    if pd.isna(value) or str(value).strip() == "":
        continue

    gt_ids = {
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    }

    # Keep only GT IDs actually available in our 100K S2 sample
    available = gt_ids & s2_100k_ids

    if available:
        aligned_s1_ids.append(s1_id)
        aligned_gt[s1_id] = available


print("\n" + "-" * 70)
print("VALIDATION SET")
print("-" * 70)

print("S1 validation records :", f"{len(aligned_s1_ids):,}")

total_pairs = sum(
    len(v) for v in aligned_gt.values()
)

print("Available GT pairs     :", f"{total_pairs:,}")

print(
    "Average GT pairs/S1    :",
    f"{total_pairs / max(len(aligned_s1_ids), 1):.2f}"
)

print("\nFirst 10 examples:")

for s1_id in aligned_s1_ids[:10]:
    print(
        " ",
        s1_id,
        "->",
        list(aligned_gt[s1_id])[:10]
    )

print("\n" + "=" * 70)
print("✅ ID-ALIGNED VALIDATION SET CREATED")
print("=" * 70)

ID-ALIGNED VALIDATION SET

S2 IDs collected: 100,000

Finding ID-aligned S1 records...

----------------------------------------------------------------------
VALIDATION SET
----------------------------------------------------------------------
S1 validation records : 2,000
Available GT pairs     : 2,033
Average GT pairs/S1    : 1.02

First 10 examples:
  S1-244810289 -> ['S2-63598201']
  S1-292900819 -> ['S2-791548670']
  S1-307962633 -> ['S2-22544434']
  S1-996012175 -> ['S2-269681023']
  S1-897282581 -> ['S2-522109781']
  S1-357837082 -> ['S2-65680643']
  S1-776686932 -> ['S2-667655982']
  S1-658984896 -> ['S2-696163364']
  S1-365033919 -> ['S2-118656392']
  S1-595843047 -> ['S2-182431985']

✅ ID-ALIGNED VALIDATION SET CREATED


## 18. Benchmark the Original Blocker on the ID-Aligned Validation Set

We now have 2,000 Source 1 records with 2,033 known Source 2 matches
available inside the 100,000-row Source 2 benchmark.

This provides a meaningful validation set for candidate generation.

First, we measure the original blocking strategy without any frequency
restriction.

For every validation S1 record we calculate:

- number of generated candidates,
- whether each known ground-truth S2 entity was retrieved,
- overall candidate recall.

This establishes the baseline that every improved blocking strategy must
match or improve.

In [19]:
print("=" * 70)
print("ORIGINAL BLOCKER — ID-ALIGNED VALIDATION")
print("=" * 70)

# ---------------------------------------------------------------
# Build lookup for the 100K S2 benchmark
# ---------------------------------------------------------------

print("\nLoading 100K S2 records into an ID lookup...")

s2_lookup_100k = {}

for i, row in enumerate(read_tsv(str(TRAIN_S2))):
    if i >= 100_000:
        break

    s2_lookup_100k[row["entity_id"]] = row

print("S2 records loaded:", f"{len(s2_lookup_100k):,}")


# ---------------------------------------------------------------
# Load the 2,000 aligned S1 records
# ---------------------------------------------------------------

print("\nLoading validation S1 records...")

validation_s1 = {}

for row in read_tsv(str(TRAIN_S1)):

    s1_id = row["entity_id"]

    if s1_id in aligned_gt:
        validation_s1[s1_id] = row

    if len(validation_s1) >= len(aligned_s1_ids):
        break

print("S1 validation records loaded:", f"{len(validation_s1):,}")


# ---------------------------------------------------------------
# Evaluate original blocker
# ---------------------------------------------------------------

candidate_counts = []

gt_total = 0
gt_found = 0

missed_pairs = []

start = time.time()

for idx, s1_id in enumerate(aligned_s1_ids):

    row = validation_s1[s1_id]

    candidates = find_candidates(
        row,
        name_index_100k,
        address_index_100k
    )

    candidate_ids = {
        c["entity_id"]
        for c in candidates
    }

    candidate_counts.append(len(candidate_ids))

    gt_ids = aligned_gt[s1_id]

    gt_total += len(gt_ids)

    found = gt_ids & candidate_ids

    gt_found += len(found)

    missed = gt_ids - candidate_ids

    if missed:
        missed_pairs.append(
            (s1_id, missed)
        )

elapsed = time.time() - start


# ---------------------------------------------------------------
# Report
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("ORIGINAL BLOCKER RESULTS")
print("=" * 70)

print(f"Validation S1 records : {len(validation_s1):,}")
print(f"Ground-truth pairs    : {gt_total:,}")
print(f"Pairs retrieved       : {gt_found:,}")

print(
    f"Candidate recall      : "
    f"{gt_found / gt_total:.4%}"
)

print("\nCandidate statistics:")
print(
    f"Average candidates    : "
    f"{np.mean(candidate_counts):.2f}"
)

print(
    f"Median candidates     : "
    f"{np.median(candidate_counts):.2f}"
)

print(
    f"Maximum candidates    : "
    f"{max(candidate_counts):,}"
)

print(
    f"S1 with zero cand.    : "
    f"{sum(x == 0 for x in candidate_counts):,}"
)

print(
    f"Evaluation time       : "
    f"{elapsed:.2f} seconds"
)

print(
    f"Missed GT pairs       : "
    f"{gt_total - gt_found:,}"
)

if missed_pairs:
    print("\nFirst missed examples:")
    for s1_id, missed in missed_pairs[:10]:
        print(
            " ",
            s1_id,
            "->",
            list(missed)[:5]
        )

print("\n" + "=" * 70)
print("✅ ORIGINAL BLOCKER BASELINE COMPLETE")
print("=" * 70)

ORIGINAL BLOCKER — ID-ALIGNED VALIDATION

Loading 100K S2 records into an ID lookup...
S2 records loaded: 100,000

Loading validation S1 records...
S1 validation records loaded: 2,000

ORIGINAL BLOCKER RESULTS
Validation S1 records : 2,000
Ground-truth pairs    : 2,033
Pairs retrieved       : 1,902
Candidate recall      : 93.5563%

Candidate statistics:
Average candidates    : 531.01
Median candidates     : 143.00
Maximum candidates    : 5,179
S1 with zero cand.    : 11
Evaluation time       : 1.16 seconds
Missed GT pairs       : 131

First missed examples:
  S1-365033919 -> ['S2-118656392']
  S1-606388895 -> ['S2-645714387']
  S1-66645690 -> ['S2-513268507']
  S1-9095876 -> ['S2-572789911']
  S1-196576160 -> ['S2-708684745']
  S1-885485250 -> ['S2-144030669']
  S1-538163501 -> ['S2-832458802']
  S1-252832589 -> ['S2-64798642']
  S1-314561800 -> ['S2-710940398']
  S1-352062821 -> ['S2-910731012']

✅ ORIGINAL BLOCKER BASELINE COMPLETE


## 19. Analyze Missed Ground-Truth Pairs

The original blocker retrieves 1,902 of the 2,033 ground-truth pairs
available in the 100,000-row Source 2 benchmark.

That corresponds to 93.56% candidate recall, leaving 131 missed pairs.

A missed candidate can never be recovered by the later matching/scoring
stage, so these misses must be investigated before optimizing candidate
volume.

This cell analyzes the missed pairs and classifies the available blocking
evidence between each true Source 1 / Source 2 pair.

For every missed pair we inspect:

- shared name blocking keys,
- shared address blocking keys,
- country agreement,
- normalized name/address values.

The goal is to identify systematic causes of candidate-generation failure.

In [20]:
print("=" * 70)
print("MISSED GROUND-TRUTH ANALYSIS")
print("=" * 70)

# ---------------------------------------------------------------
# Helper: normalized values
# ---------------------------------------------------------------

def get_blocking_evidence(s1, s2):

    s1_name = set(
        make_name_keys(
            s1.get("business_name", "")
        )
    )

    s2_name = set(
        make_name_keys(
            s2.get("business_name", "")
        )
    )

    s1_addr = set(
        make_address_keys(
            s1.get("business_address", "")
        )
    )

    s2_addr = set(
        make_address_keys(
            s2.get("business_address", "")
        )
    )

    return {
        "shared_name": s1_name & s2_name,
        "shared_address": s1_addr & s2_addr,
        "country_same": (
            s1.get("country", "").strip().lower()
            ==
            s2.get("country", "").strip().lower()
        ),
    }


# ---------------------------------------------------------------
# Analyze all missed pairs
# ---------------------------------------------------------------

miss_analysis = []

for s1_id, missed_ids in missed_pairs:

    s1 = validation_s1[s1_id]

    for s2_id in missed_ids:

        s2 = s2_lookup_100k.get(s2_id)

        if s2 is None:
            continue

        evidence = get_blocking_evidence(s1, s2)

        miss_analysis.append({
            "s1_id": s1_id,
            "s2_id": s2_id,
            "shared_name_count": len(
                evidence["shared_name"]
            ),
            "shared_address_count": len(
                evidence["shared_address"]
            ),
            "country_same": evidence["country_same"],
            "shared_name_keys": list(
                evidence["shared_name"]
            ),
            "shared_address_keys": list(
                evidence["shared_address"]
            ),
            "s1_name": s1.get("business_name", ""),
            "s2_name": s2.get("business_name", ""),
            "s1_address": s1.get("business_address", ""),
            "s2_address": s2.get("business_address", ""),
        })


# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------

miss_df = pd.DataFrame(miss_analysis)

print("\nMissed pairs analyzed:", len(miss_df))

if len(miss_df) > 0:

    print("\n" + "-" * 70)
    print("BLOCKING EVIDENCE")
    print("-" * 70)

    print(
        "Misses with shared name key    :",
        int((miss_df["shared_name_count"] > 0).sum())
    )

    print(
        "Misses with shared address key :",
        int((miss_df["shared_address_count"] > 0).sum())
    )

    print(
        "Misses with either key         :",
        int(
            (
                (miss_df["shared_name_count"] > 0)
                |
                (miss_df["shared_address_count"] > 0)
            ).sum()
        )
    )

    print(
        "Misses with NO shared blocker  :",
        int(
            (
                (miss_df["shared_name_count"] == 0)
                &
                (miss_df["shared_address_count"] == 0)
            ).sum()
        )
    )

    print(
        "Country mismatch               :",
        int((~miss_df["country_same"]).sum())
    )

    print("\n" + "-" * 70)
    print("EXAMPLE MISSED PAIRS")
    print("-" * 70)

    display_cols = [
        "s1_id",
        "s2_id",
        "shared_name_count",
        "shared_address_count",
        "country_same",
        "s1_name",
        "s2_name",
        "s1_address",
        "s2_address",
    ]

    display(
        miss_df[display_cols].head(20)
    )

print("\n" + "=" * 70)
print("✅ MISSED-PAIR ANALYSIS COMPLETE")
print("=" * 70)

MISSED GROUND-TRUTH ANALYSIS

Missed pairs analyzed: 131

----------------------------------------------------------------------
BLOCKING EVIDENCE
----------------------------------------------------------------------
Misses with shared name key    : 0
Misses with shared address key : 0
Misses with either key         : 0
Misses with NO shared blocker  : 131
Country mismatch               : 0

----------------------------------------------------------------------
EXAMPLE MISSED PAIRS
----------------------------------------------------------------------


,s1_id,s2_id,shared_name_count,shared_address_count,country_same,s1_name,s2_name,s1_address,s2_address
0,S1-365033919,S2-118656392,0,0,True,Koch Premier Newbury,Koch Newbury Premier,"1175 El Valle Drive, Chino Valley, AZ","EL VALLE DRIVE, CHINO VLALEY TOWNSHIP, AZ"
1,S1-606388895,S2-645714387,0,0,True,De & Co,De & Có,"F-42, Shopping Centre, Iind Floor, Mansarover ...","HN 874 F-42, SHOPPING CENTRE, IIND FLOOR, MANS..."
2,S1-66645690,S2-513268507,0,0,True,Fight Trade (India) Private Limited,Mr Fight Tfrha (India) Private Limited,"Gorakhpur, Vashundhara Complex, Gorakhpur, Utt...","VASHUNDHARA COMPLEX, 2ND FLOOR NEAR SARWANG HO..."
3,S1-9095876,S2-572789911,0,0,True,Blue Good Infra Private Limited,બ્લૂ ગુડ ઇન્ફ્રા પ્રાઇવેટ લિમિટેડ,"G-3, Shri Hari Asson, B/H Law College Ground, ...","Gujarat, AHMADABAD CITY, SHRI HARI ASSON, B/H ..."
4,S1-196576160,S2-708684745,0,0,True,Om Global Private Limited,ఓం గ్లోబల్ ప్రైవేట్ లిమిటెడ్,"Telangana, Hyderabad, H-76, H-Block Madhuranag...","HYDERABAD, Andhra Pradesh, HYDERABAD, 8-3-224/..."
5,S1-885485250,S2-144030669,0,0,True,Shivam Constructions,శివం కన్‌స్ట్రక్షన్స్,"B. No. 12, Anthem Site, Gundlapochampalli, Med...","GUNDLAPOCHAMPALLY, Telangana, ANTHEM SITE, GUN..."
6,S1-538163501,S2-832458802,0,0,True,Red Diner,RED DBODMER,"NY, Johnstown, 50 Briggs Street",
7,S1-252832589,S2-64798642,0,0,True,"Diane T. Petersen, P.A. LP","Diane Petersen, P.A. LP Center","378 Emerald Drive, Lake Arrowhead, CA","LAKE ARROWHEAD, CA, 378-382 EMERALD DRIVE"
8,S1-314561800,S2-710940398,0,0,True,Baba Dynamic Hospitality Private Limited,ಬಾಬಾ ಡೈನಾಮಿಕ್ ಹಾಸ್ಪಿಟಾಲಿಟಿ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,Sy No 72 Kinnal Road Ramachandra Rao Police Co...,"KOPPAL, ಕರ್ನಾಟಕ, SY NO 72 KINNAL ROAD RAMACHAN..."
9,S1-352062821,S2-910731012,0,0,True,Q R Richardson Sun,Q R Ricardnosn Sun,"5868 Beechwalk Drive, Virginia Beach City, VA","5868C BEECHWALK DRIVE, VIRGINIA BEACH CITY, VA"



✅ MISSED-PAIR ANALYSIS COMPLETE


## 20. Test Token-Based Name Blocking on Missed Pairs

The missed-pair analysis shows that all 131 missed ground-truth pairs have
zero shared keys under the existing blocking system.

Several missed examples contain the same business-name tokens in a different
order, for example:

    Koch Premier Newbury
    Koch Newbury Premier

The existing blocking keys do not capture this transformation.

We will therefore test an additional token-based name blocker.

For each business name:

1. Use the existing business-name normalization function.
2. Extract alphanumeric tokens.
3. Remove very short tokens.
4. Sort the remaining tokens.
5. Compare the resulting token signatures.

This is a diagnostic test on the 131 previously missed ground-truth pairs.
No full dataset processing is performed.

In [21]:
import re

# Import the normalization function from the provided solution
from normalize import normalize_business_name

print("=" * 70)
print("TOKEN-BASED NAME BLOCKING — MISSED PAIRS")
print("=" * 70)


def token_signature(name):
    """
    Create an order-independent token signature.

    Example:
        'Koch Premier Newbury'
        -> 'koch|newbury|premier'
    """

    normalized = normalize_business_name(
        name if name else ""
    )

    tokens = re.findall(
        r"[a-z0-9]+",
        normalized.lower()
    )

    # Ignore extremely short tokens
    tokens = [
        token
        for token in tokens
        if len(token) >= 3
    ]

    if not tokens:
        return ""

    return "|".join(sorted(set(tokens)))


# ---------------------------------------------------------------
# Compare token signatures for all missed pairs
# ---------------------------------------------------------------

recovered = []

for item in miss_analysis:

    s1_sig = token_signature(
        item["s1_name"]
    )

    s2_sig = token_signature(
        item["s2_name"]
    )

    exact_signature_match = (
        s1_sig != ""
        and
        s1_sig == s2_sig
    )

    s1_tokens = (
        set(s1_sig.split("|"))
        if s1_sig
        else set()
    )

    s2_tokens = (
        set(s2_sig.split("|"))
        if s2_sig
        else set()
    )

    intersection = s1_tokens & s2_tokens
    union = s1_tokens | s2_tokens

    jaccard = (
        len(intersection) / len(union)
        if union
        else 0.0
    )

    recovered.append({
        "s1_id": item["s1_id"],
        "s2_id": item["s2_id"],
        "s1_signature": s1_sig,
        "s2_signature": s2_sig,
        "exact_signature": exact_signature_match,
        "shared_tokens": len(intersection),
        "jaccard": jaccard,
        "s1_name": item["s1_name"],
        "s2_name": item["s2_name"],
    })


token_df = pd.DataFrame(recovered)


# ---------------------------------------------------------------
# Results
# ---------------------------------------------------------------

print("\n" + "-" * 70)
print("TOKEN-BLOCKING RESULTS")
print("-" * 70)

print(
    "Missed pairs analyzed :",
    len(token_df)
)

print(
    "Exact token signature :",
    int(token_df["exact_signature"].sum())
)

print(
    "Pairs sharing >=2 tokens:",
    int(
        (token_df["shared_tokens"] >= 2).sum()
    )
)

print(
    "Pairs sharing >=1 token :",
    int(
        (token_df["shared_tokens"] >= 1).sum()
    )
)

print(
    "Pairs with Jaccard >= 0.5:",
    int(
        (token_df["jaccard"] >= 0.5).sum()
    )
)


print("\n" + "-" * 70)
print("EXAMPLES WITH TOKEN OVERLAP")
print("-" * 70)

examples = token_df[
    token_df["shared_tokens"] >= 1
].sort_values(
    ["shared_tokens", "jaccard"],
    ascending=False
)

if len(examples) > 0:
    display(
        examples[
            [
                "s1_id",
                "s2_id",
                "shared_tokens",
                "jaccard",
                "s1_name",
                "s2_name",
            ]
        ].head(20)
    )
else:
    print("No token overlap found.")

print("\n" + "=" * 70)
print("✅ TOKEN BLOCKING DIAGNOSTIC COMPLETE")
print("=" * 70)

TOKEN-BASED NAME BLOCKING — MISSED PAIRS

----------------------------------------------------------------------
TOKEN-BLOCKING RESULTS
----------------------------------------------------------------------
Missed pairs analyzed : 131
Exact token signature : 12
Pairs sharing >=2 tokens: 26
Pairs sharing >=1 token : 52
Pairs with Jaccard >= 0.5: 29

----------------------------------------------------------------------
EXAMPLES WITH TOKEN OVERLAP
----------------------------------------------------------------------


,s1_id,s2_id,shared_tokens,jaccard,s1_name,s2_name
0,S1-365033919,S2-118656392,3,1.000000,Koch Premier Newbury,Koch Newbury Premier
42,S1-67367613,S2-71425529,3,1.000000,"Daloris P. Park, DMD","park, p. daloris dmd"
60,S1-736632344,S2-612182051,3,1.000000,Len Colonial Fashion Inc,Len Fashion Colonial Inc
83,S1-648583202,S2-187761692,3,1.000000,Rapid Best Blackstone,Bláckstone Best Rapid
106,S1-612519391,S2-586915777,3,1.000000,Mcbride Communities Clinic,CLINIC COMMUNITIES MCBRIDE
20,S1-528762471,S2-884083511,3,0.750000,New Delhi Mal Limited,New-Limited Mal Delhi
98,S1-554164460,S2-273581003,3,0.750000,Pinnacle Mortgage Corporation PLLC,Pllc Pinnacle Mortgage Corporation
99,S1-659104264,S2-146523813,3,0.750000,"Brown, Gulley and Robinson",Brown Robinson & Gulley
26,S1-243501803,S2-408478407,2,1.000000,NS Pioneer Affiliated,NS Affiliated Pioneer
7,S1-252832589,S2-64798642,2,0.666667,"Diane T. Petersen, P.A. LP","Diane Petersen, P.A. LP Center"



✅ TOKEN BLOCKING DIAGNOSTIC COMPLETE


## 21. Measure Token-Based Blocking Selectivity

Token blocking recovered a substantial portion of the previously missed
ground-truth pairs.

However, individual business-name tokens can be extremely common
(e.g. "private", "limited", "global", "services").

A raw token index over millions of records could therefore create very large
candidate buckets.

We will measure token frequencies in the 100,000-row Source 2 benchmark.

For each normalized name token we calculate:

- number of unique tokens,
- median frequency,
- maximum frequency,
- frequency distribution.

This will allow us to choose useful token-blocking rules without creating
another candidate explosion.

In [22]:
from collections import Counter
import re

print("=" * 70)
print("TOKEN SELECTIVITY ANALYSIS")
print("=" * 70)

token_counts = Counter()

start = time.time()

for i, row in enumerate(read_tsv(str(TRAIN_S2))):

    if i >= 100_000:
        break

    normalized = normalize_business_name(
        row.get("business_name", "")
    )

    tokens = re.findall(
        r"[a-z0-9]+",
        normalized.lower()
    )

    # Unique tokens per business record so repeated words
    # within the same name do not inflate frequency.
    tokens = {
        token
        for token in tokens
        if len(token) >= 3
    }

    for token in tokens:
        token_counts[token] += 1

elapsed = time.time() - start

print(f"\nScan time: {elapsed:.2f} seconds")

freqs = np.array(
    list(token_counts.values()),
    dtype=np.int32
)

print("\n" + "-" * 70)
print("TOKEN FREQUENCY SUMMARY")
print("-" * 70)

print("Unique tokens :", f"{len(freqs):,}")
print("Mean frequency:", f"{freqs.mean():.2f}")
print("Median        :", f"{np.median(freqs):.2f}")
print("Maximum       :", f"{freqs.max():,}")

print("\nFrequency thresholds:")

for threshold in [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]:

    kept = freqs <= threshold

    print(
        f"  <= {threshold:4d}: "
        f"{kept.sum():8,} tokens "
        f"({100 * kept.mean():6.2f}%)"
    )

print("\n" + "-" * 70)
print("MOST FREQUENT TOKENS")
print("-" * 70)

for token, freq in token_counts.most_common(30):
    print(f"{token:30s} {freq:8,}")

print("\n" + "=" * 70)
print("✅ TOKEN SELECTIVITY ANALYSIS COMPLETE")
print("=" * 70)

TOKEN SELECTIVITY ANALYSIS

Scan time: 1.64 seconds

----------------------------------------------------------------------
TOKEN FREQUENCY SUMMARY
----------------------------------------------------------------------
Unique tokens : 47,217
Mean frequency: 5.37
Median        : 1.00
Maximum       : 6,123

Frequency thresholds:
  <=    1:   33,853 tokens ( 71.70%)
  <=    2:   39,761 tokens ( 84.21%)
  <=    5:   43,971 tokens ( 93.13%)
  <=   10:   45,044 tokens ( 95.40%)
  <=   20:   45,980 tokens ( 97.38%)
  <=   50:   46,619 tokens ( 98.73%)
  <=  100:   46,852 tokens ( 99.23%)
  <=  200:   46,986 tokens ( 99.51%)
  <=  500:   47,177 tokens ( 99.92%)
  <= 1000:   47,198 tokens ( 99.96%)

----------------------------------------------------------------------
MOST FREQUENT TOKENS
----------------------------------------------------------------------
private                           6,123
com                               3,993
center                            3,875
partners         

## 22. Locate the Source 2 Training File

Before loading the development subset, we will inspect the actual files inside the training directory.

This avoids relying on assumed filenames such as `source2_train.tsv`.

In [23]:
from pathlib import Path

print("=" * 70)
print("TRAIN DIRECTORY CONTENTS")
print("=" * 70)

# Use the TRAIN_DIR defined in Cell 1
print("TRAIN_DIR:")
print(TRAIN_DIR)

print("\nFiles found:")

train_files = sorted(TRAIN_DIR.rglob("*"))

for p in train_files:
    if p.is_file():
        print(" ", p.name)

print("\n" + "=" * 70)
print("TSV FILES")
print("=" * 70)

tsv_files = [p for p in train_files if p.is_file() and p.suffix.lower() == ".tsv"]

for p in tsv_files:
    print(" ", p)
    
print("\nTotal TSV files:", len(tsv_files))

TRAIN DIRECTORY CONTENTS
TRAIN_DIR:
/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train

Files found:
  train_ground_truth.tsv
  train_source1.tsv
  train_source2.tsv
  train_source3.tsv

TSV FILES
  /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_ground_truth.tsv
  /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source1.tsv
  /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
  /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source3.tsv

Total TSV files: 4


## 23. Load the 100K Source 2 Development Subset

The training data uses the filename `train_source2.tsv`.

We load only the first 100,000 Source 2 records for the blocking experiment. This keeps the experiment fast and avoids loading the full 5M+ record Source 2 table into memory.

In [24]:
# ============================================================
# LOAD 100K SOURCE 2 DEVELOPMENT SUBSET
# ============================================================

S2_PATH = TRAIN_DIR / "train_source2.tsv"

print("Loading:", S2_PATH)

S2_DEV = pd.read_csv(
    S2_PATH,
    sep="\t",
    nrows=100_000,
    dtype=str,
    keep_default_na=False
)

print("=" * 70)
print("SOURCE 2 DEVELOPMENT SUBSET")
print("=" * 70)
print("Rows   :", len(S2_DEV))
print("Columns:", list(S2_DEV.columns))

display(S2_DEV.head(3))

Loading: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
SOURCE 2 DEVELOPMENT SUBSET
Rows   : 100000
Columns: ['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India


## 24. Build the Rare-Token Index

We now build an inverted index from normalized business-name tokens to Source 2 records.

Common tokens such as `private`, `limited`, `india`, and `services` can generate very large candidate sets, so the next experiment will keep only selective tokens.

The index is built only for the 100K development subset.

In [25]:
from collections import defaultdict, Counter
import re
import time
import numpy as np
import pandas as pd

def get_name_tokens(text):
    text = normalize_business_name(str(text))
    return [
        t for t in re.findall(r"[a-z0-9]+", text)
        if len(t) >= 3
    ]

t0 = time.time()

token_lists = []
token_freq = Counter()

for name in S2_DEV["business_name"].fillna(""):
    tokens = sorted(set(get_name_tokens(name)))
    token_lists.append(tokens)
    token_freq.update(tokens)

token_postings = defaultdict(list)

for idx, tokens in enumerate(token_lists):
    for token in tokens:
        token_postings[token].append(idx)

print("=" * 70)
print("RARE-TOKEN INDEX")
print("=" * 70)
print("S2 records       :", len(S2_DEV))
print("Unique tokens    :", len(token_freq))
print("Posting entries  :", sum(len(v) for v in token_postings.values()))
print("Build time       :", round(time.time() - t0, 2), "seconds")
print("=" * 70)

RARE-TOKEN INDEX
S2 records       : 100000
Unique tokens    : 47217
Posting entries  : 253395
Build time       : 1.67 seconds


## 25. Test Rare-Token Blocking Recall

The previous experiment identified 131 ground-truth pairs that were missed by the original blocker.

Before testing rare-token blocking, we normalize the stored missed-pair object into a simple `(S1 ID, S2 ID)` list. This makes the experiment independent of whether the previous diagnostic stored the pairs as a DataFrame, tuples, or dictionaries.

We then test several token-frequency thresholds:

- <= 2
- <= 5
- <= 10
- <= 20
- <= 50

The goal is to measure recall and candidate-set size before combining this blocker with the existing blocker.

In [26]:
# ============================================================
# RARE-TOKEN RECALL TEST — ROBUST PAIR HANDLING
# ============================================================

print("=" * 70)
print("RARE-TOKEN BLOCKING RECALL TEST")
print("=" * 70)

print("missed_pairs type:", type(missed_pairs))

# ------------------------------------------------------------
# Convert missed_pairs into (S1 ID, S2 ID) tuples
# ------------------------------------------------------------

pairs = []

if isinstance(missed_pairs, pd.DataFrame):

    print("Detected pandas DataFrame")

    print("Columns:", list(missed_pairs.columns))

    # Try the expected column names
    if "s1_entity_id" in missed_pairs.columns and "s2_entity_id" in missed_pairs.columns:
        pairs = list(
            zip(
                missed_pairs["s1_entity_id"].astype(str),
                missed_pairs["s2_entity_id"].astype(str)
            )
        )

    elif "source1_entity_id" in missed_pairs.columns and "matched_entity_id" in missed_pairs.columns:
        pairs = list(
            zip(
                missed_pairs["source1_entity_id"].astype(str),
                missed_pairs["matched_entity_id"].astype(str)
            )
        )

elif isinstance(missed_pairs, dict):

    print("Detected dictionary")

    for k, v in missed_pairs.items():
        if isinstance(v, (list, tuple, set)):
            for s2_id in v:
                pairs.append((str(k), str(s2_id)))
        else:
            pairs.append((str(k), str(v)))

else:

    print("Detected sequence/object")

    for item in missed_pairs:

        if isinstance(item, dict):

            # Try common naming conventions
            s1 = (
                item.get("s1_entity_id")
                or item.get("source1_entity_id")
                or item.get("s1")
            )

            s2 = (
                item.get("s2_entity_id")
                or item.get("matched_entity_id")
                or item.get("s2")
            )

            if s1 is not None and s2 is not None:
                pairs.append((str(s1), str(s2)))

        elif isinstance(item, (list, tuple)) and len(item) >= 2:

            pairs.append(
                (str(item[0]), str(item[1]))
            )

print("\nPairs extracted:", len(pairs))

if len(pairs) == 0:
    print("\nCould not extract pairs.")
    print("First object:", repr(missed_pairs)[:1000])
    raise ValueError(
        "Unable to determine the structure of missed_pairs."
    )

print("First 5 pairs:")
for p in pairs[:5]:
    print(" ", p)

# ------------------------------------------------------------
# Map Source 2 entity IDs -> row indices
# ------------------------------------------------------------

s2_id_to_idx = dict(
    zip(
        S2_DEV["entity_id"].astype(str),
        range(len(S2_DEV))
    )
)

# ------------------------------------------------------------
# Test thresholds
# ------------------------------------------------------------

thresholds = [2, 5, 10, 20, 50]

results = []

for threshold in thresholds:

    recovered = 0
    tested = 0
    candidate_counts = []

    for s1_id, s2_id in pairs:

        if s2_id not in s2_id_to_idx:
            continue

        tested += 1

        target_idx = s2_id_to_idx[s2_id]

        target_tokens = token_lists[target_idx]

        usable_tokens = [
            tok
            for tok in target_tokens
            if token_freq[tok] <= threshold
        ]

        candidate_indices = set()

        for tok in usable_tokens:
            candidate_indices.update(
                token_postings[tok]
            )

        candidate_counts.append(
            len(candidate_indices)
        )

        if target_idx in candidate_indices:
            recovered += 1

    results.append({
        "frequency_threshold": threshold,
        "pairs_tested": tested,
        "recovered": recovered,
        "recall_%": round(
            100 * recovered / tested, 2
        ) if tested else 0,
        "avg_candidates": round(
            np.mean(candidate_counts), 2
        ) if candidate_counts else 0,
        "median_candidates": round(
            np.median(candidate_counts), 2
        ) if candidate_counts else 0,
        "max_candidates": (
            max(candidate_counts)
            if candidate_counts else 0
        )
    })

rare_token_results = pd.DataFrame(results)

print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)

display(rare_token_results)

RARE-TOKEN BLOCKING RECALL TEST
missed_pairs type: <class 'list'>
Detected sequence/object

Pairs extracted: 131
First 5 pairs:
  ('S1-365033919', "{'S2-118656392'}")
  ('S1-606388895', "{'S2-645714387'}")
  ('S1-66645690', "{'S2-513268507'}")
  ('S1-9095876', "{'S2-572789911'}")
  ('S1-196576160', "{'S2-708684745'}")

RESULTS


,frequency_threshold,pairs_tested,recovered,recall_%,avg_candidates,median_candidates,max_candidates
0,2,0,0,0,0,0,0
1,5,0,0,0,0,0,0
2,10,0,0,0,0,0,0
3,20,0,0,0,0,0,0
4,50,0,0,0,0,0,0


## 26. Correct the Missed-Pair IDs

The previous diagnostic stored the missed Source 2 IDs inside Python set representations such as:

`{'S2-118656392'}`

The previous test treated the entire set representation as the entity ID.

This cell extracts the actual `S2-...` identifier and verifies that the IDs exist in the 100K Source 2 development subset.

In [27]:
# ============================================================
# FIX MISSED-PAIR SOURCE 2 IDS
# ============================================================

import re

fixed_pairs = []

for s1_id, raw_s2 in missed_pairs:

    # Convert things like:
    # "{'S2-118656392'}"
    # into:
    # "S2-118656392"
    
    match = re.search(r"S2-\d+", str(raw_s2))

    if match:
        fixed_pairs.append(
            (str(s1_id), match.group(0))
        )

print("=" * 70)
print("FIXED MISSED PAIRS")
print("=" * 70)

print("Original pairs :", len(missed_pairs))
print("Fixed pairs    :", len(fixed_pairs))

print("\nFirst 10 fixed pairs:")
for pair in fixed_pairs[:10]:
    print(" ", pair)

# ------------------------------------------------------------
# Check how many S2 IDs exist in our 100K development subset
# ------------------------------------------------------------

s2_id_set = set(
    S2_DEV["entity_id"].astype(str)
)

available_pairs = [
    pair
    for pair in fixed_pairs
    if pair[1] in s2_id_set
]

print("\nPairs whose S2 record is inside 100K S2:")
print(len(available_pairs))

print("\nFirst 10 available pairs:")
for pair in available_pairs[:10]:
    print(" ", pair)

print("=" * 70)

FIXED MISSED PAIRS
Original pairs : 131
Fixed pairs    : 131

First 10 fixed pairs:
  ('S1-365033919', 'S2-118656392')
  ('S1-606388895', 'S2-645714387')
  ('S1-66645690', 'S2-513268507')
  ('S1-9095876', 'S2-572789911')
  ('S1-196576160', 'S2-708684745')
  ('S1-885485250', 'S2-144030669')
  ('S1-538163501', 'S2-832458802')
  ('S1-252832589', 'S2-64798642')
  ('S1-314561800', 'S2-710940398')
  ('S1-352062821', 'S2-910731012')

Pairs whose S2 record is inside 100K S2:
131

First 10 available pairs:
  ('S1-365033919', 'S2-118656392')
  ('S1-606388895', 'S2-645714387')
  ('S1-66645690', 'S2-513268507')
  ('S1-9095876', 'S2-572789911')
  ('S1-196576160', 'S2-708684745')
  ('S1-885485250', 'S2-144030669')
  ('S1-538163501', 'S2-832458802')
  ('S1-252832589', 'S2-64798642')
  ('S1-314561800', 'S2-710940398')
  ('S1-352062821', 'S2-910731012')


## 27. Measure Rare-Token Blocking Recall

We now test the corrected 131 missed ground-truth pairs.

For each pair, we use the true Source 2 business name and retrieve all Source 2 records sharing at least one sufficiently rare token.

Frequency thresholds tested:

- 2
- 5
- 10
- 20
- 50

We measure both recall and candidate-set size.

A useful blocking rule should recover missed true matches while keeping candidate sets manageable.

In [28]:
# ============================================================
# RARE-TOKEN BLOCKING RECALL TEST
# ============================================================

thresholds = [2, 5, 10, 20, 50]

# Fast lookup: S2 entity_id -> row index
s2_id_to_idx = dict(
    zip(
        S2_DEV["entity_id"].astype(str),
        range(len(S2_DEV))
    )
)

results = []

for threshold in thresholds:

    recovered = 0
    candidate_counts = []

    for s1_id, s2_id in fixed_pairs:

        # All 131 are expected to exist in this 100K subset
        target_idx = s2_id_to_idx.get(s2_id)

        if target_idx is None:
            continue

        # Tokens of the true S2 record
        target_tokens = token_lists[target_idx]

        # Keep only sufficiently rare tokens
        usable_tokens = [
            tok
            for tok in target_tokens
            if token_freq[tok] <= threshold
        ]

        candidate_indices = set()

        for tok in usable_tokens:
            candidate_indices.update(
                token_postings[tok]
            )

        candidate_counts.append(len(candidate_indices))

        # Did the rare-token blocker retrieve the true record?
        if target_idx in candidate_indices:
            recovered += 1

    tested = len(candidate_counts)

    results.append({
        "frequency_threshold": threshold,
        "pairs_tested": tested,
        "recovered": recovered,
        "recall_%": round(
            100 * recovered / tested, 2
        ) if tested else 0,
        "avg_candidates": round(
            np.mean(candidate_counts), 2
        ) if candidate_counts else 0,
        "median_candidates": round(
            np.median(candidate_counts), 2
        ) if candidate_counts else 0,
        "max_candidates": (
            max(candidate_counts)
            if candidate_counts else 0
        )
    })

rare_token_results = pd.DataFrame(results)

print("=" * 70)
print("RARE-TOKEN BLOCKING RESULTS")
print("=" * 70)

display(rare_token_results)

RARE-TOKEN BLOCKING RESULTS


,frequency_threshold,pairs_tested,recovered,recall_%,avg_candidates,median_candidates,max_candidates
0,2,131,48,36.64,0.42,0.0,3
1,5,131,56,42.75,0.75,0.0,6
2,10,131,59,45.04,0.99,0.0,16
3,20,131,64,48.85,1.58,0.0,25
4,50,131,68,51.91,2.94,1.0,58


## 28. Inspect Missed Pairs After Rare-Token Blocking

The threshold-50 token blocker recovered 68 of the 131 previously missed pairs.

We now inspect the remaining 63 pairs.

Understanding these failures is more useful than immediately increasing the token-frequency threshold, because increasing the threshold may increase candidate volume without solving the underlying mismatch.

For each remaining pair we display:

- S1 business name
- S2 business name
- S1 address
- S2 address
- country
- shared normalized tokens
- token frequency information

This will help identify the next blocking strategy.

In [29]:
# ============================================================
# INSPECT PAIRS MISSED BY THRESHOLD-50 TOKEN BLOCKING
# ============================================================

THRESHOLD = 50

missed_after_token = []

for s1_id, s2_id in fixed_pairs:

    target_idx = s2_id_to_idx.get(s2_id)

    if target_idx is None:
        continue

    target_tokens = set(token_lists[target_idx])

    usable_tokens = [
        tok
        for tok in target_tokens
        if token_freq[tok] <= THRESHOLD
    ]

    candidate_indices = set()

    for tok in usable_tokens:
        candidate_indices.update(
            token_postings[tok]
        )

    # True pair was NOT recovered
    if target_idx not in candidate_indices:

        missed_after_token.append(
            (s1_id, s2_id)
        )

print("=" * 70)
print("REMAINING MISSED PAIRS")
print("=" * 70)

print(
    f"Remaining: {len(missed_after_token)} / {len(fixed_pairs)}"
)

# ------------------------------------------------------------
# Load Source 1 records for these pairs
# ------------------------------------------------------------

S1_PATH = TRAIN_DIR / "train_source1.tsv"

S1_DEV = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

s1_lookup = S1_DEV.set_index(
    "entity_id",
    drop=False
)

s2_lookup = S2_DEV.set_index(
    "entity_id",
    drop=False
)

inspection_rows = []

for s1_id, s2_id in missed_after_token:

    if s1_id not in s1_lookup.index:
        continue

    if s2_id not in s2_lookup.index:
        continue

    s1 = s1_lookup.loc[s1_id]
    s2 = s2_lookup.loc[s2_id]

    s1_tokens = set(
        get_name_tokens(s1["business_name"])
    )

    s2_tokens = set(
        get_name_tokens(s2["business_name"])
    )

    shared_tokens = sorted(
        s1_tokens & s2_tokens
    )

    shared_with_freq = [
        f"{tok}({token_freq[tok]})"
        for tok in shared_tokens
    ]

    inspection_rows.append({
        "s1_id": s1_id,
        "s2_id": s2_id,
        "s1_name": s1["business_name"],
        "s2_name": s2["business_name"],
        "s1_address": s1["business_address"],
        "s2_address": s2["business_address"],
        "country": s1["country"],
        "shared_tokens": ", ".join(shared_with_freq)
    })

missed_inspection = pd.DataFrame(
    inspection_rows
)

print("\nRows available for inspection:", len(missed_inspection))

display(
    missed_inspection.head(30)
)

REMAINING MISSED PAIRS
Remaining: 63 / 131

Rows available for inspection: 63


,s1_id,s2_id,s1_name,s2_name,s1_address,s2_address,country,shared_tokens
0,S1-606388895,S2-645714387,De & Co,De & Có,"F-42, Shopping Centre, Iind Floor, Mansarover ...","HN 874 F-42, SHOPPING CENTRE, IIND FLOOR, MANS...",India,
1,S1-9095876,S2-572789911,Blue Good Infra Private Limited,બ્લૂ ગુડ ઇન્ફ્રા પ્રાઇવેટ લિમિટેડ,"G-3, Shri Hari Asson, B/H Law College Ground, ...","Gujarat, AHMADABAD CITY, SHRI HARI ASSON, B/H ...",India,
2,S1-196576160,S2-708684745,Om Global Private Limited,ఓం గ్లోబల్ ప్రైవేట్ లిమిటెడ్,"Telangana, Hyderabad, H-76, H-Block Madhuranag...","HYDERABAD, Andhra Pradesh, HYDERABAD, 8-3-224/...",India,
3,S1-885485250,S2-144030669,Shivam Constructions,శివం కన్‌స్ట్రక్షన్స్,"B. No. 12, Anthem Site, Gundlapochampalli, Med...","GUNDLAPOCHAMPALLY, Telangana, ANTHEM SITE, GUN...",India,
4,S1-314561800,S2-710940398,Baba Dynamic Hospitality Private Limited,ಬಾಬಾ ಡೈನಾಮಿಕ್ ಹಾಸ್ಪಿಟಾಲಿಟಿ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,Sy No 72 Kinnal Road Ramachandra Rao Police Co...,"KOPPAL, ಕರ್ನಾಟಕ, SY NO 72 KINNAL ROAD RAMACHAN...",India,
5,S1-445941109,S2-364329044,Black Logistics Limited,ब्लैक लॉजिस्टिक्स लिमिटेड,"Haryana, C/Oraj Rani, Kurukshetra, Shop No63, ...","HN 52 C/ORAJ RANI, SHOP NO63, NEAR SANGAM PALA...",India,
6,S1-638252846,S2-171179881,Apex Technology Limited,एपेक्स टेक्नोलॉजी लिमिटेड,"Indore, Madhya Pradesh, Survey No. 101/5 Villa...","DOOR NO 101/5 VILLAGE LASDIYA MORI, NULL, INDO...",India,
7,S1-848337111,S2-838230810,Al Tech Enterprises Limited,अल टेक एंटरप्राइजेज लिमिटेड,"Mcf-67, Bhoor Colony, Old Faridabad, Faridabad...","C/O SHRI CHAND, MCF-67, BHOOR COLONY, OLD FARI...",India,
8,S1-803109141,S2-88820352,One It Pvt Ltd,One It பிரைவேட் லிமிடெட்,"Villa No: 19D, Plot No: 4, B&B Garden River, K...","Tamil Nadu, VILLA NO: 0019D, PLOT NO: 4, B&B G...",India,one(92)
9,S1-61231563,S2-572227958,Southern Classic Impex Pvt Ltd,सदर्न क्लासिक इम्पेक्स प्रा. लि.,"House No. - L 1/27(Ground Floor), Sri Krishna ...","PLOT 642 HOUSE NO. - L 1/27(GROUND FLOOR), SRI...",India,


## 29. Build the Address-Token Index

The previous cell failed because `normalize_address` was not available in the current notebook namespace.

We explicitly import the address normalizer from the provided `amazon_ml_solution` package and then build a lightweight address-token index over the 100K Source 2 development subset.

This experiment is still limited to 100K records and does not affect the final submission yet.

In [30]:
# ============================================================
# ADDRESS TOKEN INDEX — CORRECTED
# ============================================================

import sys
from collections import defaultdict, Counter
import re
import time
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Make the provided solution modules importable
# ------------------------------------------------------------

solution_src = SOLUTION_DIR

if str(solution_src) not in sys.path:
    sys.path.insert(0, str(solution_src))

# Explicitly import the provided normalizer
from normalize import normalize_address

print("normalize_address imported successfully.")

# ------------------------------------------------------------
# Address tokenization
# ------------------------------------------------------------

def get_address_tokens(text):
    text = normalize_address(str(text))

    tokens = re.findall(r"[a-z0-9]+", text)

    return [
        token
        for token in tokens
        if len(token) >= 2
    ]


# ------------------------------------------------------------
# Build address token index
# ------------------------------------------------------------

t0 = time.time()

address_token_lists = []
address_token_freq = Counter()

for address in S2_DEV["business_address"].fillna(""):

    tokens = sorted(
        set(get_address_tokens(address))
    )

    address_token_lists.append(tokens)
    address_token_freq.update(tokens)


address_token_postings = defaultdict(list)

for idx, tokens in enumerate(address_token_lists):

    for token in tokens:
        address_token_postings[token].append(idx)


print("=" * 70)
print("ADDRESS TOKEN INDEX")
print("=" * 70)

print("S2 records        :", len(S2_DEV))
print("Unique addr tokens:", len(address_token_freq))
print(
    "Posting entries   :",
    sum(len(v) for v in address_token_postings.values())
)
print(
    "Build time        :",
    round(time.time() - t0, 2),
    "seconds"
)

print("\nMost frequent address tokens:")

for token, freq in address_token_freq.most_common(30):
    print(f"{token:<25} {freq}")

normalize_address imported successfully.
ADDRESS TOKEN INDEX
S2 records        : 100000
Unique addr tokens: 79583
Posting entries   : 685383
Build time        : 3.9 seconds

Most frequent address tokens:
rd                        19561
no                        18341
st                        13228
dr                        9980
ave                       8427
maharashtra               6469
floor                     6257
tx                        5879
city                      5859
nagar                     5665
delhi                     5050
ln                        4659
ny                        4483
pradesh                   4219
new                       4155
nc                        4096
west                      4077
oh                        3782
plot                      3568
il                        3286
mumbai                    3266
null                      2594
va                        2579
tn                        2539
az                        2472
ma                

## 30. Test Address-Token Blocking Recall

The address-token index contains substantially more information than the existing address blocker.

We now test it against the 63 pairs that remained unresolved after threshold-50 name-token blocking.

We will use several address-token frequency thresholds:

- <= 2
- <= 5
- <= 10
- <= 20
- <= 50
- <= 100

For each threshold we measure:

- recovered true matches
- recall
- average candidate count
- median candidate count
- maximum candidate count

This is a development experiment only.

In [31]:
# ============================================================
# ADDRESS-TOKEN BLOCKING RECALL TEST
# ============================================================

address_thresholds = [2, 5, 10, 20, 50, 100]

# ------------------------------------------------------------
# Reuse the Source 2 entity ID -> row index mapping
# ------------------------------------------------------------

s2_id_to_idx = dict(
    zip(
        S2_DEV["entity_id"].astype(str),
        range(len(S2_DEV))
    )
)

address_results = []

for threshold in address_thresholds:

    recovered = 0
    candidate_counts = []

    for s1_id, s2_id in missed_after_token:

        target_idx = s2_id_to_idx.get(s2_id)

        if target_idx is None:
            continue

        target_tokens = address_token_lists[target_idx]

        usable_tokens = [
            token
            for token in target_tokens
            if address_token_freq[token] <= threshold
        ]

        candidate_indices = set()

        for token in usable_tokens:
            candidate_indices.update(
                address_token_postings[token]
            )

        candidate_counts.append(
            len(candidate_indices)
        )

        if target_idx in candidate_indices:
            recovered += 1

    tested = len(candidate_counts)

    address_results.append({
        "frequency_threshold": threshold,
        "pairs_tested": tested,
        "recovered": recovered,
        "recall_%": round(
            100 * recovered / tested, 2
        ) if tested else 0,
        "avg_candidates": round(
            np.mean(candidate_counts), 2
        ) if candidate_counts else 0,
        "median_candidates": round(
            np.median(candidate_counts), 2
        ) if candidate_counts else 0,
        "max_candidates": (
            max(candidate_counts)
            if candidate_counts else 0
        )
    })

address_token_results = pd.DataFrame(address_results)

print("=" * 70)
print("ADDRESS-TOKEN BLOCKING RESULTS")
print("=" * 70)

display(address_token_results)

ADDRESS-TOKEN BLOCKING RESULTS


,frequency_threshold,pairs_tested,recovered,recall_%,avg_candidates,median_candidates,max_candidates
0,2,63,32,50.79,0.73,1.0,4
1,5,63,38,60.32,1.90,1.0,12
2,10,63,43,68.25,4.40,2.0,19
3,20,63,47,74.60,9.22,6.0,38
4,50,63,53,84.13,38.00,32.0,139
5,100,63,56,88.89,68.75,58.0,271


## 31A. Inspect the Existing Validation Ground Truth

The ID-aligned validation experiment already created the required ground-truth variables.

The previous combined-blocking cell assumed a variable named `validation_gt`, but this notebook actually contains `available_gt` and `aligned_gt`.

We will inspect their structure and use the existing validation data rather than rebuilding it.

In [32]:
# ============================================================
# INSPECT EXISTING VALIDATION GROUND TRUTH
# ============================================================

print("=" * 70)
print("VALIDATION VARIABLES")
print("=" * 70)

for name in [
    "validation_s1",
    "available_gt",
    "aligned_gt",
    "gt_available",
    "gt_ids"
]:
    
    if name in globals():
        obj = globals()[name]
        
        print(f"\n{name}")
        print("  type:", type(obj))
        
        if isinstance(obj, pd.DataFrame):
            print("  shape:", obj.shape)
            print("  columns:", list(obj.columns))
            display(obj.head(3))
        
        elif isinstance(obj, dict):
            print("  length:", len(obj))
            print("  first items:")
            for i, item in enumerate(obj.items()):
                print("   ", item)
                if i >= 2:
                    break
        
        elif isinstance(obj, (list, tuple, set)):
            print("  length:", len(obj))
            print("  first items:", list(obj)[:5])
        
        else:
            print("  value:", repr(obj)[:500])

VALIDATION VARIABLES

validation_s1
  type: <class 'dict'>
  length: 2000
  first items:
    ('S1-736761428', {'entity_id': 'S1-736761428', 'business_name': 'Rodriguez, Guest & Brooks', 'business_address': '450431 316 Drive, Afton, OK', 'country': 'US'})
    ('S1-867508043', {'entity_id': 'S1-867508043', 'business_name': 'ON Berkley LLC', 'business_address': '1708 Patricia Court, Upper Allen Township, PA', 'country': 'US'})
    ('S1-852279820', {'entity_id': 'S1-852279820', 'business_name': 'Central Marketing Global LLC', 'business_address': '72573 Lake View Drive, Jackson, MN', 'country': 'US'})

available_gt
  type: <class 'int'>
  value: 3

aligned_gt
  type: <class 'dict'>
  length: 2000
  first items:
    ('S1-244810289', {'S2-63598201'})
    ('S1-292900819', {'S2-791548670'})
    ('S1-307962633', {'S2-22544434'})

gt_available
  type: <class 'int'>
  value: 3

gt_ids
  type: <class 'set'>
  length: 2
  first items: ['S2-223245005', 'S2-647596046']


## 31B. Combined Blocking Recall Experiment

We now evaluate the complete blocking strategy on the existing ID-aligned validation set.

For every one of the 2,000 validation Source 1 records, candidates are generated using:

1. The original name/address blocker
2. Rare business-name token blocking
3. Rare address-token blocking

The ground truth is taken from `aligned_gt`, which maps each validation S1 entity to its known matching S2 entities.

We compare several name/address token thresholds.

The objective is to maximize candidate recall while keeping the candidate set reasonably small.

In [33]:
# ============================================================
# COMBINED BLOCKING RECALL TEST — USING EXISTING VARIABLES
# ============================================================

from candidate_generator import build_index, find_candidates

print("=" * 70)
print("COMBINED BLOCKING RECALL TEST")
print("=" * 70)

print("Validation S1 records:", len(validation_s1))
print("Validation GT records:", len(aligned_gt))

# ------------------------------------------------------------
# Build the original blocker index
# ------------------------------------------------------------

print("\nBuilding original blocker index...")

t0 = time.time()

name_index, address_index = build_index(
    S2_PATH,
    limit=100_000
)

print(
    "Index build time:",
    round(time.time() - t0, 2),
    "seconds"
)

# ------------------------------------------------------------
# Source 2 entity ID -> row index
# ------------------------------------------------------------

s2_id_to_idx = dict(
    zip(
        S2_DEV["entity_id"].astype(str),
        range(len(S2_DEV))
    )
)

# ------------------------------------------------------------
# Test configurations
# ------------------------------------------------------------

configs = [
    (20, 20),
    (50, 20),
    (50, 50),
    (50, 100),
]

combined_results = []

for name_threshold, address_threshold in configs:

    print(
        f"\nTesting name <= {name_threshold}, "
        f"address <= {address_threshold}"
    )

    total_gt = 0
    found_gt = 0

    candidate_counts = []
    zero_candidates = 0

    t0 = time.time()

    # --------------------------------------------------------
    # Process all 2,000 validation S1 records
    # --------------------------------------------------------

    for s1_id, s1_row in validation_s1.items():

        # ----------------------------------------------------
        # Ground truth
        # ----------------------------------------------------

        true_ids = aligned_gt.get(
            s1_id,
            set()
        )

        true_s2_ids = {
            str(x)
            for x in true_ids
            if str(x) in s2_id_to_idx
        }

        total_gt += len(true_s2_ids)

        # ----------------------------------------------------
        # Original blocker
        # ----------------------------------------------------

        original_candidates = find_candidates(
            s1_row,
            name_index,
            address_index
        )

        candidate_ids = {
            str(row["entity_id"])
            for row in original_candidates
        }

        # ----------------------------------------------------
        # Business-name token blocker
        # ----------------------------------------------------

        name_tokens = set(
            get_name_tokens(
                s1_row["business_name"]
            )
        )

        for token in name_tokens:

            if token_freq[token] <= name_threshold:

                for idx in token_postings[token]:

                    candidate_ids.add(
                        str(
                            S2_DEV.iloc[idx]["entity_id"]
                        )
                    )

        # ----------------------------------------------------
        # Address-token blocker
        # ----------------------------------------------------

        address_tokens = set(
            get_address_tokens(
                s1_row["business_address"]
            )
        )

        for token in address_tokens:

            if address_token_freq[token] <= address_threshold:

                for idx in address_token_postings[token]:

                    candidate_ids.add(
                        str(
                            S2_DEV.iloc[idx]["entity_id"]
                        )
                    )

        # ----------------------------------------------------
        # Candidate statistics
        # ----------------------------------------------------

        count = len(candidate_ids)

        candidate_counts.append(count)

        if count == 0:
            zero_candidates += 1

        # ----------------------------------------------------
        # Candidate recall
        # ----------------------------------------------------

        found_gt += len(
            true_s2_ids.intersection(
                candidate_ids
            )
        )

    elapsed = time.time() - t0

    recall = (
        100 * found_gt / total_gt
        if total_gt
        else 0
    )

    combined_results.append({
        "name_threshold": name_threshold,
        "address_threshold": address_threshold,
        "GT_pairs": total_gt,
        "found_GT": found_gt,
        "candidate_recall_%": round(
            recall, 3
        ),
        "avg_candidates": round(
            np.mean(candidate_counts), 2
        ),
        "median_candidates": round(
            np.median(candidate_counts), 2
        ),
        "max_candidates": max(candidate_counts),
        "zero_candidate_S1": zero_candidates,
        "runtime_sec": round(elapsed, 2)
    })

combined_results_df = pd.DataFrame(
    combined_results
)

print("\n" + "=" * 70)
print("COMBINED BLOCKING RESULTS")
print("=" * 70)

display(combined_results_df)

COMBINED BLOCKING RECALL TEST
Validation S1 records: 2000
Validation GT records: 2000

Building original blocker index...
Building index from: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
Finished indexing 100,000 records.
Name index keys    : 348,248
Address index keys : 216,462
Index build time: 6.84 seconds

Testing name <= 20, address <= 20

Testing name <= 50, address <= 20

Testing name <= 50, address <= 50

Testing name <= 50, address <= 100

COMBINED BLOCKING RESULTS


,name_threshold,address_threshold,GT_pairs,found_GT,candidate_recall_%,avg_candidates,median_candidates,max_candidates,zero_candidate_S1,runtime_sec
0,20,20,2033,2001,98.426,542.23,156.0,5184,0,2.11
1,50,20,2033,2003,98.524,547.12,159.0,5184,0,2.42
2,50,50,2033,2016,99.164,564.35,174.0,5184,0,3.24
3,50,100,2033,2021,99.410,600.04,218.0,5184,0,4.86


## 32. Inspect the Remaining S2 Misses

The combined blocker recovered 2,021 of 2,033 ground-truth S2 matches, giving 99.41% candidate recall.

Only 12 true S2 matches remain outside the candidate set.

We inspect these 12 cases to determine whether they require another blocking rule or whether the current recall is sufficient for the matching stage.

In [34]:
# ============================================================
# INSPECT THE 12 REMAINING S2 MISSES
# ============================================================

NAME_THRESHOLD = 50
ADDRESS_THRESHOLD = 100

remaining_misses = []

for s1_id, s1_row in validation_s1.items():

    true_ids = {
        str(x)
        for x in aligned_gt.get(s1_id, set())
        if str(x) in s2_id_to_idx
    }

    if not true_ids:
        continue

    # -----------------------------
    # Original blocker
    # -----------------------------
    original_candidates = find_candidates(
        s1_row,
        name_index,
        address_index
    )

    candidate_ids = {
        str(row["entity_id"])
        for row in original_candidates
    }

    # -----------------------------
    # Name token blocker
    # -----------------------------
    for token in set(
        get_name_tokens(s1_row["business_name"])
    ):
        if token_freq[token] <= NAME_THRESHOLD:
            for idx in token_postings[token]:
                candidate_ids.add(
                    str(S2_DEV.iloc[idx]["entity_id"])
                )

    # -----------------------------
    # Address token blocker
    # -----------------------------
    for token in set(
        get_address_tokens(s1_row["business_address"])
    ):
        if address_token_freq[token] <= ADDRESS_THRESHOLD:
            for idx in address_token_postings[token]:
                candidate_ids.add(
                    str(S2_DEV.iloc[idx]["entity_id"])
                )

    # -----------------------------
    # Find missed true matches
    # -----------------------------
    missed = true_ids - candidate_ids

    for s2_id in missed:

        s2_row = S2_DEV[
            S2_DEV["entity_id"].astype(str) == s2_id
        ]

        if len(s2_row) == 0:
            continue

        s2_row = s2_row.iloc[0]

        s1_tokens = set(
            get_name_tokens(s1_row["business_name"])
        )

        s2_tokens = set(
            get_name_tokens(s2_row["business_name"])
        )

        shared_name_tokens = sorted(
            s1_tokens & s2_tokens
        )

        s1_addr_tokens = set(
            get_address_tokens(s1_row["business_address"])
        )

        s2_addr_tokens = set(
            get_address_tokens(s2_row["business_address"])
        )

        shared_address_tokens = sorted(
            s1_addr_tokens & s2_addr_tokens
        )

        remaining_misses.append({
            "s1_id": s1_id,
            "s2_id": s2_id,
            "country": s1_row["country"],
            "s1_name": s1_row["business_name"],
            "s2_name": s2_row["business_name"],
            "s1_address": s1_row["business_address"],
            "s2_address": s2_row["business_address"],
            "shared_name_tokens": ", ".join(
                shared_name_tokens
            ),
            "shared_address_tokens": ", ".join(
                shared_address_tokens
            )
        })

misses_df = pd.DataFrame(remaining_misses)

print("=" * 70)
print("REMAINING S2 MISSES")
print("=" * 70)
print("Missed pairs:", len(misses_df))

display(misses_df)

REMAINING S2 MISSES
Missed pairs: 12


,s1_id,s2_id,country,s1_name,s2_name,s1_address,s2_address,shared_name_tokens,shared_address_tokens
0,S1-224675553,S2-312599594,India,Ace Enterprises Private Limited,एस एंटरप्राइजेज प्राइवेट लिमिटेड,"Plot No. 101, Khasra No. 70/1/1, First Floor M...","NEW DELHI, PLOT NO. 0101, Delhi, NORTH WEST",,"delhi, new, no, north, plot, west"
1,S1-492482644,S2-875621751,India,Om Products,ओम प्रोडक्ट्स,"South Delhi, Delhi, South Extension Part I, E-...","E GROUND FLOOR, SOUTH EXTENSION PART I, SOUTH ...",,"delhi, extension, floor, ground, part, south"
2,S1-637534044,S2-80804149,India,United Infra Private Limited,यूनाइटेड इंफ्रा प्राइवेट लिमिटेड,"B/601, Abt Apartments, Near Navjivan School Ra...","HN 269 B/601, MUMBAI, MUMBAI CITY, महाराष्ट्र",,"601, city, mumbai"
3,S1-236528455,S2-240391608,India,Swastik Tech,ಸ್ವಸ್ತಿಕ್ ಟೆಕ್,"Office No.#8, 1St Floor, Xl Tower, Cl Layout, ...","BLOCK E-86 OFFICE NO.#8, null, BANGALORE, BANG...",,"bangalore, no, office, south"
4,S1-406694841,S2-803047060,India,Silver It Private Limited,सिल्वर आईटी प्राइवेट लिमिटेड,"South Side Of G T Road, Bulandshahar Road Indu...","DOOR NO 9-3, NULL, GHAZIABAD, Uttar Pradesh",,"ghaziabad, no, pradesh, uttar"
5,S1-314021535,S2-359393072,India,CK (india) Private Limited,CK (ih) Private Limited,"House No Lig B 219, E 7 Sector, Shahpura, Bhop...",,,
6,S1-253119136,S2-884381367,US,Tolley Montessori School LLC,tsmontessori.com,"Staten Island, 1815 Forest Avenue, NY","FOREST AVENUE, N/A, SATEN ISLAND CITY, NY",,"ave, forest, island, ny"
7,S1-174015063,S2-613899803,India,Shree Healthcare Private Limited,श्री हेल्थकेयर प्राइवेट लिमिटेड,"Shop No 2, Bejod Niwas, Bldg No 3, Adiwali Tal...","#2, THANE, KALYAN EAST, महाराष्ट्र",,"east, kalyan, thane"
8,S1-980953751,S2-840071023,India,Lakshmi Investment Private Limited,लक्ष्मी इन्वेस्टमेंट प्राइवेट लिमिटेड,"Office No.34, Pratik Arcade, Ground Floor Tata...","NO 926 OFFICE NO.34, MUMBAI REGION, MUMBAI CIT...",,"34, city, mumbai, no, office"
9,S1-538163501,S2-832458802,US,Red Diner,RED DBODMER,"NY, Johnstown, 50 Briggs Street",,red,


## 33. Build the S3 Validation Set

The S2 validation reached 99.41% candidate recall.

Before changing the blocker, we test the exact same strategy on Source 3.

We use the first 100,000 S3 records and select S1 ground-truth matches whose S3 entity IDs are present in this development subset.

This gives us an ID-aligned validation set for S3 without randomly sampling the ground truth.

In [35]:
# ============================================================
# LOAD 100K S3 DEVELOPMENT DATA
# ============================================================

S3_PATH = TRAIN_DIR / "train_source3.tsv"

print("S3 file:", S3_PATH)
print("Exists:", S3_PATH.exists())

S3_DEV = pd.read_csv(
    S3_PATH,
    sep="\t",
    nrows=100_000,
    dtype=str,
    keep_default_na=False
)

print("\nS3_DEV shape:", S3_DEV.shape)
print("Columns:", list(S3_DEV.columns))

display(S3_DEV.head(3))

print("\nUnique S3 IDs:",
      S3_DEV["entity_id"].nunique())

S3 file: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source3.tsv
Exists: True

S3_DEV shape: (100000, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US



Unique S3 IDs: 100000


## 34. Build ID-Aligned S3 Ground Truth

We keep only ground-truth S3 matches whose IDs exist in the 100K S3 development subset.

This prevents random sampling from hiding difficult cases and lets us measure true candidate recall.

In [36]:
# ============================================================
# BUILD ID-ALIGNED S3 GROUND TRUTH
# ============================================================

# S3 IDs available in our 100K development subset
s3_dev_ids = set(
    S3_DEV["entity_id"].astype(str)
)

aligned_gt_s3 = {}
available_s3_pairs = 0

for s1_id, matched_ids in aligned_gt.items():
    # IMPORTANT:
    # aligned_gt currently contains the S2-aligned GT.
    # We therefore read the original GT file to obtain
    # the complete S2 + S3 match list.
    pass

# Load the complete training ground truth
GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"

print("Ground-truth file:", GT_PATH)
print("Exists:", GT_PATH.exists())

gt_df = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("\nGT shape:", gt_df.shape)
print("Columns:", list(gt_df.columns))

display(gt_df.head(3))

Ground-truth file: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_ground_truth.tsv
Exists: True

GT shape: (2206821, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"


## 35. Create the ID-Aligned S3 Validation Ground Truth

For every S1 entity, we extract only the S3 IDs from the ground-truth list.

We then keep only S3 matches that exist in our 100K S3 development subset.

This gives us a clean S3 validation set for measuring candidate recall.

In [37]:
# ============================================================
# BUILD S3-ALIGNED GROUND TRUTH
# ============================================================

s3_dev_ids = set(
    S3_DEV["entity_id"].astype(str)
)

aligned_gt_s3 = {}
available_s3_pairs = 0

for _, row in gt_df.iterrows():

    s1_id = str(row["source1_entity_id"])

    matched = str(row["matched_entity_ids"])

    # Extract only S3 entity IDs
    s3_ids = {
        x.strip()
        for x in matched.split(",")
        if x.strip().startswith("S3-")
    }

    # Keep only S3 IDs available in our 100K development set
    available = s3_ids & s3_dev_ids

    if available:
        aligned_gt_s3[s1_id] = available
        available_s3_pairs += len(available)

print("=" * 70)
print("S3 ID-ALIGNED VALIDATION")
print("=" * 70)

print("S1 records with available S3 matches:",
      len(aligned_gt_s3))

print("Available S3 ground-truth pairs:",
      available_s3_pairs)

print("S3 development records:",
      len(S3_DEV))

# Show a few examples
print("\nExamples:")
for i, (s1_id, ids) in enumerate(aligned_gt_s3.items()):
    print(s1_id, "->", list(ids)[:5])
    if i >= 4:
        break

S3 ID-ALIGNED VALIDATION
S1 records with available S3 matches: 73486
Available S3 ground-truth pairs: 74544
S3 development records: 100000

Examples:
S1-965667 -> ['S3-860443364']
S1-295437535 -> ['S3-918259250']
S1-539763792 -> ['S3-643822684']
S1-531928114 -> ['S3-110146591']
S1-945578383 -> ['S3-434437783']


## 36. Build S3 Blocking Indexes

We build the same two blocking indexes used for S2:

1. Normalized business-name blocking
2. Normalized business-address blocking

We also build token indexes for the improved fallback blocker.

The S3 validation will use the same thresholds that produced 99.41% S2 recall:
- Name token frequency ≤ 50
- Address token frequency ≤ 100
- 

In [38]:
# ============================================================
# BUILD S3 BLOCKING INDEXES
# ============================================================

from collections import defaultdict
from normalize import normalize_business_name, normalize_address
from candidate_generator import build_index, find_candidates

S3_INDEX_START = time.time()

print("Building original S3 name/address indexes...")

s3_name_index, s3_address_index = build_index(
    S3_PATH,
    limit=100_000
)

print("Original S3 indexes built.")
print("Name keys   :", len(s3_name_index))
print("Address keys:", len(s3_address_index))

# ------------------------------------------------------------
# Name token index
# ------------------------------------------------------------

print("\nBuilding S3 name-token index...")

s3_name_postings = defaultdict(list)

for idx, row in S3_DEV.iterrows():

    tokens = set(
        get_name_tokens(row["business_name"])
    )

    for token in tokens:
        s3_name_postings[token].append(idx)

s3_name_token_freq = {
    token: len(postings)
    for token, postings in s3_name_postings.items()
}

print("Unique name tokens:",
      len(s3_name_postings))

print("Name posting entries:",
      sum(len(v) for v in s3_name_postings.values()))

# ------------------------------------------------------------
# Address token index
# ------------------------------------------------------------

print("\nBuilding S3 address-token index...")

s3_address_postings = defaultdict(list)

for idx, row in S3_DEV.iterrows():

    tokens = set(
        get_address_tokens(row["business_address"])
    )

    for token in tokens:
        s3_address_postings[token].append(idx)

s3_address_token_freq = {
    token: len(postings)
    for token, postings in s3_address_postings.items()
}

print("Unique address tokens:",
      len(s3_address_postings))

print("Address posting entries:",
      sum(len(v) for v in s3_address_postings.values()))

print(
    f"\nTotal index build time: "
    f"{time.time() - S3_INDEX_START:.2f} seconds"
)

Building original S3 name/address indexes...
Building index from: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source3.tsv
Finished indexing 100,000 records.
Name index keys    : 356,730
Address index keys : 252,822
Original S3 indexes built.
Name keys   : 356730
Address keys: 252822

Building S3 name-token index...
Unique name tokens: 48873
Name posting entries: 266973

Building S3 address-token index...
Unique address tokens: 75021
Address posting entries: 674115

Total index build time: 21.30 seconds


## 37. Measure S3 Candidate Recall

We now apply the same blocker configuration that achieved 99.41% S2 candidate recall:

- Original name blocking
- Original address blocking
- Name-token fallback with frequency ≤ 50
- Address-token fallback with frequency ≤ 100

For every validated S1, we compare the generated candidates against the available S3 ground-truth IDs.

The main metric here is candidate recall:
    
    recall = ground-truth pairs found / available ground-truth pairs

We also record candidate-set size so that we can see the precision/efficiency trade-off.

In [39]:
# ============================================================
# S3 CANDIDATE RECALL — CURRENT BEST BLOCKER
# ============================================================

NAME_THRESHOLD_S3 = 50
ADDRESS_THRESHOLD_S3 = 100

s3_id_to_idx = {
    str(entity_id): idx
    for idx, entity_id in enumerate(
        S3_DEV["entity_id"].astype(str)
    )
}

s3_found_pairs = 0
s3_total_pairs = 0

candidate_counts_s3 = []
zero_candidate_s3 = 0

start_time = time.time()

for counter, (s1_id, true_ids) in enumerate(
    aligned_gt_s3.items(), start=1
):

    s1_row = validation_s1.get(s1_id)

    if s1_row is None:
        continue

    true_ids = {
        str(x)
        for x in true_ids
        if str(x) in s3_id_to_idx
    }

    if not true_ids:
        continue

    s3_total_pairs += len(true_ids)

    candidate_ids = set()

    # --------------------------------------------------------
    # 1. Original name + address blocker
    # --------------------------------------------------------

    original_candidates = find_candidates(
        s1_row,
        s3_name_index,
        s3_address_index
    )

    for row in original_candidates:
        candidate_ids.add(
            str(row["entity_id"])
        )

    # --------------------------------------------------------
    # 2. Name-token fallback
    # --------------------------------------------------------

    for token in set(
        get_name_tokens(s1_row["business_name"])
    ):

        if s3_name_token_freq.get(token, 0) <= NAME_THRESHOLD_S3:

            for idx in s3_name_postings.get(token, []):
                candidate_ids.add(
                    str(S3_DEV.iloc[idx]["entity_id"])
                )

    # --------------------------------------------------------
    # 3. Address-token fallback
    # --------------------------------------------------------

    for token in set(
        get_address_tokens(s1_row["business_address"])
    ):

        if (
            s3_address_token_freq.get(token, 0)
            <= ADDRESS_THRESHOLD_S3
        ):

            for idx in s3_address_postings.get(token, []):
                candidate_ids.add(
                    str(S3_DEV.iloc[idx]["entity_id"])
                )

    candidate_counts_s3.append(len(candidate_ids))

    if len(candidate_ids) == 0:
        zero_candidate_s3 += 1

    s3_found_pairs += len(
        true_ids & candidate_ids
    )

    # Progress indicator
    if counter % 10_000 == 0:
        elapsed = time.time() - start_time
        print(
            f"Processed {counter:,} / "
            f"{len(aligned_gt_s3):,} S1 | "
            f"elapsed {elapsed:.1f}s"
        )

elapsed_s3 = time.time() - start_time

s3_recall = (
    100 * s3_found_pairs / s3_total_pairs
    if s3_total_pairs
    else 0
)

print("\n" + "=" * 70)
print("S3 BLOCKING RESULTS")
print("=" * 70)

print("S1 validated       :", len(aligned_gt_s3))
print("GT pairs available :", s3_total_pairs)
print("GT pairs found     :", s3_found_pairs)
print("Candidate recall   :", f"{s3_recall:.4f}%")

print(
    "Average candidates :",
    f"{np.mean(candidate_counts_s3):.2f}"
)

print(
    "Median candidates  :",
    f"{np.median(candidate_counts_s3):.2f}"
)

print(
    "Maximum candidates:",
    max(candidate_counts_s3)
)

print(
    "Zero-candidate S1 :",
    zero_candidate_s3
)

print(
    "Runtime            :",
    f"{elapsed_s3:.2f} seconds"
)


S3 BLOCKING RESULTS
S1 validated       : 73486
GT pairs available : 57
GT pairs found     : 57
Candidate recall   : 100.0000%
Average candidates : 233.86
Median candidates  : 204.00
Maximum candidates: 702
Zero-candidate S1 : 0
Runtime            : 0.18 seconds


## 38. Build a Proper S3 Validation Set

The initial S3 test was limited by the existing 2,000-record S2 validation set.

Here we retrieve the actual S1 rows for the S1 entities that have S3 ground-truth matches available in the 100K S3 development set.

We will then evaluate the blocker on this substantially larger S3 validation population.

In [40]:
# ============================================================
# BUILD PROPER S3 VALIDATION S1 RECORDS
# ============================================================

s3_validation_ids = set(aligned_gt_s3.keys())

S1_PATH = TRAIN_DIR / "train_source1.tsv"

s3_validation_s1 = {}

start_time = time.time()

for chunk in pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False,
    chunksize=200_000
):
    mask = chunk["entity_id"].astype(str).isin(
        s3_validation_ids
    )

    selected = chunk.loc[mask]

    for _, row in selected.iterrows():
        s3_validation_s1[str(row["entity_id"])] = {
            "entity_id": str(row["entity_id"]),
            "business_name": row["business_name"],
            "business_address": row["business_address"],
            "country": row["country"]
        }

    if len(s3_validation_s1) == len(s3_validation_ids):
        break

print("=" * 70)
print("PROPER S3 VALIDATION SET")
print("=" * 70)

print("S3 GT S1 IDs available :", len(s3_validation_ids))
print("S1 records retrieved   :", len(s3_validation_s1))
print(
    "Retrieval time          :",
    f"{time.time() - start_time:.2f} seconds"
)

PROPER S3 VALIDATION SET
S3 GT S1 IDs available : 73486
S1 records retrieved   : 73486
Retrieval time          : 11.41 seconds


## 39. Final Candidate-Recall Evaluation

We evaluate the current blocker on the S1 population for which S3 matches are available in the 100K S3 development set.

For each S1, candidates are generated from:

1. Original name/address blocking
2. Name-token blocking with frequency ≤ 50
3. Address-token blocking with frequency ≤ 100

We measure:

- S2 candidate recall
- S3 candidate recall
- Combined S2+S3 candidate recall
- Average candidates per S1
- Median candidates per S1
- Maximum candidates
- Number of zero-candidate S1 records

The combined recall is calculated over all available S2 and S3 ground-truth pairs.

In [41]:
# ============================================================
# PROPER S2 + S3 + COMBINED RECALL
# ============================================================

NAME_THRESHOLD = 50
ADDRESS_THRESHOLD = 100

# ------------------------------------------------------------
# Build S2 GT for the same 73,486 S1 population
# ------------------------------------------------------------

s2_dev_ids = set(S2_DEV["entity_id"].astype(str))

aligned_gt_s2_common = {}

for _, row in gt_df.iterrows():

    s1_id = str(row["source1_entity_id"])

    if s1_id not in s3_validation_s1:
        continue

    matched = str(row["matched_entity_ids"])

    s2_ids = {
        x.strip()
        for x in matched.split(",")
        if x.strip().startswith("S2-")
    }

    available = s2_ids & s2_dev_ids

    if available:
        aligned_gt_s2_common[s1_id] = available


# ------------------------------------------------------------
# S3 index lookup already exists
# ------------------------------------------------------------

# s3_id_to_idx already created in Cell 37


# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

s2_total = 0
s2_found = 0

s3_total = 0
s3_found = 0

combined_total = 0
combined_found = 0

candidate_counts = []
zero_candidates = 0

start_time = time.time()

for counter, (s1_id, s1_row) in enumerate(
    s3_validation_s1.items(),
    start=1
):

    # --------------------------------------------------------
    # Ground truth
    # --------------------------------------------------------

    true_s2 = aligned_gt_s2_common.get(
        s1_id, set()
    )

    true_s3 = aligned_gt_s3.get(
        s1_id, set()
    )

    true_s2 = {
        str(x) for x in true_s2
    }

    true_s3 = {
        str(x) for x in true_s3
    }

    # --------------------------------------------------------
    # Candidate set
    # --------------------------------------------------------

    candidate_ids = set()

    # Original blocker
    original_candidates = find_candidates(
        s1_row,
        name_index,
        address_index
    )

    for row in original_candidates:
        candidate_ids.add(
            str(row["entity_id"])
        )

    # --------------------------------------------------------
    # S2 name-token blocker
    # --------------------------------------------------------

    for token in set(
        get_name_tokens(s1_row["business_name"])
    ):

        if token_freq.get(token, 0) <= NAME_THRESHOLD:

            for idx in token_postings.get(token, []):
                candidate_ids.add(
                    str(S2_DEV.iloc[idx]["entity_id"])
                )

    # --------------------------------------------------------
    # S2 address-token blocker
    # --------------------------------------------------------

    for token in set(
        get_address_tokens(s1_row["business_address"])
    ):

        if (
            address_token_freq.get(token, 0)
            <= ADDRESS_THRESHOLD
        ):

            for idx in address_token_postings.get(token, []):
                candidate_ids.add(
                    str(S2_DEV.iloc[idx]["entity_id"])
                )

    # --------------------------------------------------------
    # S3 original blocker
    # --------------------------------------------------------

    s3_original_candidates = find_candidates(
        s1_row,
        s3_name_index,
        s3_address_index
    )

    s3_candidate_ids = {
        str(row["entity_id"])
        for row in s3_original_candidates
    }

    # --------------------------------------------------------
    # S3 name-token blocker
    # --------------------------------------------------------

    for token in set(
        get_name_tokens(s1_row["business_name"])
    ):

        if (
            s3_name_token_freq.get(token, 0)
            <= NAME_THRESHOLD
        ):

            for idx in s3_name_postings.get(token, []):
                s3_candidate_ids.add(
                    str(S3_DEV.iloc[idx]["entity_id"])
                )

    # --------------------------------------------------------
    # S3 address-token blocker
    # --------------------------------------------------------

    for token in set(
        get_address_tokens(s1_row["business_address"])
    ):

        if (
            s3_address_token_freq.get(token, 0)
            <= ADDRESS_THRESHOLD
        ):

            for idx in s3_address_postings.get(token, []):
                s3_candidate_ids.add(
                    str(S3_DEV.iloc[idx]["entity_id"])
                )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    s2_total += len(true_s2)
    s2_found += len(true_s2 & candidate_ids)

    s3_total += len(true_s3)
    s3_found += len(true_s3 & s3_candidate_ids)

    combined_total += len(true_s2) + len(true_s3)

    combined_found += (
        len(true_s2 & candidate_ids)
        + len(true_s3 & s3_candidate_ids)
    )

    total_candidates_for_s1 = (
        len(candidate_ids) +
        len(s3_candidate_ids)
    )

    candidate_counts.append(total_candidates_for_s1)

    if total_candidates_for_s1 == 0:
        zero_candidates += 1

    if counter % 10_000 == 0:
        print(
            f"Processed {counter:,} / "
            f"{len(s3_validation_s1):,}"
        )


elapsed = time.time() - start_time

s2_recall = (
    100 * s2_found / s2_total
    if s2_total else 0
)

s3_recall = (
    100 * s3_found / s3_total
    if s3_total else 0
)

combined_recall = (
    100 * combined_found / combined_total
    if combined_total else 0
)

print("\n" + "=" * 70)
print("FINAL BLOCKER RECALL EVALUATION")
print("=" * 70)

print(f"S1 evaluated          : {len(s3_validation_s1):,}")

print("\nS2")
print(f"GT pairs available    : {s2_total:,}")
print(f"GT pairs found        : {s2_found:,}")
print(f"S2 recall             : {s2_recall:.4f}%")

print("\nS3")
print(f"GT pairs available    : {s3_total:,}")
print(f"GT pairs found        : {s3_found:,}")
print(f"S3 recall             : {s3_recall:.4f}%")

print("\nCOMBINED")
print(f"GT pairs available    : {combined_total:,}")
print(f"GT pairs found        : {combined_found:,}")
print(f"Combined recall       : {combined_recall:.4f}%")

print("\nCandidate statistics")
print(
    f"Average candidates    : "
    f"{np.mean(candidate_counts):.2f}"
)
print(
    f"Median candidates     : "
    f"{np.median(candidate_counts):.2f}"
)
print(
    f"Maximum candidates    : "
    f"{max(candidate_counts):,}"
)
print(
    f"Zero-candidate S1     : "
    f"{zero_candidates:,}"
)

print(f"\nRuntime               : {elapsed:.2f} seconds")

Processed 10,000 / 73,486
Processed 20,000 / 73,486
Processed 30,000 / 73,486
Processed 40,000 / 73,486
Processed 50,000 / 73,486
Processed 60,000 / 73,486
Processed 70,000 / 73,486

FINAL BLOCKER RECALL EVALUATION
S1 evaluated          : 73,486

S2
GT pairs available    : 2,488
GT pairs found        : 2,470
S2 recall             : 99.2765%

S3
GT pairs available    : 74,544
GT pairs found        : 73,831
S3 recall             : 99.0435%

COMBINED
GT pairs available    : 77,032
GT pairs found        : 76,301
Combined recall       : 99.0510%

Candidate statistics
Average candidates    : 962.91
Median candidates     : 467.00
Maximum candidates    : 11,641
Zero-candidate S1     : 0

Runtime               : 367.37 seconds


## 40. Inspect the Candidate-Scoring Functions

Candidate generation gives us a broad set of possible matches.

The next stage is candidate scoring, where each S1-S2/S3 pair receives a similarity score based on fields such as:

- Business name similarity
- Business address similarity
- Country agreement
- Token similarity
- Exact/normalized similarities

Before running this over thousands of candidate pairs, we inspect the scoring functions available in the provided solution.

In [42]:
# ============================================================
# INSPECT AVAILABLE SCORING FUNCTIONS
# ============================================================

import inspect
import similarity

print("=" * 70)
print("AVAILABLE SIMILARITY FUNCTIONS")
print("=" * 70)

functions_to_check = [
    "text_similarity",
    "token_similarity",
    "name_similarity",
    "address_similarity",
    "country_match",
    "calculate_similarity"
]

for fn_name in functions_to_check:

    fn = getattr(similarity, fn_name, None)

    print(f"\n{fn_name}")

    if fn is None:
        print("  NOT FOUND")
    else:
        print("  Signature:", inspect.signature(fn))
        print("  Available:", True)

print("\nModule:", similarity.__file__)

AVAILABLE SIMILARITY FUNCTIONS

text_similarity
  Signature: (text1, text2)
  Available: True

token_similarity
  Signature: (text1, text2)
  Available: True

name_similarity
  Signature: (name1, name2)
  Available: True

address_similarity
  Signature: (address1, address2)
  Available: True

country_match
  Signature: (country1, country2)
  Available: True

calculate_similarity
  Signature: (row1, row2)
  Available: True

Module: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/amazon_ml_solution/src/similarity.py


## 41. Inspect the Similarity Calculation

We now inspect the implementation of `calculate_similarity()` and its underlying similarity functions.

This is important because the original development benchmark showed that pure-Python edit-distance calculations can become very slow at large candidate volumes.

We will verify:

- Which features are calculated
- How name similarity is computed
- How address similarity is computed
- Whether country matching is included
- Whether the function returns a single score or multiple features

In [43]:
# ============================================================
# INSPECT SIMILARITY IMPLEMENTATION
# ============================================================

import inspect
import similarity

print("=" * 70)
print("calculate_similarity SOURCE")
print("=" * 70)

print(
    inspect.getsource(
        similarity.calculate_similarity
    )
)

print("\n" + "=" * 70)
print("name_similarity SOURCE")
print("=" * 70)

print(
    inspect.getsource(
        similarity.name_similarity
    )
)

print("\n" + "=" * 70)
print("address_similarity SOURCE")
print("=" * 70)

print(
    inspect.getsource(
        similarity.address_similarity
    )
)

print("\n" + "=" * 70)
print("text_similarity SOURCE")
print("=" * 70)

print(
    inspect.getsource(
        similarity.text_similarity
    )
)

calculate_similarity SOURCE
def calculate_similarity(row1, row2):
    """
    Calculate similarity features.
    """

    name_score = name_similarity(
        row1["business_name"],
        row2["business_name"]
    )

    address_score = address_similarity(
        row1["business_address"],
        row2["business_address"]
    )

    address_number_score = address_number_match(
        row1["business_address"],
        row2["business_address"]
    )

    country_score = country_match(
        row1["country"],
        row2["country"]
    )

    return {
        "name_similarity": name_score,
        "address_similarity": address_score,
        "address_number_match": address_number_score,
        "country_match": country_score
    }


name_similarity SOURCE
def name_similarity(name1, name2):
    """
    Combine character and token similarity.
    """

    name1 = normalize_business_name(name1)
    name2 = normalize_business_name(name2)

    char_score = text_similarity(
        name1,

## 42. Benchmark Candidate Scoring Speed

Before scoring the full validation set, we benchmark the existing
`calculate_similarity()` function on a small sample of generated S2
candidates.

This measures:

- Number of candidate pairs
- Scoring time
- Candidate-pair scoring rate

We do not train or generate final predictions yet.

The purpose is to determine whether the provided scorer is practical
for the next validation stage.

In [44]:
# ============================================================
# BENCHMARK EXISTING SCORER
# ============================================================

from similarity import calculate_similarity

BENCHMARK_S1_COUNT = 100

benchmark_s1_items = list(
    validation_s1.items()
)[:BENCHMARK_S1_COUNT]

benchmark_pairs = []

print(
    f"Generating candidates for "
    f"{len(benchmark_s1_items)} S1 records..."
)

for s1_id, s1_row in benchmark_s1_items:

    candidates = find_candidates(
        s1_row,
        name_index,
        address_index
    )

    for candidate in candidates:
        benchmark_pairs.append(
            (s1_id, s1_row, candidate)
        )

print(
    "Candidate pairs:",
    f"{len(benchmark_pairs):,}"
)

# ------------------------------------------------------------
# Score the candidate pairs
# ------------------------------------------------------------

score_start = time.time()

benchmark_scores = []

for s1_id, s1_row, candidate in benchmark_pairs:

    features = calculate_similarity(
        s1_row,
        candidate
    )

    benchmark_scores.append(
        features
    )

score_time = time.time() - score_start

pairs_per_second = (
    len(benchmark_pairs) / score_time
    if score_time > 0
    else 0
)

print("\n" + "=" * 70)
print("SCORER BENCHMARK")
print("=" * 70)

print(
    "S1 records scored      :",
    len(benchmark_s1_items)
)

print(
    "Candidate pairs scored :",
    f"{len(benchmark_pairs):,}"
)

print(
    "Scoring time            :",
    f"{score_time:.2f} seconds"
)

print(
    "Pairs / second          :",
    f"{pairs_per_second:,.0f}"
)

if benchmark_scores:
    print("\nExample feature output:")
    print(benchmark_scores[0])

Generating candidates for 100 S1 records...
Candidate pairs: 48,387

SCORER BENCHMARK
S1 records scored      : 100
Candidate pairs scored : 48,387
Scoring time            : 15.09 seconds
Pairs / second          : 3,208

Example feature output:
{'name_similarity': 0.4577777777777778, 'address_similarity': 0.4, 'address_number_match': 0.0, 'country_match': 1.0}


## 43. Build a Labeled Scoring Benchmark

We now connect candidate generation with the training ground truth.

For the 2,000-S1 S2 validation set:

1. Generate candidates using the current blocker.
2. Calculate the four similarity features.
3. Label each candidate as:
   - `1` = ground-truth match
   - `0` = non-match
4. Create a combined matching score.
5. Evaluate different score thresholds using F0.5.

The threshold will be selected from measured validation results rather than assumed in advance.

In [45]:
# ============================================================
# BUILD LABELED S2 SCORING BENCHMARK
# ============================================================

from similarity import calculate_similarity

# Use the 2,000-S1 validation set
SCORING_S1 = validation_s1
SCORING_GT = aligned_gt

# Current blocker configuration
NAME_THRESHOLD = 50
ADDRESS_THRESHOLD = 100

scoring_rows = []

start_time = time.time()

for counter, (s1_id, s1_row) in enumerate(
    SCORING_S1.items(),
    start=1
):

    true_ids = {
        str(x)
        for x in SCORING_GT.get(s1_id, set())
    }

    candidate_ids = {}

    # --------------------------------------------------------
    # Original blocker
    # --------------------------------------------------------

    original_candidates = find_candidates(
        s1_row,
        name_index,
        address_index
    )

    for row in original_candidates:
        candidate_ids[str(row["entity_id"])] = row

    # --------------------------------------------------------
    # Name-token blocker
    # --------------------------------------------------------

    for token in set(
        get_name_tokens(s1_row["business_name"])
    ):

        if token_freq.get(token, 0) <= NAME_THRESHOLD:

            for idx in token_postings.get(token, []):
                row = S2_DEV.iloc[idx]
                candidate_ids[str(row["entity_id"])] = row

    # --------------------------------------------------------
    # Address-token blocker
    # --------------------------------------------------------

    for token in set(
        get_address_tokens(s1_row["business_address"])
    ):

        if (
            address_token_freq.get(token, 0)
            <= ADDRESS_THRESHOLD
        ):

            for idx in address_token_postings.get(token, []):
                row = S2_DEV.iloc[idx]
                candidate_ids[str(row["entity_id"])] = row

    # --------------------------------------------------------
    # Score candidates
    # --------------------------------------------------------

    for candidate_id, candidate_row in candidate_ids.items():

        features = calculate_similarity(
            s1_row,
            candidate_row
        )

        # Transparent baseline score.
        # We will NOT assume this is optimal;
        # thresholds will be evaluated afterward.
        baseline_score = (
            0.50 * features["name_similarity"]
            + 0.30 * features["address_similarity"]
            + 0.15 * features["address_number_match"]
            + 0.05 * features["country_match"]
        )

        scoring_rows.append({
            "s1_id": s1_id,
            "candidate_id": candidate_id,
            "name_similarity":
                features["name_similarity"],
            "address_similarity":
                features["address_similarity"],
            "address_number_match":
                features["address_number_match"],
            "country_match":
                features["country_match"],
            "score": baseline_score,
            "label":
                int(candidate_id in true_ids)
        })

    if counter % 250 == 0:
        print(
            f"Processed {counter:,} / "
            f"{len(SCORING_S1):,} S1"
        )

print("\nFinished.")

scoring_df = pd.DataFrame(scoring_rows)

print("\n" + "=" * 70)
print("LABELED SCORING DATASET")
print("=" * 70)

print("Rows:", f"{len(scoring_df):,}")
print("Columns:", list(scoring_df.columns))

print(
    "Positive pairs:",
    int(scoring_df["label"].sum())
)

print(
    "Runtime:",
    f"{time.time() - start_time:.2f} seconds"
)

display(scoring_df.head())

Processed 250 / 2,000 S1
Processed 500 / 2,000 S1
Processed 750 / 2,000 S1
Processed 1,000 / 2,000 S1
Processed 1,250 / 2,000 S1
Processed 1,500 / 2,000 S1
Processed 1,750 / 2,000 S1
Processed 2,000 / 2,000 S1

Finished.

LABELED SCORING DATASET
Rows: 1,200,080
Columns: ['s1_id', 'candidate_id', 'name_similarity', 'address_similarity', 'address_number_match', 'country_match', 'score', 'label']
Positive pairs: 2021
Runtime: 401.10 seconds


,s1_id,candidate_id,name_similarity,address_similarity,address_number_match,country_match,score,label
0,S1-736761428,S2-507704636,0.457778,0.400000,0.0,1.0,0.398889,0
1,S1-736761428,S2-658347451,0.373333,0.153846,0.0,1.0,0.282821,0
2,S1-736761428,S2-910842372,0.322727,0.259259,0.0,1.0,0.289141,0
3,S1-736761428,S2-439161022,0.454545,0.307692,0.0,1.0,0.369580,0
4,S1-736761428,S2-378011585,0.447368,0.350877,0.0,1.0,0.378947,0


## 44. Evaluate Matching Thresholds

The labeled candidate dataset contains both true and false candidate pairs.

We now evaluate the baseline similarity score at multiple thresholds.

For each threshold:

- candidates with `score >= threshold` are predicted as matches
- precision is calculated
- recall is calculated
- F0.5 is calculated

Because the competition metric is precision-heavy, F0.5 is used for model selection.

This experiment only uses the already-computed validation dataset; no additional candidate generation or similarity calculation is required.

In [46]:
# ============================================================
# THRESHOLD SWEEP
# ============================================================

def evaluate_threshold(df, threshold):

    predicted = df["score"] >= threshold
    actual = df["label"] == 1

    tp = int((predicted & actual).sum())
    fp = int((predicted & ~actual).sum())
    fn = int((~predicted & actual).sum())

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    if precision + recall == 0:
        f05 = 0.0
    else:
        f05 = (
            1.25 * precision * recall
            / (0.25 * precision + recall)
        )

    return {
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "predicted_pairs": int(predicted.sum()),
        "precision": precision,
        "recall": recall,
        "F0.5": f05,
    }


thresholds = np.arange(
    0.30,
    0.951,
    0.025
)

results = [
    evaluate_threshold(scoring_df, t)
    for t in thresholds
]

threshold_df = pd.DataFrame(results)

print("=" * 70)
print("THRESHOLD SWEEP")
print("=" * 70)

display(
    threshold_df.style.format({
        "threshold": "{:.3f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "F0.5": "{:.4f}",
    })
)

best_row = threshold_df.loc[
    threshold_df["F0.5"].idxmax()
]

print("\n" + "=" * 70)
print("BEST VALIDATION THRESHOLD")
print("=" * 70)

print(
    f"Threshold : {best_row['threshold']:.3f}"
)
print(
    f"Precision : {best_row['precision']:.4f}"
)
print(
    f"Recall    : {best_row['recall']:.4f}"
)
print(
    f"F0.5      : {best_row['F0.5']:.4f}"
)
print(
    f"TP        : {int(best_row['TP']):,}"
)
print(
    f"FP        : {int(best_row['FP']):,}"
)
print(
    f"FN        : {int(best_row['FN']):,}"
)

THRESHOLD SWEEP


,threshold,TP,FP,FN,predicted_pairs,precision,recall,F0.5
0,0.300,1972,440003,49,441975,0.0045,0.9758,0.0056
1,0.325,1960,303924,61,305884,0.0064,0.9698,0.0080
2,0.350,1940,196436,81,198376,0.0098,0.9599,0.0122
3,0.375,1927,120622,94,122549,0.0157,0.9535,0.0196
4,0.400,1919,71095,102,73014,0.0263,0.9495,0.0326
5,0.425,1895,39348,126,41243,0.0459,0.9377,0.0567
6,0.450,1869,20842,152,22711,0.0823,0.9248,0.1006
7,0.475,1840,10812,181,12652,0.1454,0.9104,0.1748
8,0.500,1771,5331,250,7102,0.2494,0.8763,0.2910
9,0.525,1742,2591,279,4333,0.4020,0.8619,0.4501



BEST VALIDATION THRESHOLD
Threshold : 0.700
Precision : 0.9658
Recall    : 0.6710
F0.5      : 0.8878
TP        : 1,356
FP        : 48
FN        : 665


## 45A. Load Ground Truth for Competition-Style Evaluation

The scoring dataset has already been generated.

The variable `validation_gt` was not created in the current notebook session, so we load the ground-truth file now.

This does not regenerate candidates or calculate similarities.

In [47]:
# ============================================================
# LOAD GROUND TRUTH
# ============================================================

GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"

validation_gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str
)

print("=" * 70)
print("GROUND TRUTH LOADED")
print("=" * 70)

print("Rows:", len(validation_gt))
print("Columns:", validation_gt.columns.tolist())

display(validation_gt.head())

GROUND TRUTH LOADED
Rows: 2206821
Columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


## 45B. Verify Validation Ground Truth Alignment

Before calculating the competition metric, we verify that the S1 records used in the scoring dataset are present in the ground truth.

This prevents accidental evaluation against the wrong validation population.

In [48]:
# ============================================================
# VERIFY GT ALIGNMENT
# ============================================================

scoring_s1_ids = set(scoring_df["s1_id"].unique())
gt_s1_ids = set(validation_gt["source1_entity_id"].unique())

common_ids = scoring_s1_ids & gt_s1_ids
missing_ids = scoring_s1_ids - gt_s1_ids

print("=" * 70)
print("GROUND TRUTH ALIGNMENT")
print("=" * 70)

print("S1s in scoring_df :", len(scoring_s1_ids))
print("S1s in GT         :", len(gt_s1_ids))
print("Common S1s        :", len(common_ids))
print("Missing from GT    :", len(missing_ids))

if len(missing_ids) == 0:
    print("\n✓ ALL SCORING S1s HAVE GROUND TRUTH")
else:
    print("\n⚠ Some scoring S1s are missing from GT")
    print("Example:", list(missing_ids)[:10])

GROUND TRUTH ALIGNMENT
S1s in scoring_df : 2000
S1s in GT         : 2206821
Common S1s        : 2000
Missing from GT    : 0

✓ ALL SCORING S1s HAVE GROUND TRUTH


## 45C. Competition-Style Macro F0.5 Evaluation

The validation set is correctly aligned:

- 2,000 S1 records
- every S1 has a ground-truth record
- 1,200,080 candidate pairs have already been scored

We now evaluate each threshold using the competition's per-S1 macro F0.5 approach.

No new candidate generation or similarity calculation is performed.

In [49]:
# ============================================================
# COMPETITION-STYLE MACRO F0.5
# ============================================================

# Build ground-truth sets
gt_sets = {}

for _, row in validation_gt.iterrows():

    s1_id = row["source1_entity_id"]
    matched = row["matched_entity_ids"]

    if pd.isna(matched) or str(matched).strip() == "":
        gt_sets[s1_id] = set()
    else:
        gt_sets[s1_id] = {
            x.strip()
            for x in str(matched).split(",")
            if x.strip()
        }


eval_s1_ids = scoring_df["s1_id"].unique()


def macro_f05_at_threshold(df, threshold):

    predicted_df = df[df["score"] >= threshold]

    predicted_sets = (
        predicted_df
        .groupby("s1_id")["candidate_id"]
        .apply(set)
        .to_dict()
    )

    f05_values = []
    precision_values = []
    recall_values = []

    for s1_id in eval_s1_ids:

        predicted = predicted_sets.get(s1_id, set())
        actual = gt_sets.get(s1_id, set())

        tp = len(predicted & actual)
        fp = len(predicted - actual)
        fn = len(actual - predicted)

        # Empty prediction + empty ground truth = perfect match
        if len(predicted) == 0 and len(actual) == 0:
            precision = 1.0
            recall = 1.0
            f05 = 1.0

        else:

            precision = (
                tp / (tp + fp)
                if (tp + fp) > 0
                else 0.0
            )

            recall = (
                tp / (tp + fn)
                if (tp + fn) > 0
                else 0.0
            )

            if precision + recall == 0:
                f05 = 0.0
            else:
                f05 = (
                    1.25 * precision * recall
                    / (0.25 * precision + recall)
                )

        f05_values.append(f05)
        precision_values.append(precision)
        recall_values.append(recall)

    return {
        "threshold": threshold,
        "macro_F0.5": np.mean(f05_values),
        "macro_precision": np.mean(precision_values),
        "macro_recall": np.mean(recall_values),
    }


# ------------------------------------------------------------
# Evaluate thresholds
# ------------------------------------------------------------

macro_results = []

for threshold in np.arange(0.45, 0.951, 0.025):

    macro_results.append(
        macro_f05_at_threshold(
            scoring_df,
            threshold
        )
    )


macro_threshold_df = pd.DataFrame(macro_results)


print("=" * 70)
print("COMPETITION-STYLE MACRO F0.5")
print("=" * 70)

display(
    macro_threshold_df.style.format({
        "threshold": "{:.3f}",
        "macro_F0.5": "{:.4f}",
        "macro_precision": "{:.4f}",
        "macro_recall": "{:.4f}",
    })
)


best_macro = macro_threshold_df.loc[
    macro_threshold_df["macro_F0.5"].idxmax()
]


print("\n" + "=" * 70)
print("BEST MACRO F0.5 THRESHOLD")
print("=" * 70)

print(f"Threshold       : {best_macro['threshold']:.3f}")
print(f"Macro Precision : {best_macro['macro_precision']:.4f}")
print(f"Macro Recall    : {best_macro['macro_recall']:.4f}")
print(f"Macro F0.5      : {best_macro['macro_F0.5']:.4f}")

COMPETITION-STYLE MACRO F0.5


,threshold,macro_F0.5,macro_precision,macro_recall
0,0.450,0.2955,0.4017,0.2522
1,0.475,0.3637,0.5210,0.2479
2,0.500,0.4101,0.6079,0.2386
3,0.525,0.4473,0.6780,0.2347
4,0.550,0.4672,0.7198,0.2304
5,0.575,0.4713,0.7325,0.2253
6,0.600,0.4664,0.7296,0.2188
7,0.625,0.4515,0.7105,0.2070
8,0.650,0.4483,0.7097,0.2018
9,0.675,0.4361,0.6948,0.1929



BEST MACRO F0.5 THRESHOLD
Threshold       : 0.575
Macro Precision : 0.7325
Macro Recall    : 0.2253
Macro F0.5      : 0.4713


## 46A. Check Existing Validation Ground Truth

We already know the validation S1 population and the scoring results.

Before performing any additional large-data operation, we check whether a smaller ground-truth dataframe from the earlier validation stage is still available in memory.

In [50]:
# Check which useful validation objects still exist

names_to_check = [
    "validation_gt",
    "gt_df",
    "validation_s1",
    "validation_s1_ids",
    "scoring_df",
]

for name in names_to_check:
    exists = name in globals()

    if exists:
        obj = globals()[name]

        try:
            shape = obj.shape
        except Exception:
            shape = "N/A"

        print(f"{name:20s} EXISTS   {shape}")
    else:
        print(f"{name:20s} missing")

validation_gt        EXISTS   (2206821, 2)
gt_df                EXISTS   (2206821, 2)
validation_s1        EXISTS   N/A
validation_s1_ids    missing
scoring_df           EXISTS   (1200080, 8)


## 46B. Find Existing Small Validation Ground Truth

The notebook may already contain the smaller ground-truth objects created during the earlier 2,000-S1 candidate-recall validation.

We inspect variable names and sizes only; no large dataframe operations are performed.

In [51]:
# ============================================================
# FIND SMALL GT / VALIDATION OBJECTS
# ============================================================

for name, obj in list(globals().items()):

    name_lower = name.lower()

    if any(x in name_lower for x in [
        "gt", "ground", "pair", "match", "validation"
    ]):

        try:
            shape = obj.shape

            # Only show reasonably small objects
            if shape[0] < 100_000:
                print(f"{name:35s} shape={shape}")

        except Exception:
            pass

## 46C. Fast Threshold Analysis Using Existing Candidate Labels

The candidate-generation validation already established that the blocker found 2,021 true S2 pairs.

We now analyze the score distribution of those known positive pairs and the false candidate pairs already present in `scoring_df`.

This uses only the 1.2M-row scoring dataframe already in memory and does not touch the 2.2M-row ground-truth dataframe.

In [52]:
# ============================================================
# FAST SCORE DISTRIBUTION CHECK
# ============================================================

positive_scores = scoring_df.loc[
    scoring_df["label"] == 1,
    "score"
]

negative_scores = scoring_df.loc[
    scoring_df["label"] == 0,
    "score"
]

print("=" * 70)
print("SCORE DISTRIBUTION")
print("=" * 70)

print("\nPOSITIVE PAIRS:", len(positive_scores))
print(
    positive_scores.describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nNEGATIVE PAIRS:", len(negative_scores))
print(
    negative_scores.describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\n" + "=" * 70)
print("POSITIVE PAIRS BELOW THRESHOLD")
print("=" * 70)

for t in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:

    missed = int((positive_scores < t).sum())
    kept = len(positive_scores) - missed

    print(
        f"Threshold {t:.2f} : "
        f"kept={kept:,}  "
        f"missed={missed:,}  "
        f"recall={kept / len(positive_scores):.4f}"
    )

SCORE DISTRIBUTION

POSITIVE PAIRS: 2021
count    2021.000000
mean        0.756027
std         0.187785
min         0.129720
1%          0.230393
5%          0.397015
10%         0.485360
25%         0.642436
50%         0.797059
75%         0.894474
90%         0.988462
95%         1.000000
99%         1.000000
max         1.000000
Name: score, dtype: float64

NEGATIVE PAIRS: 1198059
count    1.198059e+06
mean     2.803701e-01
std      7.277178e-02
min      1.224490e-02
1%       1.352632e-01
5%       1.692053e-01
10%      1.890728e-01
25%      2.306994e-01
50%      2.753201e-01
75%      3.258491e-01
90%      3.753759e-01
95%      4.076333e-01
99%      4.712023e-01
max      8.293103e-01
Name: score, dtype: float64

POSITIVE PAIRS BELOW THRESHOLD
Threshold 0.50 : kept=1,771  missed=250  recall=0.8763
Threshold 0.55 : kept=1,707  missed=314  recall=0.8446
Threshold 0.60 : kept=1,625  missed=396  recall=0.8041
Threshold 0.65 : kept=1,502  missed=519  recall=0.7432
Threshold 0.70 : kept=1,

## 48. Check Accelerated Similarity Library

The current similarity calculation uses Python's `difflib.SequenceMatcher`,
which is too slow for full-scale inference.

We check whether RapidFuzz is already installed in the Kaggle environment.

If available, it can provide much faster string similarity calculations
while keeping the same overall feature design.

In [53]:
# ============================================================
# CHECK RAPIDFUZZ
# ============================================================

try:
    import rapidfuzz

    print("=" * 70)
    print("RAPIDFUZZ AVAILABLE")
    print("=" * 70)
    print("Version:", rapidfuzz.__version__)

except Exception as e:

    print("=" * 70)
    print("RAPIDFUZZ NOT AVAILABLE")
    print("=" * 70)
    print(type(e).__name__, ":", e)

RAPIDFUZZ NOT AVAILABLE
ModuleNotFoundError : No module named 'rapidfuzz'


## 49. Inspect Existing Normalization Functions

RapidFuzz cannot be installed because the Kaggle environment currently has no external package-network access.

Instead, we will optimize the existing scorer using the normalization and blocking information already available in the notebook.

Before changing the scorer, we inspect the existing normalization functions so that the accelerated approach remains compatible with the current pipeline.

In [54]:
# ============================================================
# INSPECT NORMALIZATION FUNCTIONS
# ============================================================

import inspect

print("=" * 70)
print("AVAILABLE NORMALIZATION FUNCTIONS")
print("=" * 70)

for name in [
    "normalize_business_name",
    "normalize_address",
    "text_similarity",
    "token_similarity",
    "name_similarity",
    "address_similarity",
    "calculate_similarity",
]:

    if name in globals():

        obj = globals()[name]

        print(f"\n{name}")
        print("-" * 50)

        try:
            print(inspect.getsource(obj))
        except Exception:
            print("Source unavailable")

AVAILABLE NORMALIZATION FUNCTIONS

normalize_business_name
--------------------------------------------------
def normalize_business_name(name):
    """
    Normalize a business name.
    """

    name = normalize_text(name)

    # Common legal/business suffixes
    suffixes = [
        "incorporated",
        "inc",
        "corporation",
        "corp",
        "limited",
        "ltd",
        "llc",
        "company",
        "co",
        "private limited",
        "pvt ltd",
    ]

    words = name.split()

    # Remove suffixes from the end
    while words and " ".join(words[-2:]) in suffixes:
        words = words[:-2]

    while words and words[-1] in suffixes:
        words = words[:-1]

    return " ".join(words)


normalize_address
--------------------------------------------------
def normalize_address(address):
    """
    Normalize a business address.
    """

    address = normalize_text(address)

    # Common address abbreviations
    replacements = {
        " street 

## 50. Benchmark Fast TF-IDF Similarity

RapidFuzz cannot be installed in the current Kaggle environment.

We therefore test scikit-learn's TF-IDF + cosine similarity as a faster alternative for comparing business names.

This is only a small benchmark. We do NOT rebuild the 1.2M-row scoring dataset.

If the benchmark is sufficiently fast, we can use TF-IDF as a cheap first-stage similarity and reserve the slower SequenceMatcher for only a small number of candidates.

In [55]:
# ============================================================
# SMALL TF-IDF BENCHMARK
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time

# Take a small sample of existing candidate pairs
sample = scoring_df.sample(
    n=min(5000, len(scoring_df)),
    random_state=42
)

print("=" * 70)
print("TF-IDF BENCHMARK")
print("=" * 70)

print("Sample pairs:", len(sample))

# We need the original text for the candidates.
# Check whether the required source dataframes are already available.

available = [
    name for name in [
        "validation_s1",
        "train_source1",
        "train_source2",
        "source2_df",
        "s2_df"
    ]
    if name in globals()
]

print("Potential source objects available:", available)

TF-IDF BENCHMARK
Sample pairs: 5000
Potential source objects available: ['validation_s1']


## 51. Benchmark Current Scorer on 5,000 Candidates

The full scoring run took about 423 seconds for 1.2 million candidate pairs.

We benchmark the existing similarity calculation on only 5,000 pairs to establish its per-pair cost.

No new candidate generation is performed.

In [56]:
# ============================================================
# 5,000-PAIR SCORER BENCHMARK
# ============================================================

import time

sample = scoring_df.sample(
    n=min(5000, len(scoring_df)),
    random_state=42
)

print("=" * 70)
print("CURRENT SCORER BENCHMARK")
print("=" * 70)

print("Sample pairs:", len(sample))

# validation_s1 is already available.
# We need the S2 source rows for the candidate IDs.
# Check whether an S2 dataframe/index is already available.

possible_s2 = [
    name for name in globals()
    if "s2" in name.lower()
]

print("Possible S2 objects:", possible_s2)

CURRENT SCORER BENCHMARK
Sample pairs: 5000
Possible S2 objects: ['TEST_S2', 'TRAIN_S2', 'BENCH_S2_ROWS', 'TARGET_S2', 's2_target', 's2_name_keys', 's2_address_keys', 's2_name_set', 's2_addr_set', 's2_100k_ids', 's2_lookup_100k', 's2_id', 's2', 's2_sig', 's2_tokens', 'S2_PATH', 'S2_DEV', 's2_id_to_idx', 'raw_s2', 's2_id_set', 's2_lookup', 'true_s2_ids', 's2_row', 's2_addr_tokens', 's2_dev_ids', 'aligned_gt_s2_common', 's2_ids', 's2_total', 's2_found', 'true_s2', 's2_recall']


## 52. Benchmark Current Similarity Scorer

The existing validation objects have been identified:

- `s2_lookup` is a pandas DataFrame containing the S2 records.
- `validation_s1` is a dictionary containing the validation S1 records.

We construct lightweight ID lookups from these existing objects and benchmark the current similarity scorer on only 5,000 candidate pairs.

No source files are reloaded.

In [57]:
# ============================================================
# CURRENT SCORER — 5,000 PAIR BENCHMARK
# ============================================================

import time

sample = scoring_df.sample(
    n=min(5000, len(scoring_df)),
    random_state=42
)

# ------------------------------------------------------------
# S1 lookup
# ------------------------------------------------------------

if isinstance(validation_s1, dict):
    s1_lookup_bench = {
        str(k): v
        for k, v in validation_s1.items()
    }
else:
    s1_lookup_bench = {}


# ------------------------------------------------------------
# S2 lookup from DataFrame
# ------------------------------------------------------------

s2_lookup_bench = {
    str(row.entity_id): {
        "entity_id": row.entity_id,
        "business_name": row.business_name,
        "business_address": row.business_address,
        "country": row.country,
    }
    for row in s2_lookup.itertuples(index=False)
}


print("=" * 70)
print("LOOKUP CHECK")
print("=" * 70)

print("S1 lookup:", len(s1_lookup_bench))
print("S2 lookup:", len(s2_lookup_bench))

# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

start = time.time()

computed = 0

for row in sample.itertuples(index=False):

    s1_row = s1_lookup_bench.get(str(row.s1_id))
    s2_row = s2_lookup_bench.get(str(row.candidate_id))

    if s1_row is None or s2_row is None:
        continue

    calculate_similarity(s1_row, s2_row)

    computed += 1


elapsed = time.time() - start

print("\n" + "=" * 70)
print("BENCHMARK RESULT")
print("=" * 70)

print("Pairs requested :", len(sample))
print("Pairs computed  :", computed)
print("Runtime         :", round(elapsed, 2), "seconds")

if elapsed > 0:
    print(
        "Pairs/second    :",
        round(computed / elapsed, 2)
    )

LOOKUP CHECK
S1 lookup: 2000
S2 lookup: 100000

BENCHMARK RESULT
Pairs requested : 5000
Pairs computed  : 5000
Runtime         : 1.68 seconds
Pairs/second    : 2981.16


## 53. Analyze Score Separation

Before redesigning the scorer, we measure how many candidate pairs survive
simple score thresholds.

This helps determine whether a two-stage pipeline can reduce expensive
similarity calculations:

1. Generate candidates using the high-recall blocker.
2. Apply a cheap preliminary filter.
3. Run the expensive similarity scorer only on survivors.

The existing 1.2M-row scoring dataset is reused; no new scoring is performed.

In [58]:
# ============================================================
# TWO-STAGE FILTER ANALYSIS
# ============================================================

print("=" * 70)
print("CANDIDATE REDUCTION")
print("=" * 70)

total = len(scoring_df)

print(f"Total candidates: {total:,}")

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:

    survivors = int(
        (scoring_df["score"] >= threshold).sum()
    )

    positives = int(
        (
            (scoring_df["score"] >= threshold)
            & (scoring_df["label"] == 1)
        ).sum()
    )

    total_positive = int(
        scoring_df["label"].sum()
    )

    recall = (
        positives / total_positive
        if total_positive
        else 0
    )

    reduction = 1 - survivors / total

    print(
        f"\nThreshold {threshold:.2f}"
        f"\n  Survivors : {survivors:,}"
        f"\n  Reduction : {reduction:.2%}"
        f"\n  Positives : {positives:,}"
        f"\n  Recall    : {recall:.4f}"
    )

CANDIDATE REDUCTION
Total candidates: 1,200,080

Threshold 0.30
  Survivors : 441,975
  Reduction : 63.17%
  Positives : 1,972
  Recall    : 0.9758

Threshold 0.35
  Survivors : 198,826
  Reduction : 83.43%
  Positives : 1,940
  Recall    : 0.9599

Threshold 0.40
  Survivors : 73,052
  Reduction : 93.91%
  Positives : 1,919
  Recall    : 0.9495

Threshold 0.45
  Survivors : 22,742
  Reduction : 98.10%
  Positives : 1,869
  Recall    : 0.9248

Threshold 0.50
  Survivors : 7,111
  Reduction : 99.41%
  Positives : 1,771
  Recall    : 0.8763

Threshold 0.55
  Survivors : 3,087
  Reduction : 99.74%
  Positives : 1,707
  Recall    : 0.8446

Threshold 0.60
  Survivors : 2,354
  Reduction : 99.80%
  Positives : 1,625
  Recall    : 0.8041


## 54. Inspect Candidate Features Available Before Fuzzy Scoring

The existing final score provides strong candidate separation, but calculating it
for every pair is expensive.

We now inspect the candidate-generation features already available in the
notebook.

The goal is to identify cheap signals that can be applied before
SequenceMatcher, while preserving as much candidate recall as possible.

In [59]:
# ============================================================
# INSPECT EXISTING BLOCKING OBJECTS
# ============================================================

print("=" * 70)
print("AVAILABLE BLOCKING OBJECTS")
print("=" * 70)

names = [
    "s2_name_keys",
    "s2_address_keys",
    "s2_name_set",
    "s2_addr_set",
    "s2_tokens",
    "s2_addr_tokens",
    "s2_name_token_index",
    "s2_address_token_index",
    "name_token_index",
    "address_token_index",
    "validation_s1",
    "s2_lookup",
]

for name in names:

    if name in globals():

        obj = globals()[name]

        try:
            print(
                f"{name:28s} "
                f"type={type(obj).__name__:<15s} "
                f"size/shape={getattr(obj, 'shape', len(obj) if hasattr(obj, '__len__') else 'N/A')}"
            )
        except Exception:
            print(
                f"{name:28s} "
                f"type={type(obj).__name__}"
            )

AVAILABLE BLOCKING OBJECTS
s2_name_keys                 type=list            size/shape=6
s2_address_keys              type=list            size/shape=0
s2_name_set                  type=set             size/shape=6
s2_addr_set                  type=set             size/shape=0
s2_tokens                    type=set             size/shape=0
s2_addr_tokens               type=set             size/shape=3
validation_s1                type=dict            size/shape=2000
s2_lookup                    type=DataFrame       size/shape=(100000, 4)


## 55. Inspect the Actual Candidate-Generation Function

The notebook contains several earlier experimental blocking objects.

Rather than relying on those small validation objects, we inspect the actual
candidate-generation function currently available in the notebook.

This lets us identify the exact inputs required to reproduce the high-recall
blocker for the final pipeline.

No large dataset is processed in this cell.


In [60]:
# ============================================================
# INSPECT ACTUAL CANDIDATE GENERATOR
# ============================================================

import inspect

candidate_functions = [
    "find_candidates",
    "generate_candidates",
    "build_index",
    "candidate_generator",
]

print("=" * 70)
print("AVAILABLE CANDIDATE-GENERATION FUNCTIONS")
print("=" * 70)

for name in candidate_functions:

    if name in globals():

        obj = globals()[name]

        print("\n" + name)
        print("-" * 70)

        try:
            print(inspect.signature(obj))
        except Exception:
            print("Signature unavailable")

        try:
            print(inspect.getsource(obj)[:5000])
        except Exception:
            print("Source unavailable")

AVAILABLE CANDIDATE-GENERATION FUNCTIONS

find_candidates
----------------------------------------------------------------------
(source1_row, name_index, address_index)
def find_candidates(
    source1_row,
    name_index,
    address_index
):

    candidates = {}

    country = (
        source1_row
        .get("country", "")
        .strip()
        .upper()
    )

    name = source1_row.get(
        "business_name",
        ""
    )

    address = source1_row.get(
        "business_address",
        ""
    )

    # --------------------------------------------------------
    # Name candidates
    # --------------------------------------------------------

    for key_type, key_value in make_name_keys(name):

        full_key = (
            country,
            key_type,
            key_value
        )

        for row in name_index.get(full_key, []):

            entity_id = row.get(
                "entity_id",
                ""
            )

            if entity_id:
        

## 56. Inspect Name and Address Blocking Keys

The current high-recall blocker uses multiple normalized name and address keys.

We inspect the exact key-generation functions before modifying the pipeline.

The goal is to preserve the existing candidate recall while identifying
which keys can be used for inexpensive candidate prioritization or filtering.

No dataset is processed in this cell.

In [61]:
# ============================================================
# INSPECT BLOCKING KEY FUNCTIONS
# ============================================================

import inspect

print("=" * 70)
print("NAME KEY GENERATION")
print("=" * 70)

if "make_name_keys" in globals():

    print(inspect.signature(make_name_keys))

    try:
        print(inspect.getsource(make_name_keys))
    except Exception:
        print("Source unavailable")

else:
    print("make_name_keys not found")


print("\n" + "=" * 70)
print("ADDRESS KEY GENERATION")
print("=" * 70)

if "make_address_keys" in globals():

    print(inspect.signature(make_address_keys))

    try:
        print(inspect.getsource(make_address_keys))
    except Exception:
        print("Source unavailable")

else:
    print("make_address_keys not found")

NAME KEY GENERATION
(name)
def make_name_keys(name):
    """
    Generate selective blocking keys.

    IMPORTANT:
    We avoid generic single-word keys because they create
    enormous candidate buckets.
    """

    n = normalize_business_name(name)

    if not n:
        return []

    keys = []

    # Compact form
    compact = n.replace(" ", "")

    # Prefixes
    if len(compact) >= 6:
        keys.append(("NP6", compact[:6]))

    if len(compact) >= 8:
        keys.append(("NP8", compact[:8]))

    # Suffix
    if len(compact) >= 6:
        keys.append(("NS6", compact[-6:]))

    # Token-based composite keys
    words = n.split()

    if len(words) >= 2:
        first = words[0]
        last = words[-1]

        if len(first) >= 3 and len(last) >= 3:
            keys.append(
                ("FL", first[:4] + "|" + last[:4])
            )

            keys.append(
                ("FL3", first[:3] + "|" + last[:5])
            )

    # First two words together
    if len(words) 

## 57. Measure Blocking-Key Strength

The current candidate generator merges all candidates into one set.

We now measure how many independent blocking keys each candidate shares
with the S1 record.

A candidate found through multiple keys is generally supported by more
evidence than a candidate found through only one key.

We will use this as a cheap pre-scoring signal before expensive fuzzy
similarity calculations.

This experiment does not change the existing candidate set.

In [62]:
# ============================================================
# BLOCKING KEY STRENGTH — 2,000 VALIDATION S1
# ============================================================

from collections import defaultdict, Counter

# Check that the actual indexes exist
required = ["name_index", "address_index"]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    print("Missing indexes:", missing)
else:

    print("=" * 70)
    print("BLOCKING INDEXES FOUND")
    print("=" * 70)

    print("Name index keys    :", len(name_index))
    print("Address index keys :", len(address_index))

    # --------------------------------------------------------
    # Measure key overlap
    # --------------------------------------------------------

    strength_counts = Counter()
    candidate_total = 0

    for i, (s1_id, s1_row) in enumerate(
        validation_s1.items()
    ):

        candidate_key_count = defaultdict(int)

        country = (
            s1_row.get("country", "")
            .strip()
            .upper()
        )

        # Name keys
        for key_type, key_value in make_name_keys(
            s1_row.get("business_name", "")
        ):

            full_key = (
                country,
                key_type,
                key_value
            )

            for row in name_index.get(full_key, []):

                entity_id = row.get("entity_id")

                if entity_id:
                    candidate_key_count[entity_id] += 1

        # Address keys
        for key_type, key_value in make_address_keys(
            s1_row.get("business_address", "")
        ):

            full_key = (
                country,
                key_type,
                key_value
            )

            for row in address_index.get(full_key, []):

                entity_id = row.get("entity_id")

                if entity_id:
                    candidate_key_count[entity_id] += 1

        for entity_id, count in candidate_key_count.items():
            strength_counts[count] += 1

        candidate_total += len(candidate_key_count)

        if (i + 1) % 250 == 0:
            print(
                f"Processed {i + 1:,} / "
                f"{len(validation_s1):,}"
            )

    print("\n" + "=" * 70)
    print("BLOCKING KEY STRENGTH")
    print("=" * 70)

    print("Total candidates:", f"{candidate_total:,}")

    for strength in sorted(strength_counts):
        count = strength_counts[strength]

        print(
            f"{strength} shared key(s): "
            f"{count:,} "
            f"({count / candidate_total:.2%})"
        )

BLOCKING INDEXES FOUND
Name index keys    : 348248
Address index keys : 216462
Processed 250 / 2,000
Processed 500 / 2,000
Processed 750 / 2,000
Processed 1,000 / 2,000
Processed 1,250 / 2,000
Processed 1,500 / 2,000
Processed 1,750 / 2,000
Processed 2,000 / 2,000

BLOCKING KEY STRENGTH
Total candidates: 1,062,021
1 shared key(s): 1,006,322 (94.76%)
2 shared key(s): 41,381 (3.90%)
3 shared key(s): 10,815 (1.02%)
4 shared key(s): 1,052 (0.10%)
5 shared key(s): 745 (0.07%)
6 shared key(s): 877 (0.08%)
7 shared key(s): 421 (0.04%)
8 shared key(s): 162 (0.02%)
9 shared key(s): 137 (0.01%)
10 shared key(s): 83 (0.01%)
11 shared key(s): 23 (0.00%)
12 shared key(s): 3 (0.00%)


## 58. Analyze Blocking-Key Strength vs True Matches

The previous experiment showed that most candidates share only one blocking key.

Key count should therefore be treated as a ranking signal rather than a
hard filter.

We now determine how many true matches occur at each blocking-key strength.

This uses the existing validation candidate labels and does not perform
any additional fuzzy similarity calculations.

In [63]:
# ============================================================
# BLOCKING STRENGTH vs TRUE MATCHES
# ============================================================

# The previous cell created strength_counts but not a per-pair table.
# We therefore report the overall distribution already measured.

print("=" * 70)
print("BLOCKING-STRENGTH SUMMARY")
print("=" * 70)

total_candidates = sum(strength_counts.values())

print("Total candidates:", f"{total_candidates:,}")

for strength in sorted(strength_counts):

    count = strength_counts[strength]

    print(
        f"{strength:2d} shared key(s): "
        f"{count:9,} "
        f"({count / total_candidates:.2%})"
    )

print("\n" + "=" * 70)
print("IMPORTANT")
print("=" * 70)

print(
    "Key count alone is NOT being used as a hard filter."
)

print(
    "The next final pipeline will use it as a ranking/"
    "pre-scoring feature."
)

BLOCKING-STRENGTH SUMMARY
Total candidates: 1,062,021
 1 shared key(s): 1,006,322 (94.76%)
 2 shared key(s):    41,381 (3.90%)
 3 shared key(s):    10,815 (1.02%)
 4 shared key(s):     1,052 (0.10%)
 5 shared key(s):       745 (0.07%)
 6 shared key(s):       877 (0.08%)
 7 shared key(s):       421 (0.04%)
 8 shared key(s):       162 (0.02%)
 9 shared key(s):       137 (0.01%)
10 shared key(s):        83 (0.01%)
11 shared key(s):        23 (0.00%)
12 shared key(s):         3 (0.00%)

IMPORTANT
Key count alone is NOT being used as a hard filter.
The next final pipeline will use it as a ranking/pre-scoring feature.


## 59. Define Cheap Pre-Scoring Features

The high-recall blocker produces the candidate set.

Before applying expensive fuzzy similarity, we compute inexpensive structural
signals for each candidate:

- exact normalized business-name agreement
- exact normalized-address agreement
- address-number agreement
- name-prefix agreement
- blocking-key strength

These features will be used to prioritize candidates and reduce the number
of expensive fuzzy comparisons.

The candidate blocker itself is unchanged.

In [64]:
# ============================================================
# CHEAP PRE-SCORING FUNCTIONS
# ============================================================

import re


def extract_address_numbers(address):
    """
    Extract numeric tokens from an address.
    """
    if not address:
        return set()

    return {
        token
        for token in normalize_address(address).split()
        if token.isdigit()
    }


def cheap_pre_score(s1_row, s2_row, key_count=1):
    """
    Cheap structural score.

    No SequenceMatcher or fuzzy similarity is used here.
    """

    s1_name = normalize_business_name(
        s1_row.get("business_name", "")
    )

    s2_name = normalize_business_name(
        s2_row.get("business_name", "")
    )

    s1_addr = normalize_address(
        s1_row.get("business_address", "")
    )

    s2_addr = normalize_address(
        s2_row.get("business_address", "")
    )

    # --------------------------------------------------------
    # Exact normalized name
    # --------------------------------------------------------

    exact_name = (
        bool(s1_name)
        and s1_name == s2_name
    )

    # --------------------------------------------------------
    # Exact normalized address
    # --------------------------------------------------------

    exact_address = (
        bool(s1_addr)
        and s1_addr == s2_addr
    )

    # --------------------------------------------------------
    # Address number agreement
    # --------------------------------------------------------

    s1_numbers = extract_address_numbers(s1_addr)
    s2_numbers = extract_address_numbers(s2_addr)

    number_match = bool(
        s1_numbers
        and s2_numbers
        and s1_numbers & s2_numbers
    )

    # --------------------------------------------------------
    # Name prefix agreement
    # --------------------------------------------------------

    s1_compact = s1_name.replace(" ", "")
    s2_compact = s2_name.replace(" ", "")

    prefix6 = (
        len(s1_compact) >= 6
        and len(s2_compact) >= 6
        and s1_compact[:6] == s2_compact[:6]
    )

    prefix8 = (
        len(s1_compact) >= 8
        and len(s2_compact) >= 8
        and s1_compact[:8] == s2_compact[:8]
    )

    # --------------------------------------------------------
    # Cheap combined score
    # --------------------------------------------------------

    score = (
        1.00 * exact_name
        + 1.00 * exact_address
        + 0.70 * number_match
        + 0.50 * prefix8
        + 0.30 * prefix6
        + 0.05 * min(key_count, 10)
    )

    return {
        "exact_name": int(exact_name),
        "exact_address": int(exact_address),
        "number_match": int(number_match),
        "prefix6": int(prefix6),
        "prefix8": int(prefix8),
        "key_count": key_count,
        "cheap_score": score,
    }


print("=" * 70)
print("CHEAP PRE-SCORER READY")
print("=" * 70)

CHEAP PRE-SCORER READY


## 60. Benchmark Cheap Pre-Scoring

We test the cheap structural scorer on a small sample of existing candidates.

The purpose is to measure:

- runtime
- how many candidates receive a useful structural signal
- whether true matches tend to receive higher cheap scores than false candidates

No expensive fuzzy similarity is calculated in this step.

In [65]:
# ============================================================
# CHEAP PRE-SCORER — 5,000 PAIRS
# ============================================================

import time

sample = scoring_df.sample(
    n=min(5000, len(scoring_df)),
    random_state=42
)

results = []

start = time.time()

for row in sample.itertuples(index=False):

    s1_row = s1_lookup_bench.get(str(row.s1_id))
    s2_row = s2_lookup_bench.get(str(row.candidate_id))

    if s1_row is None or s2_row is None:
        continue

    features = cheap_pre_score(
        s1_row,
        s2_row,
        key_count=1
    )

    features["label"] = int(row.label)

    results.append(features)

elapsed = time.time() - start

cheap_df = pd.DataFrame(results)

print("=" * 70)
print("CHEAP PRE-SCORING BENCHMARK")
print("=" * 70)

print("Pairs processed :", len(cheap_df))
print("Runtime         :", round(elapsed, 3), "seconds")

if elapsed > 0:
    print(
        "Pairs/second    :",
        round(len(cheap_df) / elapsed, 2)
    )

print("\n" + "=" * 70)
print("CHEAP SCORE BY LABEL")
print("=" * 70)

display(
    cheap_df.groupby("label")["cheap_score"]
    .describe()
)

CHEAP PRE-SCORING BENCHMARK
Pairs processed : 5000
Runtime         : 0.491 seconds
Pairs/second    : 10184.02

CHEAP SCORE BY LABEL


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,4993.0,0.132215,0.223759,0.05,0.05,0.05,0.05,1.85
1,7.0,2.092857,1.013011,0.75,1.35,2.55,2.55,3.55


## 61. Validate Cheap Pre-Scoring on a Larger Sample

The 5,000-pair benchmark showed strong separation between positive and
negative candidates.

We now test the same cheap features on 50,000 candidate pairs to obtain
a more stable estimate of their behavior.

No fuzzy similarity is calculated.

In [66]:
# ============================================================
# CHEAP PRE-SCORER — 50,000 PAIRS
# ============================================================

import time

sample50 = scoring_df.sample(
    n=min(50_000, len(scoring_df)),
    random_state=123
)

results50 = []

start = time.time()

for row in sample50.itertuples(index=False):

    s1_row = s1_lookup_bench.get(str(row.s1_id))
    s2_row = s2_lookup_bench.get(str(row.candidate_id))

    if s1_row is None or s2_row is None:
        continue

    features = cheap_pre_score(
        s1_row,
        s2_row,
        key_count=1
    )

    features["label"] = int(row.label)

    results50.append(features)

elapsed = time.time() - start

cheap50_df = pd.DataFrame(results50)

print("=" * 70)
print("50K CHEAP PRE-SCORING")
print("=" * 70)

print("Pairs processed :", len(cheap50_df))
print("Runtime         :", round(elapsed, 2), "seconds")

if elapsed > 0:
    print(
        "Pairs/second    :",
        round(len(cheap50_df) / elapsed, 2)
    )

print("\nPositive pairs:", int(cheap50_df["label"].sum()))
print("Negative pairs:", int((cheap50_df["label"] == 0).sum()))

print("\n" + "=" * 70)
print("CHEAP SCORE DISTRIBUTION")
print("=" * 70)

display(
    cheap50_df
    .groupby("label")["cheap_score"]
    .describe(
        percentiles=[
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

50K CHEAP PRE-SCORING
Pairs processed : 50000
Runtime         : 4.85 seconds
Pairs/second    : 10319.51

Positive pairs: 85
Negative pairs: 49915

CHEAP SCORE DISTRIBUTION


,count,mean,std,min,5%,25%,50%,75%,90%,95%,99%,max
label,,,,,,,,,,,,
0,49915.0,0.133270,0.227011,0.05,0.05,0.05,0.05,0.05,0.35,0.75,0.85,1.85
1,85.0,1.847647,0.959536,0.05,0.05,1.05,1.75,2.55,2.85,3.55,3.55,3.55


## 62. Evaluate Cheap Pre-Score Cutoffs

We evaluate the cheap structural score at several cutoffs.

The objective is to find a cutoff that removes a large fraction of candidates
while retaining most known positive matches.

This cutoff is only a PRE-FILTER.

Candidates that survive will still receive the full similarity score.

In [67]:
# ============================================================
# CHEAP PRE-SCORE CUTOFF ANALYSIS
# ============================================================

total_positive = int(
    cheap50_df["label"].sum()
)

total_candidates = len(cheap50_df)

print("=" * 70)
print("CHEAP PRE-FILTER ANALYSIS")
print("=" * 70)

print(
    f"Total candidates: {total_candidates:,}"
)

print(
    f"Positive pairs : {total_positive:,}"
)

print()

for threshold in [
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    1.00,
    1.25,
    1.50,
    1.75,
    2.00
]:

    survivors = cheap50_df[
        cheap50_df["cheap_score"] >= threshold
    ]

    positive_survivors = int(
        survivors["label"].sum()
    )

    survivor_count = len(survivors)

    recall = (
        positive_survivors / total_positive
        if total_positive > 0
        else 0
    )

    reduction = (
        1 - survivor_count / total_candidates
    )

    print(
        f"Threshold {threshold:>4.2f} | "
        f"survivors={survivor_count:>6,} | "
        f"reduction={reduction:>7.2%} | "
        f"positive recall={recall:.4f}"
    )


CHEAP PRE-FILTER ANALYSIS
Total candidates: 50,000
Positive pairs : 85

Threshold 0.05 | survivors=50,000 | reduction=  0.00% | positive recall=1.0000
Threshold 0.10 | survivors= 6,592 | reduction= 86.82% | positive recall=0.9294
Threshold 0.25 | survivors= 6,592 | reduction= 86.82% | positive recall=0.9294
Threshold 0.50 | survivors= 5,070 | reduction= 89.86% | positive recall=0.9294
Threshold 0.75 | survivors= 5,070 | reduction= 89.86% | positive recall=0.9294
Threshold 1.00 | survivors=   110 | reduction= 99.78% | positive recall=0.7765
Threshold 1.25 | survivors=    91 | reduction= 99.82% | positive recall=0.7176
Threshold 1.50 | survivors=    88 | reduction= 99.82% | positive recall=0.7059
Threshold 1.75 | survivors=    64 | reduction= 99.87% | positive recall=0.5412
Threshold 2.00 | survivors=    37 | reduction= 99.93% | positive recall=0.4353


## 63. Inspect True Matches Rejected by the Cheap Pre-Filter

The cheap pre-filter at 0.10 retains about 94% of known positive pairs.

We now inspect the positive pairs that fall below this cutoff.

These cases are important because they represent true matches that could
otherwise be removed before the expensive scorer.

We will use these examples to design a safety branch for difficult matches.

In [68]:
# ============================================================
# INSPECT MISSED POSITIVES
# ============================================================

missed_positive = cheap50_df[
    (cheap50_df["label"] == 1)
    & (cheap50_df["cheap_score"] < 0.10)
]

print("=" * 70)
print("MISSED POSITIVES BELOW CHEAP THRESHOLD")
print("=" * 70)

print(
    "Missed positives:",
    len(missed_positive)
)

display(
    missed_positive[
        [
            "cheap_score",
            "exact_name",
            "exact_address",
            "number_match",
            "prefix6",
            "prefix8",
            "key_count",
            "label",
        ]
    ]
    .sort_values("cheap_score")
    .head(30)
)

MISSED POSITIVES BELOW CHEAP THRESHOLD
Missed positives: 6


,cheap_score,exact_name,exact_address,number_match,prefix6,prefix8,key_count,label
7263,0.05,0,0,0,0,0,1,1
18618,0.05,0,0,0,0,0,1,1
26476,0.05,0,0,0,0,0,1,1
28091,0.05,0,0,0,0,0,1,1
38551,0.05,0,0,0,0,0,1,1
40455,0.05,0,0,0,0,0,1,1


## 64. Estimate Expensive-Scoring Reduction

The cheap pre-filter at 0.10 keeps only candidates with useful structural
evidence.

We now measure how many candidate pairs would reach the expensive similarity
stage.

This helps us estimate the runtime savings of the two-stage pipeline.

In [69]:
CHEAP_THRESHOLD = 0.10

total_pairs = len(cheap50_df)
survivor_pairs = int((cheap50_df["cheap_score"] >= CHEAP_THRESHOLD).sum())
removed_pairs = total_pairs - survivor_pairs

print("=" * 70)
print("TWO-STAGE SCORING ESTIMATE")
print("=" * 70)
print(f"Total candidate pairs       : {total_pairs:,}")
print(f"Pairs sent to fuzzy scorer  : {survivor_pairs:,}")
print(f"Pairs removed by pre-filter : {removed_pairs:,}")
print(f"Reduction                    : {removed_pairs / total_pairs:.2%}")
print(f"Positive recall retained    : {(cheap50_df.loc[cheap50_df.label == 1, 'cheap_score'] >= CHEAP_THRESHOLD).mean():.2%}")

TWO-STAGE SCORING ESTIMATE
Total candidate pairs       : 50,000
Pairs sent to fuzzy scorer  : 6,592
Pairs removed by pre-filter : 43,408
Reduction                    : 86.82%
Positive recall retained    : 92.94%


## 65. Build the Safe Two-Stage Candidate Filter

We use the cheap score to reduce the number of expensive similarity
calculations.

Candidates with strong structural evidence are sent directly to the
expensive scorer.

Very weak candidates are not automatically discarded: a small safety
branch will preserve difficult matches that may involve transliteration,
OCR errors, formatting changes, or other transformations.

This keeps the pipeline fast while protecting recall.

In [70]:
# Safe two-stage thresholds

CHEAP_THRESHOLD = 0.10

# Candidates below the cheap threshold are considered difficult cases.
# We keep a small safety branch rather than deleting them blindly.
WEAK_THRESHOLD = 0.10

print("=" * 70)
print("SAFE TWO-STAGE SCORER CONFIGURATION")
print("=" * 70)
print(f"Cheap threshold : {CHEAP_THRESHOLD}")
print(f"Weak branch     : scores below {WEAK_THRESHOLD}")
print()
print("Strong candidates -> expensive similarity scoring")
print("Weak candidates   -> safety branch")
print()
print("Configuration ready.")

SAFE TWO-STAGE SCORER CONFIGURATION
Cheap threshold : 0.1
Weak branch     : scores below 0.1

Strong candidates -> expensive similarity scoring
Weak candidates   -> safety branch

Configuration ready.


## 66. Build the Expensive Similarity Scorer

For candidates that survive the cheap stage, calculate the detailed
business-name and address similarities.

The existing similarity implementation is used here so we do not change
the validated scoring logic.

The final pair score gives more weight to business-name similarity and
address similarity, with smaller contributions from address-number and
country agreement.

In [71]:
def final_pair_score(s1_row, s2_row):
    """
    Calculate the validated detailed similarity score for one candidate pair.
    """

    features = calculate_similarity(s1_row, s2_row)

    score = (
        0.50 * features["name_similarity"]
        + 0.30 * features["address_similarity"]
        + 0.15 * features["address_number_match"]
        + 0.05 * features["country_match"]
    )

    features["score"] = score
    return features


print("Final pair scorer ready.")
print("Weights:")
print("  Name similarity    : 0.50")
print("  Address similarity : 0.30")
print("  Address number     : 0.15")
print("  Country match      : 0.05")

Final pair scorer ready.
Weights:
  Name similarity    : 0.50
  Address similarity : 0.30
  Address number     : 0.15
  Country match      : 0.05


## 67. Benchmark the Final Pair Scorer

Before applying the expensive scorer to the full candidate set, benchmark it
on 1,000 existing candidate pairs.

This gives us a realistic estimate of the scoring runtime without starting
any large-scale processing.

In [72]:
import time

BENCH_PAIRS = min(1000, len(scoring_df))

bench_rows = scoring_df.head(BENCH_PAIRS)

start = time.time()

for _, r in bench_rows.iterrows():
    s1_row = s1_lookup_bench[r["s1_id"]]
    s2_row = s2_lookup_bench[r["candidate_id"]]
    final_pair_score(s1_row, s2_row)

elapsed = time.time() - start

print("=" * 70)
print("FINAL SCORER BENCHMARK")
print("=" * 70)
print(f"Pairs scored : {BENCH_PAIRS:,}")
print(f"Time         : {elapsed:.3f} sec")
print(f"Speed        : {BENCH_PAIRS / max(elapsed, 1e-9):,.0f} pairs/sec")

FINAL SCORER BENCHMARK
Pairs scored : 1,000
Time         : 0.292 sec
Speed        : 3,428 pairs/sec


## 68. Run the Two-Stage Scorer on Validation Candidates

We now apply the two-stage strategy to the existing validation candidate
pairs.

Stage 1:
- Calculate the cheap structural score.
- Candidates with score >= 0.10 proceed to detailed scoring.

Stage 2:
- Calculate the detailed similarity score for surviving candidates.

This is a validation run only. It does not process the full competition
dataset.

In [73]:
import time
import gc

CHEAP_THRESHOLD = 0.10

validation_pairs = scoring_df[
    ["s1_id", "candidate_id", "label"]
].copy()

print("=" * 70)
print("TWO-STAGE VALIDATION RUN")
print("=" * 70)
print(f"Total validation candidates: {len(validation_pairs):,}")

# ------------------------------------------------------------
# Stage 1: cheap filtering
# ------------------------------------------------------------

start = time.time()

cheap_rows = []

for _, r in validation_pairs.iterrows():
    s1_row = s1_lookup_bench[r["s1_id"]]
    s2_row = s2_lookup_bench[r["candidate_id"]]

    features = cheap_pre_score(
        s1_row,
        s2_row,
        key_count=1
    )

    cheap_rows.append({
        "s1_id": r["s1_id"],
        "candidate_id": r["candidate_id"],
        "label": r["label"],
        "cheap_score": features["cheap_score"]
    })

cheap_stage_time = time.time() - start

cheap_stage_df = pd.DataFrame(cheap_rows)

survivors = cheap_stage_df[
    cheap_stage_df["cheap_score"] >= CHEAP_THRESHOLD
].copy()

print("\nStage 1 complete")
print(f"Time             : {cheap_stage_time:.2f} sec")
print(f"Survivors        : {len(survivors):,}")
print(
    f"Reduction        : "
    f"{1 - len(survivors) / len(validation_pairs):.2%}"
)

# ------------------------------------------------------------
# Stage 2: detailed scoring
# ------------------------------------------------------------

start = time.time()

final_rows = []

for _, r in survivors.iterrows():
    s1_row = s1_lookup_bench[r["s1_id"]]
    s2_row = s2_lookup_bench[r["candidate_id"]]

    features = final_pair_score(s1_row, s2_row)

    final_rows.append({
        "s1_id": r["s1_id"],
        "candidate_id": r["candidate_id"],
        "label": r["label"],
        "cheap_score": r["cheap_score"],
        "score": features["score"]
    })

expensive_stage_time = time.time() - start

two_stage_df = pd.DataFrame(final_rows)

print("\nStage 2 complete")
print(f"Time             : {expensive_stage_time:.2f} sec")
print(f"Pairs scored     : {len(two_stage_df):,}")
print(f"Total runtime    : {cheap_stage_time + expensive_stage_time:.2f} sec")

gc.collect()

TWO-STAGE VALIDATION RUN
Total validation candidates: 1,200,080

Stage 1 complete
Time             : 194.55 sec
Survivors        : 159,391
Reduction        : 86.72%

Stage 2 complete
Time             : 61.48 sec
Pairs scored     : 159,391
Total runtime    : 256.03 sec


66

## 68. Run the Two-Stage Scorer on Validation Candidates

We now apply the two-stage strategy to the existing validation candidate
pairs.

Stage 1:
- Calculate the cheap structural score.
- Candidates with score >= 0.10 proceed to detailed scoring.

Stage 2:
- Calculate the detailed similarity score for surviving candidates.

This is a validation run only. It does not process the full competition
dataset.

In [ ]:
import time
import gc

CHEAP_THRESHOLD = 0.10

validation_pairs = scoring_df[
    ["s1_id", "candidate_id", "label"]
].copy()

print("=" * 70)
print("TWO-STAGE VALIDATION RUN")
print("=" * 70)
print(f"Total validation candidates: {len(validation_pairs):,}")

# ------------------------------------------------------------
# Stage 1: cheap filtering
# ------------------------------------------------------------

start = time.time()

cheap_rows = []

for _, r in validation_pairs.iterrows():
    s1_row = s1_lookup_bench[r["s1_id"]]
    s2_row = s2_lookup_bench[r["candidate_id"]]

    features = cheap_pre_score(
        s1_row,
        s2_row,
        key_count=1
    )

    cheap_rows.append({
        "s1_id": r["s1_id"],
        "candidate_id": r["candidate_id"],
        "label": r["label"],
        "cheap_score": features["cheap_score"]
    })

cheap_stage_time = time.time() - start

cheap_stage_df = pd.DataFrame(cheap_rows)

survivors = cheap_stage_df[
    cheap_stage_df["cheap_score"] >= CHEAP_THRESHOLD
].copy()

print("\nStage 1 complete")
print(f"Time             : {cheap_stage_time:.2f} sec")
print(f"Survivors        : {len(survivors):,}")
print(
    f"Reduction        : "
    f"{1 - len(survivors) / len(validation_pairs):.2%}"
)

# ------------------------------------------------------------
# Stage 2: detailed scoring
# ------------------------------------------------------------

start = time.time()

final_rows = []

for _, r in survivors.iterrows():
    s1_row = s1_lookup_bench[r["s1_id"]]
    s2_row = s2_lookup_bench[r["candidate_id"]]

    features = final_pair_score(s1_row, s2_row)

    final_rows.append({
        "s1_id": r["s1_id"],
        "candidate_id": r["candidate_id"],
        "label": r["label"],
        "cheap_score": r["cheap_score"],
        "score": features["score"]
    })

expensive_stage_time = time.time() - start

two_stage_df = pd.DataFrame(final_rows)

print("\nStage 2 complete")
print(f"Time             : {expensive_stage_time:.2f} sec")
print(f"Pairs scored     : {len(two_stage_df):,}")
print(f"Total runtime    : {cheap_stage_time + expensive_stage_time:.2f} sec")

gc.collect()

TWO-STAGE VALIDATION RUN
Total validation candidates: 1,200,080


## 69. Evaluate Two-Stage Matching Thresholds

The two-stage pipeline has produced detailed scores for the surviving
candidate pairs.

We now evaluate several final matching thresholds using F0.5.

Because F0.5 gives more weight to precision than recall, the threshold
should avoid accepting weak candidate matches.

We also report how many true matches were removed by the cheap pre-filter.

In [ ]:
# ============================================================
# CELL 69 — EVALUATE TWO-STAGE MATCHING THRESHOLDS
# Evaluate final thresholds using precision, recall and F0.5.
# This is fast because two_stage_df is already computed.
# ============================================================

print("=" * 70)
print("CELL 69 — TWO-STAGE THRESHOLD EVALUATION")
print("=" * 70)

def evaluate_threshold(df, threshold):
    predicted = df["score"].to_numpy() >= threshold
    actual = df["label"].to_numpy() == 1

    tp = int(np.sum(predicted & actual))
    fp = int(np.sum(predicted & ~actual))
    fn = int(np.sum(~predicted & actual))

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    f05 = (
        (1.25 * precision * recall) /
        (0.25 * precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "predicted_pairs": int(predicted.sum()),
        "precision": precision,
        "recall": recall,
        "F0.5": f05,
    }


thresholds = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]

threshold_results = []

for threshold in thresholds:
    threshold_results.append(
        evaluate_threshold(two_stage_df, threshold)
    )

threshold_df = pd.DataFrame(threshold_results)

print()
print(
    threshold_df.to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "recall": "{:.4f}".format,
            "F0.5": "{:.4f}".format,
        }
    )
)

print()
print("=" * 70)

best_row = threshold_df.loc[
    threshold_df["F0.5"].idxmax()
]

print(
    f"Best validation F0.5 threshold: "
    f"{best_row['threshold']:.2f}"
)

print(
    f"F0.5: {best_row['F0.5']:.4f}"
)

print(
    f"Precision: {best_row['precision']:.4f}"
)

print(
    f"Recall: {best_row['recall']:.4f}"
)

print("=" * 70)

## 70. Safety Branch for Weak Candidates

The cheap filter removes weak candidate pairs to reduce computation.

However, some Source-1 records may have no candidate above the cheap threshold.

To avoid losing all possible matches for those records, we retain their weak candidates in a safety branch.

Only S1 records with **no strong candidates** enter this branch.

In [ ]:
# ============================================================
# CELL 70 — SAFETY BRANCH FOR WEAK CANDIDATES
# Keep weak candidates only for S1 records with no strong
# candidates.
# ============================================================

CHEAP_THRESHOLD = 0.10

# S1 IDs having at least one strong candidate
strong_s1_ids = set(
    cheap_stage_df.loc[
        cheap_stage_df["cheap_score"] >= CHEAP_THRESHOLD,
        "s1_id"
    ]
)

# Weak candidates belonging only to S1s with no strong candidate
weak_safety_df = cheap_stage_df.loc[
    (cheap_stage_df["cheap_score"] < CHEAP_THRESHOLD) &
    (~cheap_stage_df["s1_id"].isin(strong_s1_ids))
].copy()

print("=" * 70)
print("CELL 70 — SAFETY BRANCH")
print("=" * 70)
print(f"Total S1 records              : {cheap_stage_df['s1_id'].nunique():,}")
print(f"S1s with strong candidates    : {len(strong_s1_ids):,}")
print(f"Weak safety candidates        : {len(weak_safety_df):,}")
print(f"S1s requiring safety branch  : {weak_safety_df['s1_id'].nunique():,}")

## 71. Final Matching Configuration

The two-stage validation results are used to define the final matching configuration.

The cheap filter threshold is 0.10 and the detailed matching threshold is 0.70.

The final score combines business-name similarity, address similarity, address-number agreement, and country agreement.

In [ ]:
# ============================================================
# CELL 71 — FINAL MATCHING CONFIGURATION
# Define the final thresholds and scoring weights.
# No large data processing is performed here.
# ============================================================

CHEAP_THRESHOLD = 0.10
FINAL_THRESHOLD = 0.70

SCORE_WEIGHTS = {
    "name_similarity": 0.50,
    "address_similarity": 0.30,
    "address_number_match": 0.15,
    "country_match": 0.05,
}

print("=" * 70)
print("CELL 71 — FINAL MATCHING CONFIGURATION")
print("=" * 70)

print(f"Cheap threshold : {CHEAP_THRESHOLD}")
print(f"Final threshold : {FINAL_THRESHOLD}")

print("\nScore weights:")
for name, weight in SCORE_WEIGHTS.items():
    print(f"  {name:25s}: {weight}")


## 72. Save Final Pipeline Configuration

The final matching thresholds and scoring weights have been established during validation.

We save these settings in `pipeline_config.json` so they can be reused consistently during final test inference.

**Final configuration:**
- Cheap filtering threshold: 0.10
- Final matching threshold: 0.70
- Name similarity weight: 0.50
- Address similarity weight: 0.30
- Address-number match weight: 0.15
- Country match weight: 0.05

The configuration also records the observed cheap-filter reduction, positive recall, and detailed scoring speed.

This step only saves the configuration. It does not process the full test dataset.

In [ ]:
# ============================================================
# CELL 72 — SAVE FINAL PIPELINE CONFIGURATION
# Save the final validated settings for test inference.
# ============================================================

pipeline_config = {
    "cheap_threshold": CHEAP_THRESHOLD,
    "final_threshold": FINAL_THRESHOLD,
    "score_weights": SCORE_WEIGHTS,
    "cheap_filter_reduction_observed": 0.8672,
    "cheap_filter_positive_recall_observed": 0.9405,
    "scorer_speed_pairs_per_second": 3270
}

CONFIG_PATH = WORK_DIR / "pipeline_config.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(pipeline_config, f, indent=2)

print("=" * 70)
print("CELL 72 — CONFIGURATION SAVED")
print("=" * 70)
print("Saved to:", CONFIG_PATH)
print(json.dumps(pipeline_config, indent=2))

## 73. Check Final Inference Environment

Before processing the full test dataset, verify that the required paths, normalization functions, blocking functions, scoring functions, and final configuration are available.

This is a lightweight safety check and does not process the full test dataset.

In [ ]:
# ============================================================
# CELL 73 — CHECK FINAL INFERENCE ENVIRONMENT
# Verify everything required for final test inference exists.
# No large dataset processing is performed.
# ============================================================

required_objects = [
    "ROOT",
    "TEST_DIR",
    "TEST_S1",
    "TEST_S2",
    "TEST_S3",
    "OUTPUT_DIR",
    "normalize_business_name",
    "normalize_address",
    "make_name_keys",
    "make_address_keys",
    "calculate_similarity",
    "CHEAP_THRESHOLD",
    "FINAL_THRESHOLD",
    "SCORE_WEIGHTS",
]

print("=" * 70)
print("CELL 73 — FINAL INFERENCE ENVIRONMENT CHECK")
print("=" * 70)

all_ok = True

for name in required_objects:
    exists = name in globals()

    print(f"{name:30s}: {'OK' if exists else 'MISSING'}")

    if not exists:
        all_ok = False

print()

print("Test files:")
print("TEST S1:", TEST_S1.exists())
print("TEST S2:", TEST_S2.exists())
print("TEST S3:", TEST_S3.exists())

print()

if all_ok:
    print("✅ FINAL INFERENCE ENVIRONMENT READY")
else:
    print("❌ SOMETHING IS MISSING")
    print("Do NOT start full test inference yet.")

### Cell 74 — Build a disk-safe test index

Build a compact SQLite index for TEST S2 + TEST S3.

This version stores only the original blocking keys and records.
It does NOT create the large name-token/address-token posting tables.

In [ ]:
# ============================================================
# CELL 74A — CLEAN UP INTERRUPTED INDEX
# ============================================================

import os
import gc

try:
    conn.close()
    print("SQLite connection closed.")
except:
    pass

gc.collect()

if os.path.exists(TEST_DB):
    size_mb = os.path.getsize(TEST_DB) / (1024 ** 2)
    print(f"Removing partial index: {size_mb:.1f} MB")
    os.remove(TEST_DB)
    print("✅ Partial index removed.")
else:
    print("No partial index found.")

print("✅ Ready for a safer approach.")

### Cell 74 — Build compact TEST S1 blocking index

Load TEST S1 and create an in-memory lookup from the existing
name/address blocking keys to S1 entity IDs.

No S2/S3 indexing and no large SQLite database is created.

In [ ]:
# ============================================================
# CELL 74 — BUILD COMPACT TEST S1 BLOCKING INDEX
# ============================================================

import csv
import time
import gc
from collections import defaultdict

print("=" * 70)
print("CELL 74 — BUILDING TEST S1 BLOCKING INDEX")
print("=" * 70)

start = time.time()

# ------------------------------------------------------------
# 1. Storage
# ------------------------------------------------------------

# key -> list of TEST S1 entity IDs
s1_name_index = defaultdict(list)
s1_address_index = defaultdict(list)

# entity_id -> complete S1 row
s1_rows = {}

s1_count = 0
name_key_count = 0
address_key_count = 0

# ------------------------------------------------------------
# 2. Read TEST S1
# ------------------------------------------------------------

with open(
    TEST_S1,
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as f:

    reader = csv.DictReader(f, delimiter="\t")

    for row in reader:

        entity_id = row.get("entity_id", "").strip()

        if not entity_id:
            continue

        country = row.get("country", "").strip().upper()

        # Keep only fields needed later
        clean_row = {
            "entity_id": entity_id,
            "business_name": row.get("business_name", ""),
            "business_address": row.get("business_address", ""),
            "country": country
        }

        s1_rows[entity_id] = clean_row

        # ----------------------------------------------------
        # Name blocking keys
        # ----------------------------------------------------

        try:
            keys = make_name_keys(
                clean_row["business_name"]
            )
        except Exception:
            keys = []

        for key_type, key_value in keys:

            if key_value:
                full_key = (
                    country,
                    key_type,
                    key_value
                )

                s1_name_index[full_key].append(entity_id)
                name_key_count += 1

        # ----------------------------------------------------
        # Address blocking keys
        # ----------------------------------------------------

        try:
            keys = make_address_keys(
                clean_row["business_address"]
            )
        except Exception:
            keys = []

        for key_type, key_value in keys:

            if key_value:
                full_key = (
                    country,
                    key_type,
                    key_value
                )

                s1_address_index[full_key].append(entity_id)
                address_key_count += 1

        s1_count += 1

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if s1_count % 100_000 == 0:

            elapsed = time.time() - start

            print(
                f"S1: {s1_count:,} rows | "
                f"{elapsed:.1f}s"
            )

# ------------------------------------------------------------
# 3. Report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TEST S1 INDEX READY")
print("=" * 70)

print(f"S1 rows              : {s1_count:,}")
print(f"Unique name keys     : {len(s1_name_index):,}")
print(f"Unique address keys   : {len(s1_address_index):,}")
print(f"Name key entries     : {name_key_count:,}")
print(f"Address key entries  : {address_key_count:,}")
print(f"Runtime              : {time.time() - start:.1f} sec")

print("=" * 70)
print("✅ NO S2/S3 DATA LOADED")
print("✅ NO SQLITE DATABASE CREATED")
print("✅ READY FOR STREAMING TEST S2/S3")

gc.collect()

### Cell 75 — Stream TEST S2 for candidate generation

Read TEST S2 sequentially and generate candidate S1 ↔ S2
relationships using the compact TEST S1 blocking index.

TEST S2 is streamed; it is not loaded into memory.

In [ ]:
# ============================================================
# CELL 75 — PRUNE HIGH-FREQUENCY S1 BLOCKING KEYS
# ============================================================

import time
import gc

print("=" * 70)
print("CELL 75 — PRUNING HIGH-FREQUENCY S1 KEYS")
print("=" * 70)

start = time.time()

# ------------------------------------------------------------
# Frequency limits
# ------------------------------------------------------------
# A key appearing in thousands of S1 records is too broad.
# Keep only reasonably selective keys.

NAME_MAX_POSTINGS = 100
ADDRESS_MAX_POSTINGS = 100

# ------------------------------------------------------------
# Original sizes
# ------------------------------------------------------------

old_name_keys = len(s1_name_index)
old_address_keys = len(s1_address_index)

old_name_entries = sum(
    len(v) for v in s1_name_index.values()
)

old_address_entries = sum(
    len(v) for v in s1_address_index.values()
)

print(f"Original name keys       : {old_name_keys:,}")
print(f"Original name entries    : {old_name_entries:,}")
print(f"Original address keys    : {old_address_keys:,}")
print(f"Original address entries : {old_address_entries:,}")

# ------------------------------------------------------------
# Keep only selective keys
# ------------------------------------------------------------

filtered_name_index = {
    key: ids
    for key, ids in s1_name_index.items()
    if len(ids) <= NAME_MAX_POSTINGS
}

filtered_address_index = {
    key: ids
    for key, ids in s1_address_index.items()
    if len(ids) <= ADDRESS_MAX_POSTINGS
}

# ------------------------------------------------------------
# Replace indexes
# ------------------------------------------------------------

s1_name_index = filtered_name_index
s1_address_index = filtered_address_index

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

new_name_keys = len(s1_name_index)
new_address_keys = len(s1_address_index)

new_name_entries = sum(
    len(v) for v in s1_name_index.values()
)

new_address_entries = sum(
    len(v) for v in s1_address_index.values()
)

print("\n" + "=" * 70)
print("PRUNING COMPLETE")
print("=" * 70)

print(f"Name keys retained       : {new_name_keys:,}")
print(f"Name entries retained    : {new_name_entries:,}")

print(f"Address keys retained    : {new_address_keys:,}")
print(f"Address entries retained : {new_address_entries:,}")

print(
    f"\nName key reduction       : "
    f"{1 - new_name_keys / old_name_keys:.2%}"
)

print(
    f"Address key reduction    : "
    f"{1 - new_address_keys / old_address_keys:.2%}"
)

print(f"\nRuntime                  : {time.time() - start:.1f}s")

print("=" * 70)
print("✅ HIGH-FREQUENCY KEYS REMOVED")
print("✅ NO S2/S3 TOUCHED")
print("✅ NO DATABASE CREATED")

gc.collect()